### There was a few corrections in the dataset that needed to be done after geetiing the mail back from the data providers and that is the whole purpose of this file.

## I had found that there were some errors in the names that would make it into a different projet and after confirmation from the data providers. So I am going to fix that first.

In [1]:
# ============================================================
# Step 1 - Cell 1
# Imports and file paths
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


# Immutable corrected transaction dataset
TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


# Main controlled product mapping created in this step
MASTER_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling.csv"
)


# Audit of automatically applied spelling corrections
SPELLING_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_spelling_corrections_step1_audit.csv"
)


# Names that require human confirmation
MANUAL_REVIEW_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_product_name_manual_review_candidates_step1.csv"
)


DATA_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction dataset was not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


print("Input transaction dataset:")
print(TRANSACTION_INPUT_FILE)

print()
print("Master mapping output:")
print(MASTER_MAPPING_OUTPUT_FILE)

Input transaction dataset:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv

Master mapping output:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling.csv


In [2]:
# ============================================================
# Step 1 - Cell 2
# Load and validate immutable corrected transactions
# ============================================================

transactions_step1 = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


required_columns = [
    "TransDate",
    "TransValue",
    "PLUName",
    "GroupCode",
    "GroupName",
    "PLUCode",
    "Date",
    "TransactionID",
    "UnitSold",
]


missing_columns = [
    column
    for column in required_columns
    if column not in transactions_step1.columns
]


if missing_columns:
    raise ValueError(
        "Required transaction columns are missing:\n"
        f"{missing_columns}"
    )


# ------------------------------------------------------------
# Safe type conversion
# ------------------------------------------------------------

transactions_step1["TransDate"] = pd.to_datetime(
    transactions_step1["TransDate"],
    errors="raise"
)


transactions_step1["Date"] = pd.to_datetime(
    transactions_step1["Date"],
    errors="raise"
).dt.normalize()


transactions_step1["PLUCode"] = (
    pd.to_numeric(
        transactions_step1["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions_step1["GroupCode"] = (
    pd.to_numeric(
        transactions_step1["GroupCode"],
        errors="raise"
    )
    .astype("int64")
)


transactions_step1["UnitSold"] = (
    pd.to_numeric(
        transactions_step1["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


transactions_step1["TransValue"] = pd.to_numeric(
    transactions_step1["TransValue"],
    errors="raise"
)


# Preserve exact original names
transactions_step1["PLUName"] = (
    transactions_step1["PLUName"]
    .astype("string")
    .str.strip()
)


transactions_step1["GroupName"] = (
    transactions_step1["GroupName"]
    .astype("string")
    .str.strip()
)


# ------------------------------------------------------------
# Record immutable totals
# ------------------------------------------------------------

step1_original_row_count = len(
    transactions_step1
)


step1_original_unit_total = int(
    transactions_step1["UnitSold"].sum()
)


step1_original_value_cents = int(
    (
        transactions_step1["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate exact source dataset
# ------------------------------------------------------------

assert step1_original_row_count == 138_983, (
    "Expected 138,983 transaction rows."
)


assert step1_original_unit_total == 141_480, (
    "Expected corrected UnitSold total of 141,480."
)


assert transactions_step1["PLUCode"].nunique() == 236, (
    "Expected 236 PLU codes including OPEN UL."
)


assert transactions_step1["UnitSold"].ge(1).all(), (
    "UnitSold contains a value below 1."
)


print("Immutable transaction dataset validated.")
print()
print(
    "Rows:",
    f"{step1_original_row_count:,}"
)
print(
    "PLU codes:",
    f"{transactions_step1['PLUCode'].nunique():,}"
)
print(
    "Corrected unit total:",
    f"{step1_original_unit_total:,}"
)

Immutable transaction dataset validated.

Rows: 138,983
PLU codes: 236
Corrected unit total: 141,480


In [3]:
# ============================================================
# Step 1 - Cell 3
# Create one base mapping row per PLUCode
# ============================================================

plu_consistency_check = (
    transactions_step1
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        ProductNameCount=(
            "PLUName",
            "nunique"
        ),
        GroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        GroupNameCount=(
            "GroupName",
            "nunique"
        ),
    )
)


inconsistent_plu_codes = (
    plu_consistency_check.loc[
        (
            plu_consistency_check[
                "ProductNameCount"
            ].ne(1)
            |
            plu_consistency_check[
                "GroupCodeCount"
            ].ne(1)
            |
            plu_consistency_check[
                "GroupNameCount"
            ].ne(1)
        )
    ]
)


if len(inconsistent_plu_codes) > 0:
    display(inconsistent_plu_codes)

    raise ValueError(
        "At least one PLU code has multiple names or groups. "
        "The mapping cannot be created safely."
    )


# ------------------------------------------------------------
# Create one row per PLUCode
# ------------------------------------------------------------

master_product_mapping = (
    transactions_step1
    .sort_values(
        [
            "PLUCode",
            "Date",
            "TransDate",
        ]
    )
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        PLUName_Original=(
            "PLUName",
            "first"
        ),
        GroupCode=(
            "GroupCode",
            "first"
        ),
        GroupName=(
            "GroupName",
            "first"
        ),
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
        TotalTransactionValue=(
            "TransValue",
            "sum"
        ),
    )
)


# ------------------------------------------------------------
# Basic text standardisation only
# ------------------------------------------------------------
# Original name is preserved exactly.
# Corrected name is upper-case with repeated spaces removed.

master_product_mapping[
    "PLUName_Corrected"
] = (
    master_product_mapping[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.upper()
)


# ------------------------------------------------------------
# Every PLU initially remains a separate product
# ------------------------------------------------------------

master_product_mapping[
    "CanonicalProductID"
] = (
    "PLU_"
    + master_product_mapping[
        "PLUCode"
    ].astype(str)
)


master_product_mapping[
    "CanonicalProductName"
] = master_product_mapping[
    "PLUName_Corrected"
]


# These fields will be completed in later controlled steps
master_product_mapping[
    "SupplierBrand"
] = pd.NA

master_product_mapping[
    "ProductFamily"
] = pd.NA

master_product_mapping[
    "PriceTier"
] = pd.NA


master_product_mapping[
    "MappingType"
] = "UNCHANGED"


master_product_mapping[
    "MappingReason"
] = (
    "No approved spelling correction required in Step 1."
)


master_product_mapping[
    "MappingConfidence"
] = "NOT_APPLICABLE"


master_product_mapping[
    "MappingStatus"
] = "APPROVED_STEP_1"


master_product_mapping[
    "MappingVersion"
] = "STEP_1_SPELLING_V1"


# OPEN UL remains in the mapping for audit,
# but it is explicitly excluded from forecasting.
open_ul_mapping_mask = (
    master_product_mapping[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert int(open_ul_mapping_mask.sum()) == 1


master_product_mapping.loc[
    open_ul_mapping_mask,
    "MappingStatus"
] = "EXCLUDE_FROM_FORECASTING"


master_product_mapping.loc[
    open_ul_mapping_mask,
    "MappingReason"
] = (
    "Open-value POS entry; not a forecastable product."
)


print("Base master mapping created.")
print()
print(
    "Mapping rows:",
    len(master_product_mapping)
)
print(
    "Unique temporary canonical IDs:",
    master_product_mapping[
        "CanonicalProductID"
    ].nunique()
)

Base master mapping created.

Mapping rows: 236
Unique temporary canonical IDs: 236


In [4]:
# ============================================================
# Step 1 - Cell 4
# Apply approved high-confidence spelling corrections
# ============================================================

approved_spelling_corrections = [
    {
        "PLUCode": 42552,
        "ExpectedOriginal": "ALERNATIVE MILK",
        "CorrectedName": "ALTERNATIVE MILK",
        "Reason": "Corrected ALERNATIVE to ALTERNATIVE.",
    },
    {
        "PLUCode": 4241502,
        "ExpectedOriginal": "ALUXARY CROSSANT / PASTERIES",
        "CorrectedName": "LUXURY CROISSANT / PASTRIES",
        "Reason": (
            "Corrected ALUXARY, CROSSANT and PASTERIES."
        ),
    },
    {
        "PLUCode": 4241536,
        "ExpectedOriginal": "BREAKSFAST POTS",
        "CorrectedName": "BREAKFAST POTS",
        "Reason": "Corrected BREAKSFAST to BREAKFAST.",
    },
    {
        "PLUCode": 42527103,
        "ExpectedOriginal": "BEWLEYS CAPPUCINO",
        "CorrectedName": "BEWLEYS CAPPUCCINO",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4241460,
        "ExpectedOriginal": "CAPPUCINO 12OZ",
        "CorrectedName": "CAPPUCCINO 12OZ",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4241461,
        "ExpectedOriginal": "CAPPUCINO 16OZ",
        "CorrectedName": "CAPPUCCINO 16OZ",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251511,
        "ExpectedOriginal": "CAPPUCINO SM",
        "CorrectedName": "CAPPUCCINO SM",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251512,
        "ExpectedOriginal": "CAPPUCINO MED",
        "CorrectedName": "CAPPUCCINO MED",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251513,
        "ExpectedOriginal": "CAPPUCINO LG",
        "CorrectedName": "CAPPUCCINO LG",
        "Reason": "Corrected CAPPUCINO to CAPPUCCINO.",
    },
    {
        "PLUCode": 4251505,
        "ExpectedOriginal": "CARAMEL MACHIATO SM",
        "CorrectedName": "CARAMEL MACCHIATO SM",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251506,
        "ExpectedOriginal": "CARAMEL MACHIATO MED",
        "CorrectedName": "CARAMEL MACCHIATO MED",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251507,
        "ExpectedOriginal": "CARAMEL MACHIATO LG",
        "CorrectedName": "CARAMEL MACCHIATO LG",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251520,
        "ExpectedOriginal": "ICED CARAMEL MACHIATO MED",
        "CorrectedName": "ICED CARAMEL MACCHIATO MED",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4251521,
        "ExpectedOriginal": "ICED CARAMEL MACHIATO LG",
        "CorrectedName": "ICED CARAMEL MACCHIATO LG",
        "Reason": "Corrected MACHIATO to MACCHIATO.",
    },
    {
        "PLUCode": 4241466,
        "ExpectedOriginal": "CHIA 12 OZ",
        "CorrectedName": "CHAI 12 OZ",
        "Reason": (
            "Corrected CHIA to CHAI; this is a hot beverage "
            "and corresponds with CHAI 16OZ."
        ),
    },
    {
        "PLUCode": 42543,
        "ExpectedOriginal": "DELLI SANDWICH - NO MEAT",
        "CorrectedName": "DELI SANDWICH - NO MEAT",
        "Reason": "Corrected DELLI to DELI.",
    },
    {
        "PLUCode": 42544,
        "ExpectedOriginal": (
            "DELLI SANDWICH X3 SALAD X1 MEAT"
        ),
        "CorrectedName": (
            "DELI SANDWICH X3 SALAD X1 MEAT"
        ),
        "Reason": "Corrected DELLI to DELI.",
    },
    {
        "PLUCode": 425992,
        "ExpectedOriginal": "MOCA",
        "CorrectedName": "MOCHA",
        "Reason": "Corrected MOCA to MOCHA.",
    },
    {
        "PLUCode": 42571,
        "ExpectedOriginal": (
            "PROTIAN BAR POWER BALLS CT"
        ),
        "CorrectedName": (
            "PROTEIN BAR POWER BALLS CT"
        ),
        "Reason": "Corrected PROTIAN to PROTEIN.",
    },
    {
        "PLUCode": 42555,
        "ExpectedOriginal": (
            "R0OASTED NOTES EXPRESSO CT"
        ),
        "CorrectedName": (
            "ROASTED NOTES ESPRESSO CT"
        ),
        "Reason": (
            "Corrected zero in R0OASTED and "
            "EXPRESSO to ESPRESSO."
        ),
    },
    {
        "PLUCode": 4241422,
        "ExpectedOriginal": (
            "SALTED CARAMEL PEANUR BISCUIT CAKE"
        ),
        "CorrectedName": (
            "SALTED CARAMEL PEANUT BISCUIT CAKE"
        ),
        "Reason": "Corrected PEANUR to PEANUT.",
    },
    {
        "PLUCode": 4241448,
        "ExpectedOriginal": "SINGEL ESPESSO",
        "CorrectedName": "SINGLE ESPRESSO",
        "Reason": (
            "Corrected SINGEL and ESPESSO."
        ),
    },
    {
        "PLUCode": 4241446,
        "ExpectedOriginal": (
            "SPECIALILITY TEA 12 OZ"
        ),
        "CorrectedName": (
            "SPECIALITY TEA 12 OZ"
        ),
        "Reason": (
            "Corrected SPECIALILITY to SPECIALITY."
        ),
    },
    {
        "PLUCode": 2000000130,
        "ExpectedOriginal": (
            "JUICE 200 ML 200ML BTL"
        ),
        "CorrectedName": "JUICE 200ML BTL",
        "Reason": (
            "Removed the duplicated 200 ML description."
        ),
    },
    {
        "PLUCode": 4251515,
        "ExpectedOriginal": "CARAMEL FRAPP LG",
        "CorrectedName": "CARAMEL FRAP LG",
        "Reason": (
            "Standardised FRAPP to FRAP to match "
            "the associated medium product."
        ),
    },
]


approved_spelling_corrections_df = pd.DataFrame(
    approved_spelling_corrections
)


assert approved_spelling_corrections_df[
    "PLUCode"
].is_unique, (
    "A PLU code appears more than once in the approved "
    "spelling-correction list."
)


# ------------------------------------------------------------
# Apply each correction safely
# ------------------------------------------------------------

for correction in approved_spelling_corrections:

    plu_code = correction["PLUCode"]

    mapping_mask = (
        master_product_mapping[
            "PLUCode"
        ].eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected exactly one mapping row for PLU {plu_code}."
    )


    actual_original = (
        master_product_mapping.loc[
            mapping_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_original = (
        correction["ExpectedOriginal"]
        .strip()
        .upper()
    )


    assert actual_original == expected_original, (
        f"Original-name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_original}\n"
        f"Found: {actual_original}"
    )


    corrected_name = (
        correction["CorrectedName"]
        .strip()
        .upper()
    )


    master_product_mapping.loc[
        mapping_mask,
        "PLUName_Corrected"
    ] = corrected_name


    master_product_mapping.loc[
        mapping_mask,
        "CanonicalProductName"
    ] = corrected_name


    master_product_mapping.loc[
        mapping_mask,
        "MappingType"
    ] = "SPELLING_CORRECTION"


    master_product_mapping.loc[
        mapping_mask,
        "MappingReason"
    ] = correction["Reason"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingConfidence"
    ] = "HIGH"


    master_product_mapping.loc[
        mapping_mask,
        "MappingStatus"
    ] = "APPROVED_STEP_1"


print(
    "Approved spelling corrections applied:",
    len(approved_spelling_corrections_df)
)

Approved spelling corrections applied: 25


In [5]:
# ============================================================
# Step 1 - Cell 5
# Create manual-review candidates without changing names
# ============================================================

manual_review_candidates = [
    {
        "PLUCode": 4241404,
        "ExpectedOriginal": "ODONNELL /POPCORN",
        "SuggestedName": "O'DONNELL'S POPCORN",
        "PotentialIssue": (
            "Possible missing apostrophe, possessive S "
            "and incorrect separator."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 2024241,
        "ExpectedOriginal": "JOS POWERBALL",
        "SuggestedName": "JO'S POWERBALL",
        "PotentialIssue": (
            "Possible missing apostrophe; could also be "
            "a supplier-specific brand name."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 2000000269,
        "ExpectedOriginal": (
            "BALLYGOWAN SPARK WATER 330ML"
        ),
        "SuggestedName": (
            "BALLYGOWAN SPARKLING WATER 330ML"
        ),
        "PotentialIssue": (
            "SPARK may be an abbreviation for SPARKLING."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 4241537,
        "ExpectedOriginal": "CRUSTED CROISSANTS",
        "SuggestedName": "REQUIRES CONFIRMATION",
        "PotentialIssue": (
            "The intended product may be a specific type "
            "of crusted or filled croissant."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241455,
        "ExpectedOriginal": "ICED VANILLA 16OZ",
        "SuggestedName": "ICED VANILLA LATTE 16OZ",
        "PotentialIssue": (
            "LATTE may be missing, but the transaction data "
            "alone cannot confirm this."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241405,
        "ExpectedOriginal": "HUNKY DORY",
        "SuggestedName": "HUNKY DORYS",
        "PotentialIssue": (
            "Possible shortened brand name."
        ),
        "ReviewConfidence": "MEDIUM",
    },
    {
        "PLUCode": 4241409,
        "ExpectedOriginal": (
            "PACKS OF SWEET - WINE GUMS"
        ),
        "SuggestedName": (
            "PACK OF SWEETS - WINE GUMS"
        ),
        "PotentialIssue": (
            "Possible singular/plural wording error."
        ),
        "ReviewConfidence": "LOW",
    },
    {
        "PLUCode": 4241415,
        "ExpectedOriginal": (
            "LUXURY MILLIONAIRES FLAPJACK"
        ),
        "SuggestedName": (
            "LUXURY MILLIONAIRE'S FLAPJACK"
        ),
        "PotentialIssue": (
            "Possible possessive punctuation correction."
        ),
        "ReviewConfidence": "MEDIUM",
    },
]


manual_review_candidates_df = pd.DataFrame(
    manual_review_candidates
)


assert manual_review_candidates_df[
    "PLUCode"
].is_unique


# Ensure manual-review candidates do not overlap with
# approved spelling corrections.
overlapping_review_codes = set(
    manual_review_candidates_df["PLUCode"]
).intersection(
    set(
        approved_spelling_corrections_df["PLUCode"]
    )
)


assert not overlapping_review_codes, (
    "Manual-review and approved corrections overlap:\n"
    f"{sorted(overlapping_review_codes)}"
)


# ------------------------------------------------------------
# Validate and mark manual-review rows
# ------------------------------------------------------------

for candidate in manual_review_candidates:

    plu_code = candidate["PLUCode"]

    mapping_mask = (
        master_product_mapping[
            "PLUCode"
        ].eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected one mapping row for PLU {plu_code}."
    )


    actual_original = (
        master_product_mapping.loc[
            mapping_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_original = (
        candidate["ExpectedOriginal"]
        .strip()
        .upper()
    )


    assert actual_original == expected_original, (
        f"Manual-review name mismatch for PLU {plu_code}."
    )


    # Important:
    # PLUName_Corrected and CanonicalProductName are not changed.
    master_product_mapping.loc[
        mapping_mask,
        "MappingType"
    ] = "MANUAL_REVIEW"


    master_product_mapping.loc[
        mapping_mask,
        "MappingReason"
    ] = candidate["PotentialIssue"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingConfidence"
    ] = candidate["ReviewConfidence"]


    master_product_mapping.loc[
        mapping_mask,
        "MappingStatus"
    ] = "PENDING_MANUAL_REVIEW"


print(
    "Manual-review candidates identified:",
    len(manual_review_candidates_df)
)

display(manual_review_candidates_df)

Manual-review candidates identified: 8


,PLUCode,ExpectedOriginal,SuggestedName,PotentialIssue,ReviewConfidence
0,4241404,ODONNELL /POPCORN,O'DONNELL'S POPCORN,"Possible missing apostrophe, possessive S and ...",MEDIUM
1,2024241,JOS POWERBALL,JO'S POWERBALL,Possible missing apostrophe; could also be a s...,LOW
2,2000000269,BALLYGOWAN SPARK WATER 330ML,BALLYGOWAN SPARKLING WATER 330ML,SPARK may be an abbreviation for SPARKLING.,MEDIUM
3,4241537,CRUSTED CROISSANTS,REQUIRES CONFIRMATION,The intended product may be a specific type of...,LOW
4,4241455,ICED VANILLA 16OZ,ICED VANILLA LATTE 16OZ,"LATTE may be missing, but the transaction data...",LOW
5,4241405,HUNKY DORY,HUNKY DORYS,Possible shortened brand name.,MEDIUM
6,4241409,PACKS OF SWEET - WINE GUMS,PACK OF SWEETS - WINE GUMS,Possible singular/plural wording error.,LOW
7,4241415,LUXURY MILLIONAIRES FLAPJACK,LUXURY MILLIONAIRE'S FLAPJACK,Possible possessive punctuation correction.,MEDIUM


In [6]:
# ============================================================
# Step 1 - Cell 6
# Validate Step 1 isolation and integrity
# ============================================================

# ------------------------------------------------------------
# Mapping-table integrity
# ------------------------------------------------------------

assert len(master_product_mapping) == 236, (
    "Master mapping must contain 236 PLU rows."
)


assert master_product_mapping[
    "PLUCode"
].is_unique, (
    "PLUCode is not unique in the master mapping."
)


assert master_product_mapping[
    "CanonicalProductID"
].is_unique, (
    "A product merge has accidentally occurred in Step 1."
)


assert master_product_mapping[
    "CanonicalProductID"
].nunique() == 236, (
    "Every PLU must still have its own canonical ID."
)


assert master_product_mapping[
    "PLUName_Original"
].notna().all()


assert master_product_mapping[
    "PLUName_Corrected"
].notna().all()


assert master_product_mapping[
    "CanonicalProductName"
].notna().all()


assert int(
    master_product_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == len(approved_spelling_corrections_df)


assert int(
    master_product_mapping[
        "MappingStatus"
    ].eq("PENDING_MANUAL_REVIEW").sum()
) == len(manual_review_candidates_df)


# ------------------------------------------------------------
# KIMBOX and KIMBOCK must remain separate
# ------------------------------------------------------------

kimbox_mapping = master_product_mapping.loc[
    master_product_mapping[
        "PLUName_Corrected"
    ].str.contains(
        "KIMBOX",
        case=False,
        na=False
    )
]


kimbock_mapping = master_product_mapping.loc[
    master_product_mapping[
        "PLUName_Corrected"
    ].str.contains(
        "KIMBOCK",
        case=False,
        na=False
    )
]


assert len(kimbox_mapping) > 0
assert len(kimbock_mapping) > 0


assert set(
    kimbox_mapping["CanonicalProductID"]
).isdisjoint(
    set(
        kimbock_mapping["CanonicalProductID"]
    )
), (
    "KIMBOX and KIMBOCK have been accidentally merged."
)


# ------------------------------------------------------------
# Temporarily join the mapping for validation only
# ------------------------------------------------------------

step1_validation_join = transactions_step1.merge(
    master_product_mapping[
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID",
            "CanonicalProductName",
            "MappingStatus",
        ]
    ],
    on="PLUCode",
    how="left",
    validate="many_to_one"
)


assert len(step1_validation_join) == (
    step1_original_row_count
), (
    "Joining the mapping changed the transaction row count."
)


assert step1_validation_join[
    "CanonicalProductID"
].notna().all(), (
    "At least one transaction has no product mapping."
)


assert int(
    step1_validation_join["UnitSold"].sum()
) == step1_original_unit_total, (
    "Joining the mapping changed the UnitSold total."
)


joined_value_cents = int(
    (
        step1_validation_join["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert joined_value_cents == (
    step1_original_value_cents
), (
    "Joining the mapping changed transaction value."
)


# ------------------------------------------------------------
# Create spelling-correction audit
# ------------------------------------------------------------

spelling_correction_audit = (
    master_product_mapping.loc[
        master_product_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION"),
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "GroupCode",
            "GroupName",
            "FirstObservedDate",
            "LastObservedDate",
            "TransactionRows",
            "TotalUnits",
            "MappingReason",
            "MappingConfidence",
            "MappingStatus",
        ],
    ]
    .sort_values(
        "PLUName_Original"
    )
    .reset_index(drop=True)
)


print("Step 1 integrity checks passed.")
print()
print(
    "Mapping rows:",
    len(master_product_mapping)
)
print(
    "Unique canonical IDs:",
    master_product_mapping[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Approved spelling corrections:",
    len(spelling_correction_audit)
)
print(
    "Pending manual reviews:",
    len(manual_review_candidates_df)
)
print(
    "Transaction rows unchanged:",
    f"{len(step1_validation_join):,}"
)
print(
    "UnitSold total unchanged:",
    f"{int(step1_validation_join['UnitSold'].sum()):,}"
)

print()
print("Mapping status summary:")

display(
    master_product_mapping[
        "MappingStatus"
    ]
    .value_counts()
    .rename_axis("MappingStatus")
    .reset_index(name="Products")
)

print()
print("Approved spelling corrections:")

display(spelling_correction_audit)

Step 1 integrity checks passed.

Mapping rows: 236
Unique canonical IDs: 236
Approved spelling corrections: 25
Pending manual reviews: 8
Transaction rows unchanged: 138,983
UnitSold total unchanged: 141,480

Mapping status summary:


,MappingStatus,Products
0,APPROVED_STEP_1,227
1,PENDING_MANUAL_REVIEW,8
2,EXCLUDE_FROM_FORECASTING,1



Approved spelling corrections:


,PLUCode,PLUName_Original,PLUName_Corrected,GroupCode,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,MappingReason,MappingConfidence,MappingStatus
0,42552,ALERNATIVE MILK,ALTERNATIVE MILK,1,HOT BEVS,2025-10-14,2026-03-30,839,839,Corrected ALERNATIVE to ALTERNATIVE.,HIGH,APPROVED_STEP_1
1,4241502,ALUXARY CROSSANT / PASTERIES,LUXURY CROISSANT / PASTRIES,4,BREAKFAST,2025-10-14,2026-03-30,654,654,"Corrected ALUXARY, CROSSANT and PASTERIES.",HIGH,APPROVED_STEP_1
2,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,1,HOT BEVS,2025-04-01,2025-10-13,517,517,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
3,4241536,BREAKSFAST POTS,BREAKFAST POTS,4,BREAKFAST,2025-04-01,2026-03-30,334,334,Corrected BREAKSFAST to BREAKFAST.,HIGH,APPROVED_STEP_1
4,4241460,CAPPUCINO 12OZ,CAPPUCCINO 12OZ,1,HOT BEVS,2025-04-01,2025-10-10,471,471,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
5,4241461,CAPPUCINO 16OZ,CAPPUCCINO 16OZ,1,HOT BEVS,2025-04-01,2025-08-29,179,179,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
6,4251513,CAPPUCINO LG,CAPPUCCINO LG,1,HOT BEVS,2025-04-02,2025-08-29,126,126,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
7,4251512,CAPPUCINO MED,CAPPUCCINO MED,1,HOT BEVS,2025-04-02,2025-09-03,189,189,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
8,4251511,CAPPUCINO SM,CAPPUCCINO SM,1,HOT BEVS,2025-04-01,2025-09-03,151,151,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1
9,4251515,CARAMEL FRAPP LG,CARAMEL FRAP LG,1,HOT BEVS,2025-04-01,2025-08-29,160,160,Standardised FRAPP to FRAP to match the associ...,HIGH,APPROVED_STEP_1


In [7]:
# ============================================================
# Step 1 - Cell 7
# Arrange and save controlled Step 1 outputs
# ============================================================

master_mapping_columns = [
    "PLUCode",
    "PLUName_Original",
    "PLUName_Corrected",
    "CanonicalProductID",
    "CanonicalProductName",
    "SupplierBrand",
    "ProductFamily",
    "PriceTier",
    "GroupCode",
    "GroupName",
    "FirstObservedDate",
    "LastObservedDate",
    "TransactionRows",
    "TotalUnits",
    "TotalTransactionValue",
    "MappingType",
    "MappingReason",
    "MappingConfidence",
    "MappingStatus",
    "MappingVersion",
]


master_product_mapping = (
    master_product_mapping[
        master_mapping_columns
    ]
    .sort_values(
        [
            "PLUName_Corrected",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


# Add observed product information to the manual-review file
manual_review_output = (
    manual_review_candidates_df
    .merge(
        master_product_mapping[
            [
                "PLUCode",
                "PLUName_Original",
                "GroupCode",
                "GroupName",
                "FirstObservedDate",
                "LastObservedDate",
                "TransactionRows",
                "TotalUnits",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="one_to_one"
    )
)


manual_review_output = manual_review_output[
    [
        "PLUCode",
        "PLUName_Original",
        "SuggestedName",
        "GroupCode",
        "GroupName",
        "FirstObservedDate",
        "LastObservedDate",
        "TransactionRows",
        "TotalUnits",
        "PotentialIssue",
        "ReviewConfidence",
    ]
]


# ------------------------------------------------------------
# Save only mapping and audit files
# The transaction dataset is not modified or resaved.
# ------------------------------------------------------------

master_product_mapping.to_csv(
    MASTER_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


spelling_correction_audit.to_csv(
    SPELLING_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


manual_review_output.to_csv(
    MANUAL_REVIEW_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Step 1 files saved successfully.")
print()

print("1. Master product mapping:")
print(MASTER_MAPPING_OUTPUT_FILE)

print()
print("2. Approved spelling-correction audit:")
print(SPELLING_AUDIT_OUTPUT_FILE)

print()
print("3. Manual-review candidates:")
print(MANUAL_REVIEW_OUTPUT_FILE)

Step 1 files saved successfully.

1. Master product mapping:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling.csv

2. Approved spelling-correction audit:
eden_datasets/UL_EDEN_product_spelling_corrections_step1_audit.csv

3. Manual-review candidates:
eden_datasets/UL_EDEN_product_name_manual_review_candidates_step1.csv


In [8]:
# ============================================================
# Step 1 - Cell 8
# Reload and validate saved Step 1 mapping
# ============================================================

saved_master_mapping = pd.read_csv(
    MASTER_MAPPING_OUTPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


saved_master_mapping["PLUCode"] = (
    pd.to_numeric(
        saved_master_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_master_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_master_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


assert len(saved_master_mapping) == 236


assert saved_master_mapping[
    "PLUCode"
].is_unique


assert saved_master_mapping[
    "CanonicalProductID"
].is_unique, (
    "The saved Step 1 mapping contains an accidental merge."
)


assert saved_master_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    saved_master_mapping["TotalUnits"].sum()
) == 141_480


assert int(
    saved_master_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 25


assert int(
    saved_master_mapping[
        "MappingStatus"
    ].eq("PENDING_MANUAL_REVIEW").sum()
) == 8


saved_open_ul = saved_master_mapping.loc[
    saved_master_mapping[
        "PLUName_Original"
    ].astype("string").str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
]


assert len(saved_open_ul) == 1


assert (
    saved_open_ul[
        "MappingStatus"
    ].iloc[0]
    == "EXCLUDE_FROM_FORECASTING"
)


print("=" * 72)
print("STEP 1 MASTER PRODUCT MAPPING COMPLETED")
print("=" * 72)
print()
print(
    "PLU mapping rows:",
    len(saved_master_mapping)
)
print(
    "Approved spelling corrections:",
    int(
        saved_master_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Pending manual reviews:",
    int(
        saved_master_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print(
    "Accidental product merges:",
    236 - saved_master_mapping[
        "CanonicalProductID"
    ].nunique()
)
print(
    "Mapped total units:",
    f"{int(saved_master_mapping['TotalUnits'].sum()):,}"
)

display(
    saved_master_mapping.loc[
        saved_master_mapping[
            "MappingType"
        ].ne("UNCHANGED")
    ]
)

STEP 1 MASTER PRODUCT MAPPING COMPLETED

PLU mapping rows: 236
Approved spelling corrections: 25
Pending manual reviews: 8
Accidental product merges: 0
Mapped total units: 141,480


,PLUCode,PLUName_Original,PLUName_Corrected,CanonicalProductID,CanonicalProductName,SupplierBrand,ProductFamily,PriceTier,GroupCode,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,TotalTransactionValue,MappingType,MappingReason,MappingConfidence,MappingStatus,MappingVersion
3,42552,ALERNATIVE MILK,ALTERNATIVE MILK,PLU_42552,ALTERNATIVE MILK,NaN,NaN,NaN,1,HOT BEVS,2025-10-14,2026-03-30,839,839,419.50,SPELLING_CORRECTION,Corrected ALERNATIVE to ALTERNATIVE.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
13,2000000269,BALLYGOWAN SPARK WATER 330ML,BALLYGOWAN SPARK WATER 330ML,PLU_2000000269,BALLYGOWAN SPARK WATER 330ML,NaN,NaN,NaN,2,COLD BEVS,2025-04-01,2025-10-02,47,47,117.50,MANUAL_REVIEW,SPARK may be an abbreviation for SPARKLING.,MEDIUM,PENDING_MANUAL_REVIEW,STEP_1_SPELLING_V1
17,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,PLU_42527103,BEWLEYS CAPPUCCINO,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-10-13,517,517,1614.80,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
27,4241536,BREAKSFAST POTS,BREAKFAST POTS,PLU_4241536,BREAKFAST POTS,NaN,NaN,NaN,4,BREAKFAST,2025-04-01,2026-03-30,334,334,835.00,SPELLING_CORRECTION,Corrected BREAKSFAST to BREAKFAST.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
35,4241460,CAPPUCINO 12OZ,CAPPUCCINO 12OZ,PLU_4241460,CAPPUCCINO 12OZ,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-10-10,471,471,1799.10,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
36,4241461,CAPPUCINO 16OZ,CAPPUCCINO 16OZ,PLU_4241461,CAPPUCCINO 16OZ,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-08-29,179,179,769.70,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
37,4251513,CAPPUCINO LG,CAPPUCCINO LG,PLU_4251513,CAPPUCCINO LG,NaN,NaN,NaN,1,HOT BEVS,2025-04-02,2025-08-29,126,126,579.60,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
38,4251512,CAPPUCINO MED,CAPPUCCINO MED,PLU_4251512,CAPPUCCINO MED,NaN,NaN,NaN,1,HOT BEVS,2025-04-02,2025-09-03,189,189,812.70,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
39,4251511,CAPPUCINO SM,CAPPUCCINO SM,PLU_4251511,CAPPUCCINO SM,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-09-03,151,151,581.35,SPELLING_CORRECTION,Corrected CAPPUCINO to CAPPUCCINO.,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1
40,4251515,CARAMEL FRAPP LG,CARAMEL FRAP LG,PLU_4251515,CARAMEL FRAP LG,NaN,NaN,NaN,1,HOT BEVS,2025-04-01,2025-08-29,160,160,856.00,SPELLING_CORRECTION,Standardised FRAPP to FRAP to match the associ...,HIGH,APPROVED_STEP_1,STEP_1_SPELLING_V1


### I am not changing any of the manual review needed names because it is not necessary.

In [10]:
# ============================================================
# Step 1 - Final Lock Cell
# Close the eight reviewed names without changing them
# ============================================================

from pathlib import Path

import pandas as pd


DATA_FOLDER = Path("eden_datasets")


STEP1_DRAFT_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling.csv"
)


STEP1_LOCKED_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_locked.csv"
)


step1_locked_mapping = pd.read_csv(
    STEP1_DRAFT_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


reviewed_no_change_codes = {
    4241404: "ODONNELL /POPCORN",
    2024241: "JOS POWERBALL",
    2000000269: "BALLYGOWAN SPARK WATER 330ML",
    4241537: "CRUSTED CROISSANTS",
    4241455: "ICED VANILLA 16OZ",
    4241405: "HUNKY DORY",
    4241409: "PACKS OF SWEET - WINE GUMS",
    4241415: "LUXURY MILLIONAIRES FLAPJACK",
}


for plu_code, expected_name in reviewed_no_change_codes.items():

    row_mask = (
        step1_locked_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(row_mask.sum()) == 1, (
        f"Expected exactly one row for PLUCode {plu_code}."
    )


    actual_name = (
        step1_locked_mapping.loc[
            row_mask,
            "PLUName_Original"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    assert actual_name == expected_name.upper(), (
        f"Unexpected product name for PLUCode {plu_code}.\n"
        f"Expected: {expected_name}\n"
        f"Found: {actual_name}"
    )


    # The product name remains unchanged.
    step1_locked_mapping.loc[
        row_mask,
        "PLUName_Corrected"
    ] = actual_name


    step1_locked_mapping.loc[
        row_mask,
        "CanonicalProductName"
    ] = actual_name


    step1_locked_mapping.loc[
        row_mask,
        "MappingType"
    ] = "REVIEWED_NO_CHANGE"


    step1_locked_mapping.loc[
        row_mask,
        "MappingReason"
    ] = (
        "Name is consistently recorded throughout the dataset "
        "and the suggested alternative cannot be confirmed."
    )


    step1_locked_mapping.loc[
        row_mask,
        "MappingConfidence"
    ] = "REVIEWED"


    step1_locked_mapping.loc[
        row_mask,
        "MappingStatus"
    ] = "APPROVED_STEP_1"


    step1_locked_mapping.loc[
        row_mask,
        "MappingVersion"
    ] = "STEP_1_SPELLING_FINAL"


# ------------------------------------------------------------
# Final Step 1 validation
# ------------------------------------------------------------

assert len(step1_locked_mapping) == 236

assert step1_locked_mapping[
    "PLUCode"
].is_unique

assert step1_locked_mapping[
    "CanonicalProductID"
].is_unique

assert step1_locked_mapping[
    "CanonicalProductID"
].nunique() == 236

assert not step1_locked_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any(), (
    "Step 1 still contains pending manual-review products."
)

assert int(
    step1_locked_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 25

assert int(
    step1_locked_mapping[
        "MappingType"
    ].eq("REVIEWED_NO_CHANGE").sum()
) == 8

assert int(
    step1_locked_mapping["TotalUnits"].sum()
) == 141_480


step1_locked_mapping.to_csv(
    STEP1_LOCKED_MAPPING_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("=" * 70)
print("STEP 1 LOCKED SUCCESSFULLY")
print("=" * 70)
print()
print("Mapping rows:", len(step1_locked_mapping))
print(
    "Spelling corrections:",
    int(
        step1_locked_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Reviewed with no change:",
    int(
        step1_locked_mapping[
            "MappingType"
        ].eq("REVIEWED_NO_CHANGE").sum()
    )
)
print(
    "Pending reviews:",
    int(
        step1_locked_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print(
    "Canonical product IDs:",
    step1_locked_mapping[
        "CanonicalProductID"
    ].nunique()
)
print()
print("Locked Step 1 mapping:")
print(STEP1_LOCKED_MAPPING_FILE)

STEP 1 LOCKED SUCCESSFULLY

Mapping rows: 236
Spelling corrections: 25
Reviewed with no change: 8
Pending reviews: 0
Canonical product IDs: 236

Locked Step 1 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling_locked.csv


## Next is to normalise the branded beverags like there was Bewleys ameriano there was RT americano. I am going to make them all to one but keep the tea 16 oz efor example as it is or change it to standard

In [11]:
# ============================================================
# Step 2 - Cell 0
# Finalise Step 1 by correcting RN CAPPUCINO
#
# This creates a new Step 1 final file.
# It does not overwrite the existing locked mapping.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


STEP1_LOCKED_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_locked.csv"
)


STEP1_FINAL_MAPPING_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step1_spelling_final.csv"
)


if not STEP1_LOCKED_MAPPING_FILE.exists():
    raise FileNotFoundError(
        "The locked Step 1 mapping was not found:\n"
        f"{STEP1_LOCKED_MAPPING_FILE}"
    )


step1_final_mapping = pd.read_csv(
    STEP1_LOCKED_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


step1_final_mapping["PLUCode"] = (
    pd.to_numeric(
        step1_final_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step1_final_mapping["TotalUnits"] = (
    pd.to_numeric(
        step1_final_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Validate the locked Step 1 mapping
# ------------------------------------------------------------

assert len(step1_final_mapping) == 236, (
    "Step 1 mapping should contain 236 PLU codes."
)


assert step1_final_mapping["PLUCode"].is_unique, (
    "PLUCode is not unique in the Step 1 mapping."
)


assert step1_final_mapping[
    "CanonicalProductID"
].is_unique, (
    "Step 1 should not contain any product merges."
)


assert not step1_final_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any(), (
    "Step 1 still contains pending manual-review products."
)


assert int(
    step1_final_mapping["TotalUnits"].sum()
) == 141_480, (
    "Step 1 mapping UnitSold total is incorrect."
)


# ------------------------------------------------------------
# Correct the one missed definite spelling error
# ------------------------------------------------------------

rn_cappuccino_plu = 44382


rn_cappuccino_mask = (
    step1_final_mapping["PLUCode"]
    .eq(rn_cappuccino_plu)
)


assert int(rn_cappuccino_mask.sum()) == 1, (
    "Expected exactly one mapping row for PLUCode 44382."
)


actual_original_name = (
    step1_final_mapping.loc[
        rn_cappuccino_mask,
        "PLUName_Original"
    ]
    .iloc[0]
    .strip()
    .upper()
)


assert actual_original_name == "RN CAPPUCINO", (
    "Unexpected original name for PLUCode 44382.\n"
    f"Found: {actual_original_name}"
)


current_corrected_name = (
    step1_final_mapping.loc[
        rn_cappuccino_mask,
        "PLUName_Corrected"
    ]
    .iloc[0]
    .strip()
    .upper()
)


assert current_corrected_name in {
    "RN CAPPUCINO",
    "RN CAPPUCCINO",
}, (
    "Unexpected corrected name for PLUCode 44382.\n"
    f"Found: {current_corrected_name}"
)


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "PLUName_Corrected"
] = "RN CAPPUCCINO"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "CanonicalProductName"
] = "RN CAPPUCCINO"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingType"
] = "SPELLING_CORRECTION"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingReason"
] = (
    "Corrected CAPPUCINO to CAPPUCCINO."
)


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingConfidence"
] = "HIGH"


step1_final_mapping.loc[
    rn_cappuccino_mask,
    "MappingStatus"
] = "APPROVED_STEP_1"


# Apply one consistent version to the complete final mapping
step1_final_mapping[
    "MappingVersion"
] = "STEP_1_SPELLING_FINAL_V2"


# ------------------------------------------------------------
# Validate final Step 1 state
# ------------------------------------------------------------

assert int(
    step1_final_mapping[
        "MappingType"
    ].eq("SPELLING_CORRECTION").sum()
) == 26, (
    "The final Step 1 mapping should contain "
    "26 definite spelling corrections."
)


assert int(
    step1_final_mapping[
        "MappingType"
    ].eq("REVIEWED_NO_CHANGE").sum()
) == 8, (
    "Expected eight reviewed names retained unchanged."
)


assert step1_final_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    step1_final_mapping["TotalUnits"].sum()
) == 141_480


step1_final_mapping.to_csv(
    STEP1_FINAL_MAPPING_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("=" * 72)
print("STEP 1 FINAL MAPPING CREATED")
print("=" * 72)
print()
print(
    "Mapping rows:",
    len(step1_final_mapping)
)
print(
    "Spelling corrections:",
    int(
        step1_final_mapping[
            "MappingType"
        ].eq("SPELLING_CORRECTION").sum()
    )
)
print(
    "Reviewed without change:",
    int(
        step1_final_mapping[
            "MappingType"
        ].eq("REVIEWED_NO_CHANGE").sum()
    )
)
print(
    "Pending reviews:",
    int(
        step1_final_mapping[
            "MappingStatus"
        ].eq("PENDING_MANUAL_REVIEW").sum()
    )
)
print()
print("Final Step 1 file:")
print(STEP1_FINAL_MAPPING_FILE)

STEP 1 FINAL MAPPING CREATED

Mapping rows: 236
Spelling corrections: 26
Reviewed without change: 8
Pending reviews: 0

Final Step 1 file:
eden_datasets/UL_EDEN_master_product_mapping_step1_spelling_final.csv


In [12]:
# ============================================================
# Step 2 - Cell 1
# Load final Step 1 mapping and corrected transactions
# ============================================================

TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


STEP2_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step2_branded_beverages.csv"
)


STEP2_PLU_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_plu_audit.csv"
)


STEP2_CANONICAL_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_canonical_summary.csv"
)


STEP2_SUPPLIER_PERIOD_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step2_branded_beverage_supplier_periods.csv"
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction file not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


if not STEP1_FINAL_MAPPING_FILE.exists():
    raise FileNotFoundError(
        "Final Step 1 mapping not found:\n"
        f"{STEP1_FINAL_MAPPING_FILE}"
    )


step2_transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


step2_mapping = pd.read_csv(
    STEP1_FINAL_MAPPING_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


# Preserve exact source-column order for later validation
step2_source_columns = (
    step2_transactions.columns.tolist()
)


# ------------------------------------------------------------
# Convert transaction columns safely
# ------------------------------------------------------------

step2_transactions["TransDate"] = pd.to_datetime(
    step2_transactions["TransDate"],
    errors="raise"
)


step2_transactions["Date"] = (
    pd.to_datetime(
        step2_transactions["Date"],
        errors="raise"
    )
    .dt.normalize()
)


step2_transactions["PLUCode"] = (
    pd.to_numeric(
        step2_transactions["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step2_transactions["UnitSold"] = (
    pd.to_numeric(
        step2_transactions["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step2_transactions["TransValue"] = pd.to_numeric(
    step2_transactions["TransValue"],
    errors="raise"
)


step2_mapping["PLUCode"] = (
    pd.to_numeric(
        step2_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step2_mapping["TotalUnits"] = (
    pd.to_numeric(
        step2_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Record immutable source totals
# ------------------------------------------------------------

step2_source_row_count = len(
    step2_transactions
)


step2_source_unit_total = int(
    step2_transactions["UnitSold"].sum()
)


step2_source_value_cents = int(
    (
        step2_transactions["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate inputs
# ------------------------------------------------------------

assert step2_source_row_count == 138_983


assert step2_source_unit_total == 141_480


assert step2_transactions[
    "PLUCode"
].nunique() == 236


assert len(step2_mapping) == 236


assert step2_mapping[
    "PLUCode"
].is_unique


assert step2_mapping[
    "CanonicalProductID"
].is_unique


assert step2_mapping[
    "CanonicalProductID"
].nunique() == 236


assert int(
    step2_mapping["TotalUnits"].sum()
) == 141_480


assert not step2_mapping[
    "MappingStatus"
].eq("PENDING_MANUAL_REVIEW").any()


print("Step 2 source files loaded and validated.")
print()
print(
    "Transaction rows:",
    f"{step2_source_row_count:,}"
)
print(
    "Transaction units:",
    f"{step2_source_unit_total:,}"
)
print(
    "Step 1 PLU identities:",
    step2_mapping[
        "CanonicalProductID"
    ].nunique()
)

Step 2 source files loaded and validated.

Transaction rows: 138,983
Transaction units: 141,480
Step 1 PLU identities: 236


In [13]:
# ============================================================
# Step 2 - Cell 2
# Verify one consistent product identity per PLUCode
# ============================================================

source_plu_consistency = (
    step2_transactions
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        SourceNameCount=(
            "PLUName",
            "nunique"
        ),
        SourceGroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        SourceGroupNameCount=(
            "GroupName",
            "nunique"
        ),
        SourcePLUName=(
            "PLUName",
            "first"
        ),
        SourceGroupCode=(
            "GroupCode",
            "first"
        ),
        SourceGroupName=(
            "GroupName",
            "first"
        ),
        SourceRows=(
            "PLUCode",
            "size"
        ),
        SourceUnits=(
            "UnitSold",
            "sum"
        ),
    )
)


assert source_plu_consistency[
    "SourceNameCount"
].eq(1).all(), (
    "At least one PLUCode has multiple product names."
)


assert source_plu_consistency[
    "SourceGroupCodeCount"
].eq(1).all(), (
    "At least one PLUCode has multiple GroupCode values."
)


assert source_plu_consistency[
    "SourceGroupNameCount"
].eq(1).all(), (
    "At least one PLUCode has multiple GroupName values."
)


mapping_source_check = step2_mapping[
    [
        "PLUCode",
        "PLUName_Original",
        "GroupCode",
        "GroupName",
        "TotalUnits",
    ]
].merge(
    source_plu_consistency,
    on="PLUCode",
    how="outer",
    validate="one_to_one"
)


assert len(mapping_source_check) == 236


assert mapping_source_check[
    "PLUName_Original"
].notna().all()


assert mapping_source_check[
    "SourcePLUName"
].notna().all()


name_matches = (
    mapping_source_check[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    ==
    mapping_source_check[
        "SourcePLUName"
    ]
    .astype("string")
    .str.strip()
)


assert name_matches.all(), (
    "A Step 1 original product name does not match "
    "the transaction source."
)


assert (
    mapping_source_check["GroupCode"]
    ==
    mapping_source_check["SourceGroupCode"]
).all()


assert (
    mapping_source_check["GroupName"]
    .astype("string")
    .str.strip()
    ==
    mapping_source_check["SourceGroupName"]
    .astype("string")
    .str.strip()
).all()


assert (
    mapping_source_check["TotalUnits"]
    ==
    mapping_source_check["SourceUnits"]
).all(), (
    "A PLU-level unit total differs between the mapping "
    "and the source transaction dataset."
)


print("Step 1 mapping matches the transaction source exactly.")

Step 1 mapping matches the transaction source exactly.


In [14]:
# ============================================================
# Step 2 - Cell 3
# Create isolated Step 2 product-identity columns
# ============================================================

# Preserve the final Step 1 identity explicitly
step2_mapping[
    "CanonicalProductID_Step1"
] = step2_mapping[
    "CanonicalProductID"
].astype("string")


step2_mapping[
    "CanonicalProductName_Step1"
] = step2_mapping[
    "CanonicalProductName"
].astype("string")


# Step 2 initially equals Step 1 for every product
step2_mapping[
    "CanonicalProductID_Step2"
] = step2_mapping[
    "CanonicalProductID_Step1"
].copy()


step2_mapping[
    "CanonicalProductName_Step2"
] = step2_mapping[
    "CanonicalProductName_Step1"
].copy()


# New Step 2 metadata
step2_mapping[
    "BeverageSeries_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "BeverageType_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "SupplierLabel_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "ProductFamily_Step2"
] = pd.Series(
    pd.NA,
    index=step2_mapping.index,
    dtype="string"
)


step2_mapping[
    "SupplierNormalizationApplied_Step2"
] = False


step2_mapping[
    "SupplierNormalizationType_Step2"
] = "NOT_APPLICABLE"


step2_mapping[
    "SupplierNormalizationReason_Step2"
] = (
    "Product identity retained from Step 1."
)


step2_mapping[
    "SupplierNormalizationStatus_Step2"
] = "UNCHANGED_STEP_2"


step2_mapping[
    "SupplierNormalizationVersion_Step2"
] = (
    "STEP_2_BRANDED_BEVERAGES_V1"
)


print("Isolated Step 2 columns created.")

Isolated Step 2 columns created.


In [15]:
# ============================================================
# Step 2 - Cell 4
# Define confirmed branded-beverage mappings
# ============================================================

branded_beverage_mappings = [
    # --------------------------------------------------------
    # B-AMERICANO
    # --------------------------------------------------------
    {
        "PLUCode": 42527101,
        "ExpectedCorrectedName": "BEWLEYS AMERICANO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44381,
        "ExpectedCorrectedName": "RN AMERICANO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "RN",
    },
    {
        "PLUCode": 425531,
        "ExpectedCorrectedName":
            "ROASTED NOTES AMERICANO CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_AMERICANO",
        "CanonicalProductName_Step2":
            "B-AMERICANO",
        "BeverageType_Step2": "AMERICANO",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-CAPPUCCINO
    # --------------------------------------------------------
    {
        "PLUCode": 42527103,
        "ExpectedCorrectedName":
            "BEWLEYS CAPPUCCINO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_CAPPUCCINO",
        "CanonicalProductName_Step2":
            "B-CAPPUCCINO",
        "BeverageType_Step2": "CAPPUCCINO",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44382,
        "ExpectedCorrectedName":
            "RN CAPPUCCINO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_CAPPUCCINO",
        "CanonicalProductName_Step2":
            "B-CAPPUCCINO",
        "BeverageType_Step2": "CAPPUCCINO",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-FLAT WHITE
    # --------------------------------------------------------
    {
        "PLUCode": 42527104,
        "ExpectedCorrectedName":
            "BEWLEYS FLAT WHITE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FLAT_WHITE",
        "CanonicalProductName_Step2":
            "B-FLAT WHITE",
        "BeverageType_Step2": "FLAT WHITE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 417173,
        "ExpectedCorrectedName":
            "RN FLAT WHITE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FLAT_WHITE",
        "CanonicalProductName_Step2":
            "B-FLAT WHITE",
        "BeverageType_Step2": "FLAT WHITE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-LATTE
    # --------------------------------------------------------
    {
        "PLUCode": 42527102,
        "ExpectedCorrectedName":
            "BEWLEYS LATTE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_LATTE",
        "CanonicalProductName_Step2":
            "B-LATTE",
        "BeverageType_Step2": "LATTE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44384,
        "ExpectedCorrectedName":
            "RN LATTE",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_LATTE",
        "CanonicalProductName_Step2":
            "B-LATTE",
        "BeverageType_Step2": "LATTE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-MOCHA
    # --------------------------------------------------------
    {
        "PLUCode": 42527105,
        "ExpectedCorrectedName":
            "BEWLEYS MOCHA",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_MOCHA",
        "CanonicalProductName_Step2":
            "B-MOCHA",
        "BeverageType_Step2": "MOCHA",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44385,
        "ExpectedCorrectedName":
            "RN MOCHA",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_MOCHA",
        "CanonicalProductName_Step2":
            "B-MOCHA",
        "BeverageType_Step2": "MOCHA",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-HOT CHOCOLATE
    # --------------------------------------------------------
    {
        "PLUCode": 42527106,
        "ExpectedCorrectedName":
            "BEWLEYS HOT CHOC",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_HOT_CHOCOLATE",
        "CanonicalProductName_Step2":
            "B-HOT CHOCOLATE",
        "BeverageType_Step2": "HOT CHOCOLATE",
        "SupplierLabel_Step2": "BEWLEYS",
    },
    {
        "PLUCode": 44386,
        "ExpectedCorrectedName":
            "RN HOT CHOC",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_HOT_CHOCOLATE",
        "CanonicalProductName_Step2":
            "B-HOT CHOCOLATE",
        "BeverageType_Step2": "HOT CHOCOLATE",
        "SupplierLabel_Step2": "RN",
    },

    # --------------------------------------------------------
    # B-ESPRESSO
    # --------------------------------------------------------
    {
        "PLUCode": 4241523,
        "ExpectedCorrectedName":
            "ROASTED NOTES ESPRESSO",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_ESPRESSO",
        "CanonicalProductName_Step2":
            "B-ESPRESSO",
        "BeverageType_Step2": "ESPRESSO",
        "SupplierLabel_Step2": "RN",
    },
    {
        "PLUCode": 42555,
        "ExpectedCorrectedName":
            "ROASTED NOTES ESPRESSO CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_ESPRESSO",
        "CanonicalProductName_Step2":
            "B-ESPRESSO",
        "BeverageType_Step2": "ESPRESSO",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-FILTER COFFEE
    # --------------------------------------------------------
    {
        "PLUCode": 42563,
        "ExpectedCorrectedName":
            "FILTER COFFEE CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_FILTER_COFFEE",
        "CanonicalProductName_Step2":
            "B-FILTER COFFEE",
        "BeverageType_Step2": "FILTER COFFEE",
        "SupplierLabel_Step2": "CT",
    },

    # --------------------------------------------------------
    # B-TEA
    # --------------------------------------------------------
    {
        "PLUCode": 42561,
        "ExpectedCorrectedName":
            "TEA CT",
        "CanonicalProductID_Step2":
            "BEV_BRANDED_TEA",
        "CanonicalProductName_Step2":
            "B-TEA",
        "BeverageType_Step2": "TEA",
        "SupplierLabel_Step2": "CT",
    },
]


branded_beverage_mapping_df = pd.DataFrame(
    branded_beverage_mappings
)


assert len(branded_beverage_mapping_df) == 17


assert branded_beverage_mapping_df[
    "PLUCode"
].is_unique


assert branded_beverage_mapping_df[
    "CanonicalProductID_Step2"
].nunique() == 9


assert branded_beverage_mapping_df[
    "CanonicalProductName_Step2"
].nunique() == 9


approved_branded_plu_codes = set(
    branded_beverage_mapping_df["PLUCode"]
)


print("Confirmed branded-beverage mapping created.")
print()
print(
    "Source PLUs:",
    len(branded_beverage_mapping_df)
)
print(
    "Canonical branded beverages:",
    branded_beverage_mapping_df[
        "CanonicalProductID_Step2"
    ].nunique()
)

display(
    branded_beverage_mapping_df.sort_values(
        [
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
        ]
    )
)

Confirmed branded-beverage mapping created.

Source PLUs: 17
Canonical branded beverages: 9


,PLUCode,ExpectedCorrectedName,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageType_Step2,SupplierLabel_Step2
0,42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,BEWLEYS
2,425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,CT
1,44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,RN
3,42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,BEWLEYS
4,44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,RN
14,42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,CT
13,4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,RN
15,42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,FILTER COFFEE,CT
5,42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,BEWLEYS
6,417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,RN


In [16]:
# ============================================================
# Step 2 - Cell 5
# Validate every proposed mapping before applying it
# ============================================================

source_branded_transactions = (
    step2_transactions.loc[
        step2_transactions[
            "PLUCode"
        ].isin(approved_branded_plu_codes)
    ]
    .copy()
)


assert source_branded_transactions[
    "PLUCode"
].nunique() == 17


assert len(source_branded_transactions) == 12_165


assert int(
    source_branded_transactions["UnitSold"].sum()
) == 12_165


assert source_branded_transactions[
    "GroupName"
].eq("HOT BEVS").all(), (
    "At least one approved branded PLU is not in HOT BEVS."
)


for proposed_mapping in branded_beverage_mappings:

    plu_code = proposed_mapping["PLUCode"]

    mapping_row_mask = (
        step2_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(mapping_row_mask.sum()) == 1, (
        f"Expected one Step 1 mapping row for PLU {plu_code}."
    )


    actual_corrected_name = (
        step2_mapping.loc[
            mapping_row_mask,
            "PLUName_Corrected"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_corrected_name = (
        proposed_mapping[
            "ExpectedCorrectedName"
        ]
        .strip()
        .upper()
    )


    assert actual_corrected_name == (
        expected_corrected_name
    ), (
        f"Corrected-name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_corrected_name}\n"
        f"Found: {actual_corrected_name}"
    )


    source_product_rows = (
        step2_transactions.loc[
            step2_transactions[
                "PLUCode"
            ].eq(plu_code)
        ]
    )


    assert len(source_product_rows) > 0


    assert source_product_rows[
        "PLUName"
    ].nunique() == 1


    assert source_product_rows[
        "GroupName"
    ].eq("HOT BEVS").all()


print("All 17 proposed branded-beverage mappings passed preflight.")
print()
print(
    "Transaction rows covered:",
    f"{len(source_branded_transactions):,}"
)
print(
    "Units covered:",
    f"{int(source_branded_transactions['UnitSold'].sum()):,}"
)

All 17 proposed branded-beverage mappings passed preflight.

Transaction rows covered: 12,165
Units covered: 12,165


In [17]:
# ============================================================
# Step 2 - Cell 6
# Apply mappings only to the 17 confirmed branded PLUs
# ============================================================

for approved_mapping in branded_beverage_mappings:

    plu_code = approved_mapping["PLUCode"]

    row_mask = (
        step2_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(row_mask.sum()) == 1


    step2_mapping.loc[
        row_mask,
        "CanonicalProductID_Step2"
    ] = approved_mapping[
        "CanonicalProductID_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "CanonicalProductName_Step2"
    ] = approved_mapping[
        "CanonicalProductName_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "BeverageSeries_Step2"
    ] = "BRANDED"


    step2_mapping.loc[
        row_mask,
        "BeverageType_Step2"
    ] = approved_mapping[
        "BeverageType_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "SupplierLabel_Step2"
    ] = approved_mapping[
        "SupplierLabel_Step2"
    ]


    step2_mapping.loc[
        row_mask,
        "ProductFamily_Step2"
    ] = "HOT BEVERAGE"


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationApplied_Step2"
    ] = True


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationType_Step2"
    ] = "CONFIRMED_BRANDED_EQUIVALENCE"


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationReason_Step2"
    ] = (
        "Confirmed branded beverage equivalent sold "
        "under Bewleys, RN or CT supplier labels."
    )


    step2_mapping.loc[
        row_mask,
        "SupplierNormalizationStatus_Step2"
    ] = "APPROVED_STEP_2"


print(
    "Branded PLU mappings applied:",
    int(
        step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ].sum()
    )
)

Branded PLU mappings applied: 17


In [18]:
# ============================================================
# Step 2 - Cell 7
# Validate isolation and prevent accidental product merging
# ============================================================

step2_changed_identity_mask = (
    step2_mapping[
        "CanonicalProductID_Step2"
    ]
    !=
    step2_mapping[
        "CanonicalProductID_Step1"
    ]
)


changed_plu_codes = set(
    step2_mapping.loc[
        step2_changed_identity_mask,
        "PLUCode",
    ]
)


assert changed_plu_codes == approved_branded_plu_codes, (
    "A non-approved product was changed in Step 2."
)


assert int(
    step2_changed_identity_mask.sum()
) == 17


# ------------------------------------------------------------
# Every other product must remain exactly as Step 1
# ------------------------------------------------------------

unchanged_step2_mask = (
    ~step2_mapping[
        "PLUCode"
    ].isin(approved_branded_plu_codes)
)


assert (
    step2_mapping.loc[
        unchanged_step2_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            unchanged_step2_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A non-branded product canonical ID changed."
)


assert (
    step2_mapping.loc[
        unchanged_step2_mask,
        "CanonicalProductName_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            unchanged_step2_mask,
            "CanonicalProductName_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A non-branded product canonical name changed."
)


# ------------------------------------------------------------
# Standard HOT BEVS must remain unchanged
# ------------------------------------------------------------

standard_hot_beverage_mask = (
    step2_mapping["GroupName"].eq("HOT BEVS")
    &
    ~step2_mapping[
        "PLUCode"
    ].isin(approved_branded_plu_codes)
)


assert (
    step2_mapping.loc[
        standard_hot_beverage_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            standard_hot_beverage_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
), (
    "A standard restaurant beverage was accidentally merged."
)


# ------------------------------------------------------------
# Expected identity counts
# ------------------------------------------------------------

assert step2_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert step2_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228, (
    "Expected 228 canonical identities after merging "
    "17 source PLUs into nine branded beverages."
)


assert step2_mapping.loc[
    step2_changed_identity_mask,
    "CanonicalProductName_Step2"
].str.startswith("B-").all()


assert step2_mapping.loc[
    step2_changed_identity_mask,
    "BeverageSeries_Step2"
].eq("BRANDED").all()


# Confirm KIMBOX and KIMBOCK were untouched
kim_product_mask = (
    step2_mapping["PLUName_Corrected"]
    .astype("string")
    .str.contains(
        r"KIMBOX|KIMBOCK",
        case=False,
        na=False,
        regex=True
    )
)


assert (
    step2_mapping.loc[
        kim_product_mask,
        "CanonicalProductID_Step2"
    ]
    .reset_index(drop=True)
    .equals(
        step2_mapping.loc[
            kim_product_mask,
            "CanonicalProductID_Step1"
        ]
        .reset_index(drop=True)
    )
)


print("Step 2 isolation checks passed.")
print()
print(
    "Step 1 canonical identities:",
    step2_mapping[
        "CanonicalProductID_Step1"
    ].nunique()
)
print(
    "Step 2 canonical identities:",
    step2_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)
print(
    "PLUs changed:",
    int(step2_changed_identity_mask.sum())
)
print(
    "Standard HOT BEVS changed:",
    int(
        (
            standard_hot_beverage_mask
            &
            step2_changed_identity_mask
        ).sum()
    )
)

Step 2 isolation checks passed.

Step 1 canonical identities: 236
Step 2 canonical identities: 228
PLUs changed: 17
Standard HOT BEVS changed: 0


In [19]:
# ============================================================
# Step 2 - Cell 8
# Create supplier-period and canonical-beverage audits
# ============================================================

branded_transaction_audit = (
    source_branded_transactions
    .merge(
        branded_beverage_mapping_df[
            [
                "PLUCode",
                "CanonicalProductID_Step2",
                "CanonicalProductName_Step2",
                "BeverageType_Step2",
                "SupplierLabel_Step2",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one"
    )
)


supplier_period_summary = (
    branded_transaction_audit
    .groupby(
        "SupplierLabel_Step2",
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
    )
    .sort_values(
        "FirstObservedDate"
    )
    .reset_index(drop=True)
)


canonical_branded_summary = (
    branded_transaction_audit
    .groupby(
        [
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "BeverageType_Step2",
        ],
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
    )
    .sort_values(
        "TotalUnits",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate observed supplier periods
# ------------------------------------------------------------

bewleys_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("BEWLEYS")
]


rn_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("RN")
]


ct_row = supplier_period_summary.loc[
    supplier_period_summary[
        "SupplierLabel_Step2"
    ].eq("CT")
]


assert len(bewleys_row) == 1
assert len(rn_row) == 1
assert len(ct_row) == 1


assert (
    bewleys_row["LastObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-13")
)


assert (
    rn_row["FirstObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-14")
)


assert (
    ct_row["FirstObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-28")
)


assert (
    ct_row["LastObservedDate"].iloc[0]
    == pd.Timestamp("2025-10-28")
)


assert len(canonical_branded_summary) == 9


assert int(
    canonical_branded_summary["TotalUnits"].sum()
) == 12_165


print("Branded-beverage period audit completed.")
print()
print("Supplier periods:")

display(supplier_period_summary)

print()
print("Canonical branded-beverage summary:")

display(canonical_branded_summary)

Branded-beverage period audit completed.

Supplier periods:


,SupplierLabel_Step2,FirstObservedDate,LastObservedDate,SourcePLUCount,TransactionRows,TotalUnits
0,BEWLEYS,2025-04-01,2025-10-13,6,3261,3261
1,RN,2025-10-14,2026-03-30,7,8900,8900
2,CT,2025-10-28,2025-10-28,4,4,4



Canonical branded-beverage summary:


,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageType_Step2,FirstObservedDate,LastObservedDate,SourcePLUCount,TransactionRows,TotalUnits
0,BEV_BRANDED_AMERICANO,B-AMERICANO,AMERICANO,2025-04-01,2026-03-30,3,4812,4812
1,BEV_BRANDED_LATTE,B-LATTE,LATTE,2025-04-01,2026-03-30,2,2739,2739
2,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,CAPPUCCINO,2025-04-01,2026-03-30,2,2284,2284
3,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,FLAT WHITE,2025-04-01,2026-03-30,2,1350,1350
4,BEV_BRANDED_HOT_CHOCOLATE,B-HOT CHOCOLATE,HOT CHOCOLATE,2025-04-02,2026-03-30,2,484,484
5,BEV_BRANDED_MOCHA,B-MOCHA,MOCHA,2025-04-01,2026-03-30,2,454,454
6,BEV_BRANDED_ESPRESSO,B-ESPRESSO,ESPRESSO,2025-10-17,2026-03-30,2,40,40
7,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,FILTER COFFEE,2025-10-28,2025-10-28,1,1,1
8,BEV_BRANDED_TEA,B-TEA,TEA,2025-10-28,2025-10-28,1,1,1


In [20]:
# ============================================================
# Step 2 - Cell 9
# Join for validation only and prove no source data changed
# ============================================================

step2_transactions_with_order = (
    step2_transactions.copy()
)


step2_transactions_with_order[
    "_SourceRowOrder"
] = np.arange(
    len(step2_transactions_with_order)
)


step2_validation_join = (
    step2_transactions_with_order
    .merge(
        step2_mapping[
            [
                "PLUCode",
                "PLUName_Original",
                "PLUName_Corrected",
                "CanonicalProductID_Step1",
                "CanonicalProductName_Step1",
                "CanonicalProductID_Step2",
                "CanonicalProductName_Step2",
                "BeverageSeries_Step2",
                "BeverageType_Step2",
                "SupplierLabel_Step2",
                "SupplierNormalizationApplied_Step2",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values("_SourceRowOrder")
    .reset_index(drop=True)
)


source_for_comparison = (
    step2_transactions
    .reset_index(drop=True)
)


assert len(step2_validation_join) == (
    step2_source_row_count
)


assert step2_validation_join[
    "CanonicalProductID_Step2"
].notna().all()


# ------------------------------------------------------------
# Confirm every original source column remained unchanged
# ------------------------------------------------------------

for source_column in step2_source_columns:

    assert (
        step2_validation_join[source_column]
        .reset_index(drop=True)
        .equals(
            source_for_comparison[source_column]
            .reset_index(drop=True)
        )
    ), (
        f"Source column changed during validation join: "
        f"{source_column}"
    )


assert int(
    step2_validation_join["UnitSold"].sum()
) == step2_source_unit_total


joined_value_cents = int(
    (
        step2_validation_join["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert joined_value_cents == step2_source_value_cents


mapped_transaction_mask = (
    step2_validation_join[
        "SupplierNormalizationApplied_Step2"
    ]
)


assert int(
    mapped_transaction_mask.sum()
) == 12_165


assert int(
    step2_validation_join.loc[
        mapped_transaction_mask,
        "UnitSold",
    ].sum()
) == 12_165


# OPEN UL must never be affected
open_ul_validation_mask = (
    step2_validation_join["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert not step2_validation_join.loc[
    open_ul_validation_mask,
    "SupplierNormalizationApplied_Step2"
].any()


print("Full transaction validation passed.")
print()
print(
    "Source rows unchanged:",
    f"{len(step2_validation_join):,}"
)
print(
    "Source units unchanged:",
    f"{int(step2_validation_join['UnitSold'].sum()):,}"
)
print(
    "Source value unchanged:",
    joined_value_cents == step2_source_value_cents
)
print(
    "Branded transaction rows:",
    f"{int(mapped_transaction_mask.sum()):,}"
)
print(
    "Branded units:",
    f"{int(step2_validation_join.loc[mapped_transaction_mask, 'UnitSold'].sum()):,}"
)

Full transaction validation passed.

Source rows unchanged: 138,983
Source units unchanged: 141,480
Source value unchanged: True
Branded transaction rows: 12,165
Branded units: 12,165


In [21]:
# ============================================================
# Step 2 - Cell 10
# Create the detailed branded-beverage PLU audit
# ============================================================

branded_beverage_plu_audit = (
    step2_mapping.loc[
        step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID_Step1",
            "CanonicalProductName_Step1",
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "BeverageSeries_Step2",
            "BeverageType_Step2",
            "SupplierLabel_Step2",
            "ProductFamily_Step2",
            "GroupCode",
            "GroupName",
            "FirstObservedDate",
            "LastObservedDate",
            "TransactionRows",
            "TotalUnits",
            "TotalTransactionValue",
            "SupplierNormalizationType_Step2",
            "SupplierNormalizationReason_Step2",
            "SupplierNormalizationStatus_Step2",
            "SupplierNormalizationVersion_Step2",
        ],
    ]
    .sort_values(
        [
            "CanonicalProductName_Step2",
            "FirstObservedDate",
            "SupplierLabel_Step2",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


assert len(branded_beverage_plu_audit) == 17


assert branded_beverage_plu_audit[
    "CanonicalProductID_Step2"
].nunique() == 9


assert int(
    branded_beverage_plu_audit[
        "TotalUnits"
    ].sum()
) == 12_165


display(branded_beverage_plu_audit)

,PLUCode,PLUName_Original,PLUName_Corrected,CanonicalProductID_Step1,CanonicalProductName_Step1,CanonicalProductID_Step2,CanonicalProductName_Step2,BeverageSeries_Step2,BeverageType_Step2,SupplierLabel_Step2,...,GroupName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,TotalTransactionValue,SupplierNormalizationType_Step2,SupplierNormalizationReason_Step2,SupplierNormalizationStatus_Step2,SupplierNormalizationVersion_Step2
0,42527101,BEWLEYS AMERICANO,BEWLEYS AMERICANO,PLU_42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,1876,1876,5333.20,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
1,44381,RN AMERICANO,RN AMERICANO,PLU_44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,RN,...,HOT BEVS,2025-10-14,2026-03-30,2935,2935,9392.00,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
2,425531,ROASTED NOTES AMERICANO CT,ROASTED NOTES AMERICANO CT,PLU_425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,BRANDED,AMERICANO,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,3.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
3,42527103,BEWLEYS CAPPUCINO,BEWLEYS CAPPUCCINO,PLU_42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BRANDED,CAPPUCCINO,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,517,517,1614.80,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
4,44382,RN CAPPUCINO,RN CAPPUCCINO,PLU_44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BRANDED,CAPPUCCINO,RN,...,HOT BEVS,2025-10-14,2026-03-30,1767,1767,6361.20,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
5,4241523,ROASTED NOTES ESPRESSO,ROASTED NOTES ESPRESSO,PLU_4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,BRANDED,ESPRESSO,RN,...,HOT BEVS,2025-10-17,2026-03-30,39,39,124.80,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
6,42555,R0OASTED NOTES EXPRESSO CT,ROASTED NOTES ESPRESSO CT,PLU_42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,BRANDED,ESPRESSO,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,3.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
7,42563,FILTER COFFEE CT,FILTER COFFEE CT,PLU_42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,BRANDED,FILTER COFFEE,CT,...,HOT BEVS,2025-10-28,2025-10-28,1,1,2.50,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
8,42527104,BEWLEYS FLAT WHITE,BEWLEYS FLAT WHITE,PLU_42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BRANDED,FLAT WHITE,BEWLEYS,...,HOT BEVS,2025-04-01,2025-10-13,191,191,577.40,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1
9,417173,RN FLAT WHITE,RN FLAT WHITE,PLU_417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BRANDED,FLAT WHITE,RN,...,HOT BEVS,2025-10-14,2026-03-30,1159,1159,4172.40,CONFIRMED_BRANDED_EQUIVALENCE,Confirmed branded beverage equivalent sold und...,APPROVED_STEP_2,STEP_2_BRANDED_BEVERAGES_V1


In [22]:
# ============================================================
# Step 2 - Cell 11
# Save Step 2 mapping and audit outputs
# ============================================================

step2_mapping.to_csv(
    STEP2_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


branded_beverage_plu_audit.to_csv(
    STEP2_PLU_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


canonical_branded_summary.to_csv(
    STEP2_CANONICAL_SUMMARY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


supplier_period_summary.to_csv(
    STEP2_SUPPLIER_PERIOD_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Step 2 files saved successfully.")
print()

print("1. Step 2 master mapping:")
print(STEP2_MAPPING_OUTPUT_FILE)

print()
print("2. Branded-beverage PLU audit:")
print(STEP2_PLU_AUDIT_OUTPUT_FILE)

print()
print("3. Canonical branded-beverage summary:")
print(STEP2_CANONICAL_SUMMARY_OUTPUT_FILE)

print()
print("4. Supplier-period audit:")
print(STEP2_SUPPLIER_PERIOD_OUTPUT_FILE)

Step 2 files saved successfully.

1. Step 2 master mapping:
eden_datasets/UL_EDEN_master_product_mapping_step2_branded_beverages.csv

2. Branded-beverage PLU audit:
eden_datasets/UL_EDEN_step2_branded_beverage_plu_audit.csv

3. Canonical branded-beverage summary:
eden_datasets/UL_EDEN_step2_branded_beverage_canonical_summary.csv

4. Supplier-period audit:
eden_datasets/UL_EDEN_step2_branded_beverage_supplier_periods.csv


In [23]:
# ============================================================
# Step 2 - Cell 12
# Reload and validate the saved Step 2 mapping
# ============================================================

saved_step2_mapping = pd.read_csv(
    STEP2_MAPPING_OUTPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


saved_step2_mapping["PLUCode"] = (
    pd.to_numeric(
        saved_step2_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_step2_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_step2_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


saved_step2_applied = (
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
    })
)


assert saved_step2_applied.notna().all()


saved_step2_mapping[
    "SupplierNormalizationApplied_Step2"
] = saved_step2_applied.astype(bool)


# ------------------------------------------------------------
# Final saved-file validation
# ------------------------------------------------------------

assert len(saved_step2_mapping) == 236


assert saved_step2_mapping[
    "PLUCode"
].is_unique


assert saved_step2_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert saved_step2_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228


assert int(
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ].sum()
) == 17


assert saved_step2_mapping.loc[
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ],
    "CanonicalProductID_Step2"
].nunique() == 9


assert saved_step2_mapping.loc[
    saved_step2_mapping[
        "SupplierNormalizationApplied_Step2"
    ],
    "CanonicalProductName_Step2"
].str.startswith("B-").all()


assert int(
    saved_step2_mapping["TotalUnits"].sum()
) == 141_480


saved_branded_unit_total = int(
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        "TotalUnits",
    ].sum()
)


assert saved_branded_unit_total == 12_165


# Confirm no non-approved PLU changed identity
saved_changed_mask = (
    saved_step2_mapping[
        "CanonicalProductID_Step2"
    ]
    !=
    saved_step2_mapping[
        "CanonicalProductID_Step1"
    ]
)


assert set(
    saved_step2_mapping.loc[
        saved_changed_mask,
        "PLUCode",
    ]
) == approved_branded_plu_codes


print("=" * 74)
print("STEP 2 BRANDED-BEVERAGE NORMALISATION COMPLETED")
print("=" * 74)
print()
print(
    "Mapping rows:",
    len(saved_step2_mapping)
)
print(
    "Step 1 product identities:",
    saved_step2_mapping[
        "CanonicalProductID_Step1"
    ].nunique()
)
print(
    "Step 2 product identities:",
    saved_step2_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)
print(
    "Branded source PLUs mapped:",
    int(
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ].sum()
    )
)
print(
    "Canonical branded beverages:",
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        "CanonicalProductID_Step2",
    ].nunique()
)
print(
    "Branded beverage units:",
    f"{saved_branded_unit_total:,}"
)
print(
    "Source transaction units preserved:",
    f"{step2_source_unit_total:,}"
)
print()
print("Final Step 2 mapping:")
print(STEP2_MAPPING_OUTPUT_FILE)

display(
    saved_step2_mapping.loc[
        saved_step2_mapping[
            "SupplierNormalizationApplied_Step2"
        ],
        [
            "PLUCode",
            "PLUName_Corrected",
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
            "TotalUnits",
        ],
    ]
    .sort_values(
        [
            "CanonicalProductName_Step2",
            "SupplierLabel_Step2",
        ]
    )
)

STEP 2 BRANDED-BEVERAGE NORMALISATION COMPLETED

Mapping rows: 236
Step 1 product identities: 236
Step 2 product identities: 228
Branded source PLUs mapped: 17
Canonical branded beverages: 9
Branded beverage units: 12,165
Source transaction units preserved: 141,480

Final Step 2 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step2_branded_beverages.csv


,PLUCode,PLUName_Corrected,CanonicalProductID_Step2,CanonicalProductName_Step2,SupplierLabel_Step2,TotalUnits
16,42527101,BEWLEYS AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,BEWLEYS,1876
187,425531,ROASTED NOTES AMERICANO CT,BEV_BRANDED_AMERICANO,B-AMERICANO,CT,1
181,44381,RN AMERICANO,BEV_BRANDED_AMERICANO,B-AMERICANO,RN,2935
17,42527103,BEWLEYS CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,BEWLEYS,517
182,44382,RN CAPPUCCINO,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,RN,1767
189,42555,ROASTED NOTES ESPRESSO CT,BEV_BRANDED_ESPRESSO,B-ESPRESSO,CT,1
188,4241523,ROASTED NOTES ESPRESSO,BEV_BRANDED_ESPRESSO,B-ESPRESSO,RN,39
84,42563,FILTER COFFEE CT,BEV_BRANDED_FILTER_COFFEE,B-FILTER COFFEE,CT,1
18,42527104,BEWLEYS FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,BEWLEYS,191
183,417173,RN FLAT WHITE,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,RN,1159


### Now what I am going to do is forecasting identity of price-tier products independent of later price increases, without altering their transaction values.

In [24]:
# ============================================================
# Step 3 - Cell 1
# Load Step 2 mapping and immutable corrected transactions
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


STEP2_MAPPING_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step2_branded_beverages.csv"
)


STEP3_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv"
)


STEP3_TIER_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_tier_product_audit.csv"
)


STEP3_PRICE_PERIOD_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_tier_price_periods.csv"
)


STEP3_PRICE_TRANSITION_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_price_change_transitions.csv"
)


STEP3_OTHER_DINNER_PRICE_CHANGES_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step3_other_dinner_price_changes.csv"
)


if not TRANSACTION_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Corrected transaction file not found:\n"
        f"{TRANSACTION_INPUT_FILE}"
    )


if not STEP2_MAPPING_INPUT_FILE.exists():
    raise FileNotFoundError(
        "Completed Step 2 mapping not found:\n"
        f"{STEP2_MAPPING_INPUT_FILE}"
    )


step3_transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


step3_mapping = pd.read_csv(
    STEP2_MAPPING_INPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
    ]
)


# Preserve source structure for integrity validation
step3_source_columns = (
    step3_transactions.columns.tolist()
)


# ------------------------------------------------------------
# Convert source columns safely
# ------------------------------------------------------------

step3_transactions["TransDate"] = pd.to_datetime(
    step3_transactions["TransDate"],
    errors="raise"
)


step3_transactions["Date"] = (
    pd.to_datetime(
        step3_transactions["Date"],
        errors="raise"
    )
    .dt.normalize()
)


step3_transactions["PLUCode"] = (
    pd.to_numeric(
        step3_transactions["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step3_transactions["UnitSold"] = (
    pd.to_numeric(
        step3_transactions["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step3_transactions["TransValue"] = pd.to_numeric(
    step3_transactions["TransValue"],
    errors="raise"
)


step3_mapping["PLUCode"] = (
    pd.to_numeric(
        step3_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step3_mapping["TotalUnits"] = (
    pd.to_numeric(
        step3_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Record immutable source totals
# ------------------------------------------------------------

step3_source_row_count = len(
    step3_transactions
)


step3_source_unit_total = int(
    step3_transactions["UnitSold"].sum()
)


step3_source_value_cents = int(
    (
        step3_transactions["TransValue"]
        * 100
    )
    .round()
    .sum()
)


# ------------------------------------------------------------
# Validate Step 2 state
# ------------------------------------------------------------

assert step3_source_row_count == 138_983


assert step3_source_unit_total == 141_480


assert step3_transactions[
    "PLUCode"
].nunique() == 236


assert len(step3_mapping) == 236


assert step3_mapping[
    "PLUCode"
].is_unique


assert step3_mapping[
    "CanonicalProductID_Step1"
].nunique() == 236


assert step3_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228


assert int(
    step3_mapping["TotalUnits"].sum()
) == 141_480


print("Step 3 input files loaded and validated.")
print()
print(
    "Transaction rows:",
    f"{step3_source_row_count:,}"
)
print(
    "Transaction units:",
    f"{step3_source_unit_total:,}"
)
print(
    "Step 2 canonical identities:",
    step3_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)

Step 3 input files loaded and validated.

Transaction rows: 138,983
Transaction units: 141,480
Step 2 canonical identities: 228


In [25]:
# ============================================================
# Step 3 - Cell 2
# Derive observed per-unit prices
# ============================================================

assert step3_transactions[
    "UnitSold"
].gt(0).all(), (
    "UnitSold must be above zero before calculating prices."
)


step3_transactions[
    "ObservedUnitPrice_Step3"
] = (
    step3_transactions["TransValue"]
    / step3_transactions["UnitSold"]
).round(2)


# ------------------------------------------------------------
# Confirm that price × quantity reconstructs transaction value
# ------------------------------------------------------------

reconstructed_transaction_values = (
    step3_transactions[
        "ObservedUnitPrice_Step3"
    ]
    * step3_transactions["UnitSold"]
)


assert np.allclose(
    reconstructed_transaction_values,
    step3_transactions["TransValue"],
    atol=0.011
), (
    "At least one transaction value cannot be reconstructed "
    "from ObservedUnitPrice × UnitSold."
)


print("Observed unit prices calculated successfully.")
print()
print(
    "Rows:",
    f"{len(step3_transactions):,}"
)

Observed unit prices calculated successfully.

Rows: 138,983


In [26]:
# ============================================================
# Step 3 - Cell 3
# Audit all DINNER products and observed unit prices
# ============================================================

dinner_transactions_step3 = (
    step3_transactions.loc[
        step3_transactions[
            "GroupName"
        ].eq("DINNER")
    ]
    .copy()
)


dinner_product_price_audit = (
    dinner_transactions_step3
    .groupby(
        [
            "PLUCode",
            "PLUName",
        ],
        as_index=False
    )
    .agg(
        FirstObservedDate=(
            "Date",
            "min"
        ),
        LastObservedDate=(
            "Date",
            "max"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
        UniqueObservedPrices=(
            "ObservedUnitPrice_Step3",
            "nunique"
        ),
        MinimumObservedUnitPrice=(
            "ObservedUnitPrice_Step3",
            "min"
        ),
        MaximumObservedUnitPrice=(
            "ObservedUnitPrice_Step3",
            "max"
        ),
        ObservedPrices=(
            "ObservedUnitPrice_Step3",
            lambda values: ", ".join(
                f"{price:.2f}"
                for price in sorted(
                    values.unique()
                )
            )
        ),
    )
    .sort_values(
        [
            "FirstObservedDate",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


print("DINNER product-price audit created.")
print()
print(
    "Dinner PLUs:",
    dinner_product_price_audit[
        "PLUCode"
    ].nunique()
)

display(dinner_product_price_audit)

DINNER product-price audit created.

Dinner PLUs: 21


,PLUCode,PLUName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,UniqueObservedPrices,MinimumObservedUnitPrice,MaximumObservedUnitPrice,ObservedPrices
0,4241475,KIMBOX MAINS 1,2025-04-01,2025-10-13,1037,1037,1,5.0,5.0,5.00
1,4241476,KIMBOX MAINS 2,2025-04-01,2025-10-13,6575,6623,1,7.0,7.0,7.00
2,4241478,MAINS 1,2025-04-01,2025-10-09,1903,1903,1,5.0,5.0,5.00
3,4241479,MAINS 2,2025-04-01,2025-10-13,2265,2265,1,7.0,7.0,7.00
4,4241481,VEGT MAINS 1,2025-04-01,2025-09-16,878,878,1,5.0,5.0,5.00
5,4241482,VEGT MAINS 2,2025-04-02,2025-09-15,502,502,1,7.0,7.0,7.00
6,4241480,MAINS 3,2025-07-01,2025-08-01,37,474,1,9.0,9.0,9.00
7,4241483,VEGT MAINS 3,2025-07-01,2025-07-25,8,262,1,9.0,9.0,9.00
8,4241477,KIMBOX MAINS 3,2025-07-02,2025-09-02,5,5,1,9.0,9.0,9.00
9,42529,€5.00 DINNER,2025-10-14,2026-03-30,3312,3312,2,5.0,5.3,"5.00, 5.30"


In [27]:
# ============================================================
# Step 3 - Cell 4
# Define price-tier metadata
#
# Important:
# This assigns metadata only.
# It does not merge any PLUs.
# ============================================================

price_tier_metadata = [
    # --------------------------------------------------------
    # Current DINNER tiers
    # --------------------------------------------------------
    {
        "PLUCode": 42529,
        "ExpectedCorrectedName": "€5.00 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [5.00, 5.30],
    },
    {
        "PLUCode": 42530,
        "ExpectedCorrectedName": "€7 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [7.00, 7.30],
    },
    {
        "PLUCode": 42531,
        "ExpectedCorrectedName": "€9 DINNER",
        "TierProductFamily": "DINNER",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [9.00, 9.30],
    },

    # --------------------------------------------------------
    # Current KIMBOCK tiers
    # --------------------------------------------------------
    {
        "PLUCode": 42532,
        "ExpectedCorrectedName": "€5 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [5.00, 5.30],
    },
    {
        "PLUCode": 42533,
        "ExpectedCorrectedName": "€7 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [7.00, 7.30],
    },
    {
        "PLUCode": 42534,
        "ExpectedCorrectedName": "€9 KIMBOCK",
        "TierProductFamily": "KIMBOCK",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": pd.NA,
        "MenuGeneration": "CURRENT_PRICE_LABELLED",
        "AllowedObservedPrices": [9.00, 9.30],
    },

    # --------------------------------------------------------
    # Legacy KIMBOX tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241475,
        "ExpectedCorrectedName": "KIMBOX MAINS 1",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241476,
        "ExpectedCorrectedName": "KIMBOX MAINS 2",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241477,
        "ExpectedCorrectedName": "KIMBOX MAINS 3",
        "TierProductFamily": "KIMBOX",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },

    # --------------------------------------------------------
    # Legacy general MAINS tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241478,
        "ExpectedCorrectedName": "MAINS 1",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241479,
        "ExpectedCorrectedName": "MAINS 2",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241480,
        "ExpectedCorrectedName": "MAINS 3",
        "TierProductFamily": "MAINS",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },

    # --------------------------------------------------------
    # Legacy vegetarian MAINS tiers
    # --------------------------------------------------------
    {
        "PLUCode": 4241481,
        "ExpectedCorrectedName": "VEGT MAINS 1",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 5,
        "LegacyMenuLevel": 1,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [5.00],
    },
    {
        "PLUCode": 4241482,
        "ExpectedCorrectedName": "VEGT MAINS 2",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 7,
        "LegacyMenuLevel": 2,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [7.00],
    },
    {
        "PLUCode": 4241483,
        "ExpectedCorrectedName": "VEGT MAINS 3",
        "TierProductFamily": "VEGETARIAN MAINS",
        "NominalPriceTier": 9,
        "LegacyMenuLevel": 3,
        "MenuGeneration": "LEGACY_LEVEL_LABELLED",
        "AllowedObservedPrices": [9.00],
    },
]


price_tier_metadata_df = pd.DataFrame(
    price_tier_metadata
)


assert len(price_tier_metadata_df) == 15


assert price_tier_metadata_df[
    "PLUCode"
].is_unique


approved_price_tier_plu_codes = set(
    price_tier_metadata_df["PLUCode"]
)


print("Price-tier metadata defined.")
print()
print(
    "Price-tier PLUs:",
    len(price_tier_metadata_df)
)

display(
    price_tier_metadata_df[
        [
            "PLUCode",
            "ExpectedCorrectedName",
            "TierProductFamily",
            "NominalPriceTier",
            "LegacyMenuLevel",
            "MenuGeneration",
            "AllowedObservedPrices",
        ]
    ]
)

Price-tier metadata defined.

Price-tier PLUs: 15


,PLUCode,ExpectedCorrectedName,TierProductFamily,NominalPriceTier,LegacyMenuLevel,MenuGeneration,AllowedObservedPrices
0,42529,€5.00 DINNER,DINNER,5,<NA>,CURRENT_PRICE_LABELLED,"[5.0, 5.3]"
1,42530,€7 DINNER,DINNER,7,<NA>,CURRENT_PRICE_LABELLED,"[7.0, 7.3]"
2,42531,€9 DINNER,DINNER,9,<NA>,CURRENT_PRICE_LABELLED,"[9.0, 9.3]"
3,42532,€5 KIMBOCK,KIMBOCK,5,<NA>,CURRENT_PRICE_LABELLED,"[5.0, 5.3]"
4,42533,€7 KIMBOCK,KIMBOCK,7,<NA>,CURRENT_PRICE_LABELLED,"[7.0, 7.3]"
5,42534,€9 KIMBOCK,KIMBOCK,9,<NA>,CURRENT_PRICE_LABELLED,"[9.0, 9.3]"
6,4241475,KIMBOX MAINS 1,KIMBOX,5,1,LEGACY_LEVEL_LABELLED,[5.0]
7,4241476,KIMBOX MAINS 2,KIMBOX,7,2,LEGACY_LEVEL_LABELLED,[7.0]
8,4241477,KIMBOX MAINS 3,KIMBOX,9,3,LEGACY_LEVEL_LABELLED,[9.0]
9,4241478,MAINS 1,MAINS,5,1,LEGACY_LEVEL_LABELLED,[5.0]


In [28]:
# ============================================================
# Step 3 - Cell 5
# Preflight validation of all 15 tier products
# ============================================================

tier_transactions_step3 = (
    step3_transactions.loc[
        step3_transactions[
            "PLUCode"
        ].isin(approved_price_tier_plu_codes)
    ]
    .copy()
)


assert tier_transactions_step3[
    "PLUCode"
].nunique() == 15


assert len(tier_transactions_step3) == 32_733, (
    "Expected 32,733 transaction rows for the "
    "15 price-tier products."
)


assert int(
    tier_transactions_step3["UnitSold"].sum()
) == 33_552, (
    "Expected 33,552 units for the "
    "15 price-tier products."
)


assert tier_transactions_step3[
    "GroupName"
].eq("DINNER").all()


for tier_mapping in price_tier_metadata:

    plu_code = tier_mapping["PLUCode"]


    # --------------------------------------------------------
    # Validate Step 2 mapping name
    # --------------------------------------------------------

    mapping_mask = (
        step3_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(mapping_mask.sum()) == 1, (
        f"Expected one Step 2 mapping row for PLU {plu_code}."
    )


    actual_corrected_name = (
        step3_mapping.loc[
            mapping_mask,
            "PLUName_Corrected"
        ]
        .iloc[0]
        .strip()
        .upper()
    )


    expected_corrected_name = (
        tier_mapping[
            "ExpectedCorrectedName"
        ]
        .strip()
        .upper()
    )


    assert actual_corrected_name == (
        expected_corrected_name
    ), (
        f"Name mismatch for PLU {plu_code}.\n"
        f"Expected: {expected_corrected_name}\n"
        f"Found: {actual_corrected_name}"
    )


    # --------------------------------------------------------
    # Validate observed unit prices
    # --------------------------------------------------------

    product_transactions = (
        tier_transactions_step3.loc[
            tier_transactions_step3[
                "PLUCode"
            ].eq(plu_code)
        ]
    )


    actual_prices = {
        round(float(price), 2)
        for price in product_transactions[
            "ObservedUnitPrice_Step3"
        ].unique()
    }


    allowed_prices = {
        round(float(price), 2)
        for price in tier_mapping[
            "AllowedObservedPrices"
        ]
    }


    assert actual_prices == allowed_prices, (
        f"Observed-price mismatch for PLU {plu_code}.\n"
        f"Expected: {sorted(allowed_prices)}\n"
        f"Found: {sorted(actual_prices)}"
    )


print("All 15 price-tier products passed preflight.")
print()
print(
    "Tier transaction rows:",
    f"{len(tier_transactions_step3):,}"
)
print(
    "Tier units:",
    f"{int(tier_transactions_step3['UnitSold'].sum()):,}"
)

All 15 price-tier products passed preflight.

Tier transaction rows: 32,733
Tier units: 33,552


In [29]:
# ============================================================
# Step 3 - Cell 6
# Create price-period and price-transition audits
# ============================================================

price_tier_price_periods = (
    tier_transactions_step3
    .groupby(
        [
            "PLUCode",
            "PLUName",
            "ObservedUnitPrice_Step3",
        ],
        as_index=False
    )
    .agg(
        FirstPriceDate=(
            "Date",
            "min"
        ),
        LastPriceDate=(
            "Date",
            "max"
        ),
        TransactionRows=(
            "PLUCode",
            "size"
        ),
        TotalUnits=(
            "UnitSold",
            "sum"
        ),
    )
    .sort_values(
        [
            "PLUCode",
            "FirstPriceDate",
            "ObservedUnitPrice_Step3",
        ]
    )
    .reset_index(drop=True)
)


price_change_transition_rows = []


for plu_code, product_prices in (
    price_tier_price_periods
    .groupby(
        "PLUCode",
        sort=False
    )
):

    product_prices = (
        product_prices
        .sort_values(
            [
                "FirstPriceDate",
                "ObservedUnitPrice_Step3",
            ]
        )
        .reset_index(drop=True)
    )


    product_name = (
        product_prices["PLUName"]
        .iloc[0]
    )


    if len(product_prices) == 1:

        price_change_transition_rows.append({
            "PLUCode": plu_code,
            "PLUName": product_name,
            "PriceChangeDetected": False,
            "InitialObservedUnitPrice":
                product_prices[
                    "ObservedUnitPrice_Step3"
                ].iloc[0],
            "CurrentObservedUnitPrice":
                product_prices[
                    "ObservedUnitPrice_Step3"
                ].iloc[-1],
            "InitialPriceLastDate":
                product_prices[
                    "LastPriceDate"
                ].iloc[0],
            "ChangedPriceFirstDate": pd.NaT,
            "PricePeriodsOverlap": False,
        })

    elif len(product_prices) == 2:

        first_period = product_prices.iloc[0]
        second_period = product_prices.iloc[1]


        periods_overlap = (
            first_period["LastPriceDate"]
            >= second_period["FirstPriceDate"]
        )


        price_change_transition_rows.append({
            "PLUCode": plu_code,
            "PLUName": product_name,
            "PriceChangeDetected": True,
            "InitialObservedUnitPrice":
                first_period[
                    "ObservedUnitPrice_Step3"
                ],
            "CurrentObservedUnitPrice":
                second_period[
                    "ObservedUnitPrice_Step3"
                ],
            "InitialPriceLastDate":
                first_period[
                    "LastPriceDate"
                ],
            "ChangedPriceFirstDate":
                second_period[
                    "FirstPriceDate"
                ],
            "PricePeriodsOverlap":
                periods_overlap,
        })

    else:
        raise ValueError(
            f"PLU {plu_code} has more than two observed "
            "price periods and requires manual review."
        )


price_change_transitions = pd.DataFrame(
    price_change_transition_rows
)


price_change_transitions[
    "PLUCode"
] = price_change_transitions[
    "PLUCode"
].astype("int64")


# ------------------------------------------------------------
# Validate price-transition structure
# ------------------------------------------------------------

assert len(price_change_transitions) == 15


assert price_change_transitions[
    "PLUCode"
].is_unique


assert int(
    price_change_transitions[
        "PriceChangeDetected"
    ].sum()
) == 6, (
    "Expected six tier products with confirmed price changes."
)


assert not price_change_transitions.loc[
    price_change_transitions[
        "PriceChangeDetected"
    ],
    "PricePeriodsOverlap",
].any(), (
    "An old and new tier price period overlap."
)


print("Price-period audits created.")
print()
print(
    "Tier products with a price change:",
    int(
        price_change_transitions[
            "PriceChangeDetected"
        ].sum()
    )
)
print(
    "Tier products with a fixed observed price:",
    int(
        (
            ~price_change_transitions[
                "PriceChangeDetected"
            ]
        ).sum()
    )
)

display(price_tier_price_periods)

display(price_change_transitions)

Price-period audits created.

Tier products with a price change: 6
Tier products with a fixed observed price: 9


,PLUCode,PLUName,ObservedUnitPrice_Step3,FirstPriceDate,LastPriceDate,TransactionRows,TotalUnits
0,42529,€5.00 DINNER,5.0,2025-10-14,2026-01-23,1644,1644
1,42529,€5.00 DINNER,5.3,2026-01-26,2026-03-30,1668,1668
2,42530,€7 DINNER,7.0,2025-10-14,2026-01-23,2172,2172
3,42530,€7 DINNER,7.3,2026-01-26,2026-03-30,1020,1020
4,42531,€9 DINNER,9.0,2025-11-27,2026-01-22,130,130
5,42531,€9 DINNER,9.3,2026-01-26,2026-03-27,594,594
6,42532,€5 KIMBOCK,5.0,2025-10-14,2026-01-23,756,756
7,42532,€5 KIMBOCK,5.3,2026-01-26,2026-03-30,579,579
8,42533,€7 KIMBOCK,7.0,2025-10-14,2026-01-23,5960,5960
9,42533,€7 KIMBOCK,7.3,2026-01-26,2026-03-30,4986,4986


,PLUCode,PLUName,PriceChangeDetected,InitialObservedUnitPrice,CurrentObservedUnitPrice,InitialPriceLastDate,ChangedPriceFirstDate,PricePeriodsOverlap
0,42529,€5.00 DINNER,True,5.0,5.3,2026-01-23,2026-01-26,False
1,42530,€7 DINNER,True,7.0,7.3,2026-01-23,2026-01-26,False
2,42531,€9 DINNER,True,9.0,9.3,2026-01-22,2026-01-26,False
3,42532,€5 KIMBOCK,True,5.0,5.3,2026-01-23,2026-01-26,False
4,42533,€7 KIMBOCK,True,7.0,7.3,2026-01-23,2026-01-26,False
5,42534,€9 KIMBOCK,True,9.0,9.3,2026-01-17,2026-01-28,False
6,4241475,KIMBOX MAINS 1,False,5.0,5.0,2025-10-13,NaT,False
7,4241476,KIMBOX MAINS 2,False,7.0,7.0,2025-10-13,NaT,False
8,4241477,KIMBOX MAINS 3,False,9.0,9.0,2025-09-02,NaT,False
9,4241478,MAINS 1,False,5.0,5.0,2025-10-09,NaT,False


In [30]:
# ============================================================
# Step 3 - Cell 7
# Audit other DINNER products with price changes
# ============================================================

other_dinner_price_changes = (
    dinner_product_price_audit.loc[
        (
            dinner_product_price_audit[
                "UniqueObservedPrices"
            ].gt(1)
        )
        &
        (
            ~dinner_product_price_audit[
                "PLUCode"
            ].isin(
                approved_price_tier_plu_codes
            )
        )
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(other_dinner_price_changes) == 1, (
    "Expected one non-tier DINNER product with "
    "multiple observed unit prices."
)


assert int(
    other_dinner_price_changes[
        "PLUCode"
    ].iloc[0]
) == 42535


assert (
    other_dinner_price_changes[
        "PLUName"
    ].iloc[0]
    == "PORTION CHIPS"
)


print(
    "Other DINNER price changes were kept separate "
    "from the tier mapping."
)

display(other_dinner_price_changes)

Other DINNER price changes were kept separate from the tier mapping.


,PLUCode,PLUName,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,UniqueObservedPrices,MinimumObservedUnitPrice,MaximumObservedUnitPrice,ObservedPrices
0,42535,PORTION CHIPS,2025-10-14,2026-03-30,1110,1110,2,2.0,2.2,"2.00, 2.20"


In [31]:
# ============================================================
# Step 3 - Cell 8
# Create isolated Step 3 metadata columns
# ============================================================

# Step 3 identity begins as an exact copy of Step 2
step3_mapping[
    "CanonicalProductID_Step3"
] = step3_mapping[
    "CanonicalProductID_Step2"
].astype("string")


step3_mapping[
    "CanonicalProductName_Step3"
] = step3_mapping[
    "CanonicalProductName_Step2"
].astype("string")


# Tier metadata
step3_mapping[
    "TierProductFamily_Step3"
] = pd.Series(
    pd.NA,
    index=step3_mapping.index,
    dtype="string"
)


step3_mapping[
    "NominalPriceTier_Step3"
] = pd.Series(
    pd.NA,
    index=step3_mapping.index,
    dtype="Int64"
)


step3_mapping[
    "LegacyMenuLevel_Step3"
] = pd.Series(
    pd.NA,
    index=step3_mapping.index,
    dtype="Int64"
)


step3_mapping[
    "MenuGeneration_Step3"
] = pd.Series(
    pd.NA,
    index=step3_mapping.index,
    dtype="string"
)


step3_mapping[
    "InitialObservedUnitPrice_Step3"
] = np.nan


step3_mapping[
    "CurrentObservedUnitPrice_Step3"
] = np.nan


step3_mapping[
    "ObservedUnitPrices_Step3"
] = pd.Series(
    pd.NA,
    index=step3_mapping.index,
    dtype="string"
)


step3_mapping[
    "PriceChangeDetected_Step3"
] = False


step3_mapping[
    "PriceChangeDate_Step3"
] = pd.NaT


step3_mapping[
    "PriceTierMetadataApplied_Step3"
] = False


step3_mapping[
    "PriceIdentityChanged_Step3"
] = False


step3_mapping[
    "PriceTierProcessingType_Step3"
] = "NOT_APPLICABLE"


step3_mapping[
    "PriceTierProcessingReason_Step3"
] = (
    "Product is not part of the approved Step 3 "
    "price-tier metadata set."
)


step3_mapping[
    "PriceTierProcessingStatus_Step3"
] = "UNCHANGED_STEP_3"


step3_mapping[
    "PriceTierProcessingVersion_Step3"
] = (
    "STEP_3_PRICE_TIER_METADATA_V1"
)


print("Isolated Step 3 columns created.")

Isolated Step 3 columns created.


In [32]:
# ============================================================
# Step 3 - Cell 9
# Apply price-tier metadata without merging products
# ============================================================

transition_lookup = (
    price_change_transitions
    .set_index("PLUCode")
)


for tier_mapping in price_tier_metadata:

    plu_code = tier_mapping["PLUCode"]


    row_mask = (
        step3_mapping["PLUCode"]
        .eq(plu_code)
    )


    assert int(row_mask.sum()) == 1


    transition = transition_lookup.loc[
        plu_code
    ]


    observed_prices_for_product = (
        price_tier_price_periods.loc[
            price_tier_price_periods[
                "PLUCode"
            ].eq(plu_code),
            "ObservedUnitPrice_Step3",
        ]
        .sort_values()
        .tolist()
    )


    observed_prices_text = ", ".join(
        f"{price:.2f}"
        for price in observed_prices_for_product
    )


    step3_mapping.loc[
        row_mask,
        "TierProductFamily_Step3"
    ] = tier_mapping[
        "TierProductFamily"
    ]


    step3_mapping.loc[
        row_mask,
        "NominalPriceTier_Step3"
    ] = tier_mapping[
        "NominalPriceTier"
    ]


    if pd.notna(
        tier_mapping["LegacyMenuLevel"]
    ):
        step3_mapping.loc[
            row_mask,
            "LegacyMenuLevel_Step3"
        ] = int(
            tier_mapping[
                "LegacyMenuLevel"
            ]
        )


    step3_mapping.loc[
        row_mask,
        "MenuGeneration_Step3"
    ] = tier_mapping[
        "MenuGeneration"
    ]


    step3_mapping.loc[
        row_mask,
        "InitialObservedUnitPrice_Step3"
    ] = transition[
        "InitialObservedUnitPrice"
    ]


    step3_mapping.loc[
        row_mask,
        "CurrentObservedUnitPrice_Step3"
    ] = transition[
        "CurrentObservedUnitPrice"
    ]


    step3_mapping.loc[
        row_mask,
        "ObservedUnitPrices_Step3"
    ] = observed_prices_text


    step3_mapping.loc[
        row_mask,
        "PriceChangeDetected_Step3"
    ] = bool(
        transition[
            "PriceChangeDetected"
        ]
    )


    if pd.notna(
        transition["ChangedPriceFirstDate"]
    ):
        step3_mapping.loc[
            row_mask,
            "PriceChangeDate_Step3"
        ] = transition[
            "ChangedPriceFirstDate"
        ]


    step3_mapping.loc[
        row_mask,
        "PriceTierMetadataApplied_Step3"
    ] = True


    # Explicitly confirm no identity merge
    step3_mapping.loc[
        row_mask,
        "PriceIdentityChanged_Step3"
    ] = False


    step3_mapping.loc[
        row_mask,
        "PriceTierProcessingType_Step3"
    ] = "METADATA_ONLY_NO_PRODUCT_MERGE"


    step3_mapping.loc[
        row_mask,
        "PriceTierProcessingReason_Step3"
    ] = (
        "Nominal menu tier recorded independently from "
        "the actual historical selling price. "
        "Canonical product identity retained from Step 2."
    )


    step3_mapping.loc[
        row_mask,
        "PriceTierProcessingStatus_Step3"
    ] = "APPROVED_STEP_3"


print(
    "Price-tier metadata applied:",
    int(
        step3_mapping[
            "PriceTierMetadataApplied_Step3"
        ].sum()
    )
)

Price-tier metadata applied: 15


In [35]:
# ============================================================
# Step 3 - Cell 10
# Validate that Step 3 changed metadata only
#
# Corrected:
# Text values are normalised before comparison so that
# pandas object/string dtype differences do not create
# a false assertion failure.
# ============================================================


def normalise_text_for_comparison(series):
    """
    Convert a pandas Series to a consistent nullable-string
    representation for safe value comparison.
    """

    return (
        series
        .astype("string")
        .str.strip()
        .fillna("<MISSING>")
    )


# ------------------------------------------------------------
# 1. Compare Step 2 and Step 3 canonical product IDs
# ------------------------------------------------------------

step2_ids_normalised = normalise_text_for_comparison(
    step3_mapping[
        "CanonicalProductID_Step2"
    ]
)


step3_ids_normalised = normalise_text_for_comparison(
    step3_mapping[
        "CanonicalProductID_Step3"
    ]
)


canonical_id_difference_mask = (
    step2_ids_normalised
    != step3_ids_normalised
)


actual_id_differences = step3_mapping.loc[
    canonical_id_difference_mask,
    [
        "PLUCode",
        "PLUName_Corrected",
        "CanonicalProductID_Step2",
        "CanonicalProductID_Step3",
    ],
].copy()


if len(actual_id_differences) > 0:

    print(
        "Actual canonical product ID differences found:"
    )

    display(actual_id_differences)

    raise AssertionError(
        "A canonical product ID genuinely changed "
        "during Step 3."
    )


# ------------------------------------------------------------
# 2. Compare Step 2 and Step 3 canonical product names
# ------------------------------------------------------------

step2_names_normalised = normalise_text_for_comparison(
    step3_mapping[
        "CanonicalProductName_Step2"
    ]
)


step3_names_normalised = normalise_text_for_comparison(
    step3_mapping[
        "CanonicalProductName_Step3"
    ]
)


canonical_name_difference_mask = (
    step2_names_normalised
    != step3_names_normalised
)


actual_name_differences = step3_mapping.loc[
    canonical_name_difference_mask,
    [
        "PLUCode",
        "PLUName_Corrected",
        "CanonicalProductName_Step2",
        "CanonicalProductName_Step3",
    ],
].copy()


if len(actual_name_differences) > 0:

    print(
        "Actual canonical product-name differences found:"
    )

    display(actual_name_differences)

    raise AssertionError(
        "A canonical product name genuinely changed "
        "during Step 3."
    )


# ------------------------------------------------------------
# 3. Canonical identity counts must remain unchanged
# ------------------------------------------------------------

step2_identity_count = (
    step2_ids_normalised
    .nunique()
)


step3_identity_count = (
    step3_ids_normalised
    .nunique()
)


assert step2_identity_count == 228, (
    "Step 2 should contain 228 canonical identities."
)


assert step3_identity_count == 228, (
    "Step 3 should contain 228 canonical identities."
)


# ------------------------------------------------------------
# 4. Validate the approved Step 3 metadata rows
# ------------------------------------------------------------

assert int(
    step3_mapping[
        "PriceTierMetadataApplied_Step3"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
) == 15, (
    "Exactly 15 products should receive "
    "price-tier metadata."
)


assert int(
    step3_mapping[
        "PriceChangeDetected_Step3"
    ]
    .fillna(False)
    .astype(bool)
    .sum()
) == 6, (
    "Exactly six tier products should have "
    "confirmed price changes."
)


assert not step3_mapping[
    "PriceIdentityChanged_Step3"
].fillna(False).astype(bool).any(), (
    "Step 3 should not contain any product-identity changes."
)


# ------------------------------------------------------------
# 5. Only the approved PLUs may receive tier metadata
# ------------------------------------------------------------

metadata_applied_mask = (
    step3_mapping[
        "PriceTierMetadataApplied_Step3"
    ]
    .fillna(False)
    .astype(bool)
)


metadata_codes = set(
    step3_mapping.loc[
        metadata_applied_mask,
        "PLUCode",
    ]
)


assert metadata_codes == approved_price_tier_plu_codes, (
    "The PLUs receiving price-tier metadata do not "
    "exactly match the approved list."
)


# ------------------------------------------------------------
# 6. Validate required metadata on approved tier products
# ------------------------------------------------------------

assert step3_mapping.loc[
    metadata_applied_mask,
    "TierProductFamily_Step3",
].notna().all(), (
    "A price-tier product has no product-family metadata."
)


assert step3_mapping.loc[
    metadata_applied_mask,
    "NominalPriceTier_Step3",
].notna().all(), (
    "A price-tier product has no nominal-price-tier metadata."
)


assert step3_mapping.loc[
    metadata_applied_mask,
    "MenuGeneration_Step3",
].notna().all(), (
    "A price-tier product has no menu-generation metadata."
)


assert step3_mapping.loc[
    metadata_applied_mask,
    "InitialObservedUnitPrice_Step3",
].notna().all(), (
    "A price-tier product has no initial observed price."
)


assert step3_mapping.loc[
    metadata_applied_mask,
    "CurrentObservedUnitPrice_Step3",
].notna().all(), (
    "A price-tier product has no current observed price."
)


# ------------------------------------------------------------
# 7. KIMBOX and KIMBOCK must remain separate
# ------------------------------------------------------------

kimbox_ids = set(
    normalise_text_for_comparison(
        step3_mapping.loc[
            step3_mapping[
                "TierProductFamily_Step3"
            ].eq("KIMBOX"),
            "CanonicalProductID_Step3",
        ]
    )
)


kimbock_ids = set(
    normalise_text_for_comparison(
        step3_mapping.loc[
            step3_mapping[
                "TierProductFamily_Step3"
            ].eq("KIMBOCK"),
            "CanonicalProductID_Step3",
        ]
    )
)


assert len(kimbox_ids) == 3, (
    "Expected three separate KIMBOX tier identities."
)


assert len(kimbock_ids) == 3, (
    "Expected three separate KIMBOCK tier identities."
)


assert kimbox_ids.isdisjoint(
    kimbock_ids
), (
    "KIMBOX and KIMBOCK were accidentally combined."
)


# ------------------------------------------------------------
# 8. Dinner, Kimbock, Kimbox, Mains and Vegetarian Mains
#    must remain distinct product families
# ------------------------------------------------------------

expected_tier_family_counts = {
    "DINNER": 3,
    "KIMBOCK": 3,
    "KIMBOX": 3,
    "MAINS": 3,
    "VEGETARIAN MAINS": 3,
}


actual_tier_family_counts = (
    step3_mapping.loc[
        metadata_applied_mask,
        "TierProductFamily_Step3",
    ]
    .value_counts()
    .to_dict()
)


assert actual_tier_family_counts == (
    expected_tier_family_counts
), (
    "Unexpected number of products in one or more "
    "price-tier families.\n"
    f"Expected: {expected_tier_family_counts}\n"
    f"Found: {actual_tier_family_counts}"
)


# ------------------------------------------------------------
# 9. Branded-beverage mappings from Step 2 must remain intact
# ------------------------------------------------------------

step2_branded_mask = (
    step3_mapping[
        "CanonicalProductName_Step2"
    ]
    .astype("string")
    .str.startswith(
        "B-",
        na=False
    )
)


assert int(
    step2_branded_mask.sum()
) == 17, (
    "Expected 17 branded source PLUs from Step 2."
)


branded_step2_ids = normalise_text_for_comparison(
    step3_mapping.loc[
        step2_branded_mask,
        "CanonicalProductID_Step2",
    ]
)


branded_step3_ids = normalise_text_for_comparison(
    step3_mapping.loc[
        step2_branded_mask,
        "CanonicalProductID_Step3",
    ]
)


assert (
    branded_step2_ids
    .reset_index(drop=True)
    .eq(
        branded_step3_ids
        .reset_index(drop=True)
    )
    .all()
), (
    "A branded-beverage identity from Step 2 "
    "changed during Step 3."
)


# ------------------------------------------------------------
# 10. Mapping totals must remain unchanged
# ------------------------------------------------------------

assert len(step3_mapping) == 236, (
    "The mapping should contain 236 PLU rows."
)


assert step3_mapping[
    "PLUCode"
].is_unique, (
    "PLUCode is not unique in the Step 3 mapping."
)


assert int(
    step3_mapping["TotalUnits"].sum()
) == 141_480, (
    "The mapped UnitSold total changed."
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("=" * 72)
print("STEP 3 STRICT ISOLATION VALIDATION PASSED")
print("=" * 72)
print()

print(
    "Step 2 canonical identities:",
    step2_identity_count
)

print(
    "Step 3 canonical identities:",
    step3_identity_count
)

print(
    "Actual canonical ID changes:",
    len(actual_id_differences)
)

print(
    "Actual canonical name changes:",
    len(actual_name_differences)
)

print(
    "Product identities changed:",
    int(
        step3_mapping[
            "PriceIdentityChanged_Step3"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )
)

print(
    "Tier metadata rows:",
    int(metadata_applied_mask.sum())
)

print(
    "Confirmed price-change products:",
    int(
        step3_mapping[
            "PriceChangeDetected_Step3"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )
)

print(
    "Total units preserved:",
    f"{int(step3_mapping['TotalUnits'].sum()):,}"
)

print()
print("Price-tier family counts:")

display(
    step3_mapping.loc[
        metadata_applied_mask,
        "TierProductFamily_Step3",
    ]
    .value_counts()
    .rename_axis("TierProductFamily")
    .reset_index(name="Products")
)

STEP 3 STRICT ISOLATION VALIDATION PASSED

Step 2 canonical identities: 228
Step 3 canonical identities: 228
Actual canonical ID changes: 0
Actual canonical name changes: 0
Product identities changed: 0
Tier metadata rows: 15
Confirmed price-change products: 6
Total units preserved: 141,480

Price-tier family counts:


,TierProductFamily,Products
0,KIMBOX,3
1,MAINS,3
2,VEGETARIAN MAINS,3
3,KIMBOCK,3
4,DINNER,3


In [36]:
# ============================================================
# Step 3 - Cell 11
# Join for validation only and preserve source transactions
# ============================================================

step3_transactions_with_order = (
    step3_transactions.copy()
)


step3_transactions_with_order[
    "_SourceRowOrder"
] = np.arange(
    len(step3_transactions_with_order)
)


step3_validation_join = (
    step3_transactions_with_order
    .merge(
        step3_mapping[
            [
                "PLUCode",
                "CanonicalProductID_Step2",
                "CanonicalProductName_Step2",
                "CanonicalProductID_Step3",
                "CanonicalProductName_Step3",
                "TierProductFamily_Step3",
                "NominalPriceTier_Step3",
                "PriceTierMetadataApplied_Step3",
                "PriceChangeDetected_Step3",
            ]
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values(
        "_SourceRowOrder"
    )
    .reset_index(drop=True)
)


source_comparison = (
    step3_transactions
    .reset_index(drop=True)
)


assert len(step3_validation_join) == (
    step3_source_row_count
)


assert step3_validation_join[
    "CanonicalProductID_Step3"
].notna().all()


# Confirm every source column remained unchanged
for source_column in step3_source_columns:

    assert (
        step3_validation_join[
            source_column
        ]
        .reset_index(drop=True)
        .equals(
            source_comparison[
                source_column
            ]
            .reset_index(drop=True)
        )
    ), (
        f"Source column changed during Step 3 join: "
        f"{source_column}"
    )


assert int(
    step3_validation_join[
        "UnitSold"
    ].sum()
) == step3_source_unit_total


joined_value_cents = int(
    (
        step3_validation_join[
            "TransValue"
        ]
        * 100
    )
    .round()
    .sum()
)


assert joined_value_cents == (
    step3_source_value_cents
)


tier_join_mask = (
    step3_validation_join[
        "PriceTierMetadataApplied_Step3"
    ]
)


assert int(
    tier_join_mask.sum()
) == 32_733


assert int(
    step3_validation_join.loc[
        tier_join_mask,
        "UnitSold",
    ].sum()
) == 33_552


print("Full transaction validation passed.")
print()
print(
    "Transaction rows unchanged:",
    f"{len(step3_validation_join):,}"
)
print(
    "UnitSold total unchanged:",
    f"{int(step3_validation_join['UnitSold'].sum()):,}"
)
print(
    "Transaction value unchanged:",
    joined_value_cents == step3_source_value_cents
)
print(
    "Price-tier transaction rows:",
    f"{int(tier_join_mask.sum()):,}"
)
print(
    "Price-tier units:",
    f"{int(step3_validation_join.loc[tier_join_mask, 'UnitSold'].sum()):,}"
)

Full transaction validation passed.

Transaction rows unchanged: 138,983
UnitSold total unchanged: 141,480
Transaction value unchanged: True
Price-tier transaction rows: 32,733
Price-tier units: 33,552


In [37]:
# ============================================================
# Step 3 - Cell 12
# Create final price-tier product audit
# ============================================================

price_tier_product_audit = (
    step3_mapping.loc[
        step3_mapping[
            "PriceTierMetadataApplied_Step3"
        ],
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID_Step2",
            "CanonicalProductName_Step2",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
            "TierProductFamily_Step3",
            "NominalPriceTier_Step3",
            "LegacyMenuLevel_Step3",
            "MenuGeneration_Step3",
            "InitialObservedUnitPrice_Step3",
            "CurrentObservedUnitPrice_Step3",
            "ObservedUnitPrices_Step3",
            "PriceChangeDetected_Step3",
            "PriceChangeDate_Step3",
            "GroupCode",
            "GroupName",
            "FirstObservedDate",
            "LastObservedDate",
            "TransactionRows",
            "TotalUnits",
            "TotalTransactionValue",
            "PriceIdentityChanged_Step3",
            "PriceTierProcessingType_Step3",
            "PriceTierProcessingReason_Step3",
            "PriceTierProcessingStatus_Step3",
            "PriceTierProcessingVersion_Step3",
        ],
    ]
    .sort_values(
        [
            "TierProductFamily_Step3",
            "NominalPriceTier_Step3",
            "PLUCode",
        ]
    )
    .reset_index(drop=True)
)


assert len(price_tier_product_audit) == 15


assert int(
    price_tier_product_audit[
        "TotalUnits"
    ].sum()
) == 33_552


assert not price_tier_product_audit[
    "PriceIdentityChanged_Step3"
].any()


display(price_tier_product_audit)

,PLUCode,PLUName_Original,PLUName_Corrected,CanonicalProductID_Step2,CanonicalProductName_Step2,CanonicalProductID_Step3,CanonicalProductName_Step3,TierProductFamily_Step3,NominalPriceTier_Step3,LegacyMenuLevel_Step3,...,FirstObservedDate,LastObservedDate,TransactionRows,TotalUnits,TotalTransactionValue,PriceIdentityChanged_Step3,PriceTierProcessingType_Step3,PriceTierProcessingReason_Step3,PriceTierProcessingStatus_Step3,PriceTierProcessingVersion_Step3
0,42529,€5.00 DINNER,€5.00 DINNER,PLU_42529,€5.00 DINNER,PLU_42529,€5.00 DINNER,DINNER,5,<NA>,...,2025-10-14,2026-03-30,3312,3312,17060.4,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
1,42530,€7 DINNER,€7 DINNER,PLU_42530,€7 DINNER,PLU_42530,€7 DINNER,DINNER,7,<NA>,...,2025-10-14,2026-03-30,3192,3192,22650.0,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
2,42531,€9 DINNER,€9 DINNER,PLU_42531,€9 DINNER,PLU_42531,€9 DINNER,DINNER,9,<NA>,...,2025-11-27,2026-03-27,724,724,6694.2,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
3,42532,€5 KIMBOCK,€5 KIMBOCK,PLU_42532,€5 KIMBOCK,PLU_42532,€5 KIMBOCK,KIMBOCK,5,<NA>,...,2025-10-14,2026-03-30,1335,1335,6848.7,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
4,42533,€7 KIMBOCK,€7 KIMBOCK,PLU_42533,€7 KIMBOCK,PLU_42533,€7 KIMBOCK,KIMBOCK,7,<NA>,...,2025-10-14,2026-03-30,10946,10946,78117.8,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
5,42534,€9 KIMBOCK,€9 KIMBOCK,PLU_42534,€9 KIMBOCK,PLU_42534,€9 KIMBOCK,KIMBOCK,9,<NA>,...,2025-10-15,2026-03-25,14,94,847.8,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
6,4241475,KIMBOX MAINS 1,KIMBOX MAINS 1,PLU_4241475,KIMBOX MAINS 1,PLU_4241475,KIMBOX MAINS 1,KIMBOX,5,1,...,2025-04-01,2025-10-13,1037,1037,5185.0,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
7,4241476,KIMBOX MAINS 2,KIMBOX MAINS 2,PLU_4241476,KIMBOX MAINS 2,PLU_4241476,KIMBOX MAINS 2,KIMBOX,7,2,...,2025-04-01,2025-10-13,6575,6623,46361.0,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
8,4241477,KIMBOX MAINS 3,KIMBOX MAINS 3,PLU_4241477,KIMBOX MAINS 3,PLU_4241477,KIMBOX MAINS 3,KIMBOX,9,3,...,2025-07-02,2025-09-02,5,5,45.0,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1
9,4241478,MAINS 1,MAINS 1,PLU_4241478,MAINS 1,PLU_4241478,MAINS 1,MAINS,5,1,...,2025-04-01,2025-10-09,1903,1903,9515.0,False,METADATA_ONLY_NO_PRODUCT_MERGE,Nominal menu tier recorded independently from ...,APPROVED_STEP_3,STEP_3_PRICE_TIER_METADATA_V1


In [38]:
# ============================================================
# Step 3 - Cell 13
# Save Step 3 mapping and audit outputs
# ============================================================

step3_mapping.to_csv(
    STEP3_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


price_tier_product_audit.to_csv(
    STEP3_TIER_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


price_tier_price_periods.to_csv(
    STEP3_PRICE_PERIOD_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


price_change_transitions.to_csv(
    STEP3_PRICE_TRANSITION_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


other_dinner_price_changes.to_csv(
    STEP3_OTHER_DINNER_PRICE_CHANGES_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


print("Step 3 files saved successfully.")
print()

print("1. Step 3 master mapping:")
print(STEP3_MAPPING_OUTPUT_FILE)

print()
print("2. Price-tier product audit:")
print(STEP3_TIER_AUDIT_OUTPUT_FILE)

print()
print("3. Price periods:")
print(STEP3_PRICE_PERIOD_OUTPUT_FILE)

print()
print("4. Price-change transitions:")
print(STEP3_PRICE_TRANSITION_OUTPUT_FILE)

print()
print("5. Other DINNER price changes:")
print(STEP3_OTHER_DINNER_PRICE_CHANGES_FILE)

Step 3 files saved successfully.

1. Step 3 master mapping:
eden_datasets/UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv

2. Price-tier product audit:
eden_datasets/UL_EDEN_step3_price_tier_product_audit.csv

3. Price periods:
eden_datasets/UL_EDEN_step3_price_tier_price_periods.csv

4. Price-change transitions:
eden_datasets/UL_EDEN_step3_price_change_transitions.csv

5. Other DINNER price changes:
eden_datasets/UL_EDEN_step3_other_dinner_price_changes.csv


In [39]:
# ============================================================
# Step 3 - Cell 14
# Reload and validate the saved Step 3 mapping
# ============================================================

saved_step3_mapping = pd.read_csv(
    STEP3_MAPPING_OUTPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
        "PriceChangeDate_Step3",
    ]
)


saved_step3_mapping["PLUCode"] = (
    pd.to_numeric(
        saved_step3_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


saved_step3_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_step3_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


def convert_saved_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    converted = normalized.map({
        "true": True,
        "false": False,
    })

    if converted.isna().any():
        invalid_values = (
            series.loc[
                converted.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Invalid saved Boolean values:\n"
            f"{invalid_values}"
        )

    return converted.astype(bool)


for boolean_column in [
    "PriceTierMetadataApplied_Step3",
    "PriceIdentityChanged_Step3",
    "PriceChangeDetected_Step3",
]:

    saved_step3_mapping[
        boolean_column
    ] = convert_saved_boolean(
        saved_step3_mapping[
            boolean_column
        ]
    )


# ------------------------------------------------------------
# Final saved-file validation
# ------------------------------------------------------------

assert len(saved_step3_mapping) == 236


assert saved_step3_mapping[
    "PLUCode"
].is_unique


assert saved_step3_mapping[
    "CanonicalProductID_Step2"
].nunique() == 228


assert saved_step3_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228


assert (
    saved_step3_mapping[
        "CanonicalProductID_Step3"
    ]
    .equals(
        saved_step3_mapping[
            "CanonicalProductID_Step2"
        ]
    )
)


assert (
    saved_step3_mapping[
        "CanonicalProductName_Step3"
    ]
    .equals(
        saved_step3_mapping[
            "CanonicalProductName_Step2"
        ]
    )
)


assert int(
    saved_step3_mapping[
        "PriceTierMetadataApplied_Step3"
    ].sum()
) == 15


assert int(
    saved_step3_mapping[
        "PriceChangeDetected_Step3"
    ].sum()
) == 6


assert not saved_step3_mapping[
    "PriceIdentityChanged_Step3"
].any()


assert int(
    saved_step3_mapping["TotalUnits"].sum()
) == 141_480


saved_tier_units = int(
    saved_step3_mapping.loc[
        saved_step3_mapping[
            "PriceTierMetadataApplied_Step3"
        ],
        "TotalUnits",
    ].sum()
)


assert saved_tier_units == 33_552


print("=" * 74)
print("STEP 3 PRICE-TIER METADATA COMPLETED")
print("=" * 74)
print()
print(
    "Mapping rows:",
    len(saved_step3_mapping)
)
print(
    "Step 2 canonical identities:",
    saved_step3_mapping[
        "CanonicalProductID_Step2"
    ].nunique()
)
print(
    "Step 3 canonical identities:",
    saved_step3_mapping[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Product identities changed:",
    int(
        saved_step3_mapping[
            "PriceIdentityChanged_Step3"
        ].sum()
    )
)
print(
    "Price-tier PLUs labelled:",
    int(
        saved_step3_mapping[
            "PriceTierMetadataApplied_Step3"
        ].sum()
    )
)
print(
    "Products with confirmed price changes:",
    int(
        saved_step3_mapping[
            "PriceChangeDetected_Step3"
        ].sum()
    )
)
print(
    "Price-tier units:",
    f"{saved_tier_units:,}"
)
print(
    "All source units preserved:",
    f"{int(saved_step3_mapping['TotalUnits'].sum()):,}"
)
print()
print("Final Step 3 mapping:")
print(STEP3_MAPPING_OUTPUT_FILE)

STEP 3 PRICE-TIER METADATA COMPLETED

Mapping rows: 236
Step 2 canonical identities: 228
Step 3 canonical identities: 228
Product identities changed: 0
Price-tier PLUs labelled: 15
Products with confirmed price changes: 6
Price-tier units: 33,552
All source units preserved: 141,480

Final Step 3 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv


### Now I have to seperate the bulk demand as that is really important.

In [41]:
# ============================================================
# Step 4 - Cell 1
# Load immutable transactions, correction audit and
# completed Step 3 product mapping
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


# Immutable corrected transaction source
TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


# Full unit-correction audit created earlier
UNIT_CORRECTION_AUDIT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_unitsold_corrected_with_audit_columns.csv"
)


# Completed product mapping from Step 3
STEP3_MAPPING_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv"
)


# Step 4 outputs
STEP4_ENRICHED_TRANSACTION_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_step4_bulk_separated_enriched.csv"
)


STEP4_BULK_LINE_REGISTRY_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_confirmed_bulk_line_registry.csv"
)


STEP4_BULK_PRODUCT_SUMMARY_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_bulk_product_summary.csv"
)


STEP4_DAILY_SPLIT_AUDIT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_observed_daily_demand_split_audit.csv"
)


STEP4_CLASSIFICATION_SUMMARY_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_bulk_classification_summary.csv"
)


for required_file in [
    TRANSACTION_INPUT_FILE,
    UNIT_CORRECTION_AUDIT_FILE,
    STEP3_MAPPING_INPUT_FILE,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Required Step 4 input file was not found:\n"
            f"{required_file}"
        )


step4_source_transactions = pd.read_csv(
    TRANSACTION_INPUT_FILE
)


step4_unit_audit = pd.read_csv(
    UNIT_CORRECTION_AUDIT_FILE
)


step4_product_mapping = pd.read_csv(
    STEP3_MAPPING_INPUT_FILE,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
        "PriceChangeDate_Step3",
    ]
)


# Preserve original transaction-column order
step4_source_columns = (
    step4_source_transactions.columns.tolist()
)


# ------------------------------------------------------------
# Safe type conversions
# ------------------------------------------------------------

for dataframe in [
    step4_source_transactions,
    step4_unit_audit,
]:

    dataframe["TransDate"] = pd.to_datetime(
        dataframe["TransDate"],
        errors="raise"
    )

    dataframe["Date"] = (
        pd.to_datetime(
            dataframe["Date"],
            errors="raise"
        )
        .dt.normalize()
    )

    dataframe["PLUCode"] = (
        pd.to_numeric(
            dataframe["PLUCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["GroupCode"] = (
        pd.to_numeric(
            dataframe["GroupCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["UnitSold"] = (
        pd.to_numeric(
            dataframe["UnitSold"],
            errors="raise"
        )
        .round()
        .astype("int64")
    )

    dataframe["TransValue"] = pd.to_numeric(
        dataframe["TransValue"],
        errors="raise"
    )


step4_unit_audit["UnitSold_Original"] = (
    pd.to_numeric(
        step4_unit_audit["UnitSold_Original"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step4_unit_audit["UnitPrice"] = pd.to_numeric(
    step4_unit_audit["UnitPrice"],
    errors="coerce"
)


step4_product_mapping["PLUCode"] = (
    pd.to_numeric(
        step4_product_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step4_product_mapping["TotalUnits"] = (
    pd.to_numeric(
        step4_product_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Robust Boolean converter
# ------------------------------------------------------------

def convert_csv_boolean(
    series,
    allow_missing=False
):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    converted = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })

    missing_text_mask = normalized.isin([
        "",
        "<na>",
        "nan",
        "none",
    ])

    invalid_mask = (
        converted.isna()
        & ~missing_text_mask
    )

    if invalid_mask.any():
        invalid_values = (
            series.loc[invalid_mask]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unexpected Boolean values found:\n"
            f"{invalid_values}"
        )

    if allow_missing:
        return converted.astype("boolean")

    if converted.isna().any():
        raise ValueError(
            "A required Boolean column contains "
            "missing values."
        )

    return converted.astype(bool)


step4_unit_audit[
    "MultiUnitAdjustedFlag_Parsed"
] = convert_csv_boolean(
    step4_unit_audit[
        "MultiUnitAdjustedFlag"
    ]
)


step4_unit_audit[
    "IsExactPriceMultiple_Parsed"
] = convert_csv_boolean(
    step4_unit_audit[
        "IsExactPriceMultiple"
    ],
    allow_missing=True
)


# ------------------------------------------------------------
# Initial validation
# ------------------------------------------------------------

assert len(step4_source_transactions) == 138_983

assert len(step4_unit_audit) == 138_983

assert len(step4_product_mapping) == 236

assert step4_product_mapping["PLUCode"].is_unique

assert step4_product_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228

assert int(
    step4_source_transactions["UnitSold"].sum()
) == 141_480

assert int(
    step4_product_mapping["TotalUnits"].sum()
) == 141_480


print("Step 4 input files loaded and validated.")
print()
print(
    "Transaction rows:",
    f"{len(step4_source_transactions):,}"
)
print(
    "Source units:",
    f"{int(step4_source_transactions['UnitSold'].sum()):,}"
)
print(
    "Step 3 canonical products:",
    step4_product_mapping[
        "CanonicalProductID_Step3"
    ].nunique()
)

Step 4 input files loaded and validated.

Transaction rows: 138,983
Source units: 141,480
Step 3 canonical products: 228


In [42]:
# ============================================================
# Step 4 - Cell 2
# Confirm that the unit-correction audit and source file
# represent the same rows in the same order
# ============================================================

missing_audit_source_columns = [
    column
    for column in step4_source_columns
    if column not in step4_unit_audit.columns
]


if missing_audit_source_columns:
    raise ValueError(
        "The full audit is missing original source columns:\n"
        f"{missing_audit_source_columns}"
    )


def prepare_alignment_frame(
    dataframe,
    columns
):
    output = pd.DataFrame(
        index=dataframe.index
    )

    date_columns = {
        "TransDate",
        "Date",
    }

    numeric_columns = {
        "TransValue",
        "GroupCode",
        "PLUCode",
        "Hour",
        "Month",
        "WeekOfYear",
        "UnitSold",
    }

    for column in columns:

        if column in date_columns:
            output[column] = pd.to_datetime(
                dataframe[column],
                errors="raise"
            )

        elif column in numeric_columns:
            output[column] = pd.to_numeric(
                dataframe[column],
                errors="raise"
            )

        else:
            output[column] = (
                dataframe[column]
                .astype("string")
                .str.strip()
                .fillna("<MISSING>")
            )

    return output


source_alignment = prepare_alignment_frame(
    step4_source_transactions,
    step4_source_columns
)


audit_alignment = prepare_alignment_frame(
    step4_unit_audit,
    step4_source_columns
)


pd.testing.assert_frame_equal(
    source_alignment,
    audit_alignment,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000000001,
)


print(
    "The correction audit matches the transaction "
    "source row by row."
)
print()
print(
    "Matched rows:",
    f"{len(source_alignment):,}"
)

The correction audit matches the transaction source row by row.

Matched rows: 138,983


In [43]:
# ============================================================
# Step 4 - Cell 3
# Validate the 61 confirmed bulk transaction lines
# ============================================================

step4_confirmed_bulk_flag = (
    step4_unit_audit[
        "MultiUnitAdjustedFlag_Parsed"
    ]
    .astype(bool)
)


assert int(
    step4_confirmed_bulk_flag.sum()
) == 61, (
    "Expected exactly 61 confirmed bulk lines."
)


# Every confirmed line must have more than one unit
assert step4_source_transactions.loc[
    step4_confirmed_bulk_flag,
    "UnitSold",
].gt(1).all(), (
    "A confirmed bulk row has UnitSold <= 1."
)


# Every confirmed bulk line originally had UnitSold = 1
assert step4_unit_audit.loc[
    step4_confirmed_bulk_flag,
    "UnitSold_Original",
].eq(1).all()


# All confirmed rows must have an exact unit-price multiple
assert step4_unit_audit.loc[
    step4_confirmed_bulk_flag,
    "IsExactPriceMultiple_Parsed",
].eq(True).all(), (
    "A confirmed bulk row is not an exact price multiple."
)


assert step4_unit_audit.loc[
    step4_confirmed_bulk_flag,
    "UnitPrice",
].notna().all()


# Confirm the audit flag exactly matches the corrected
# multi-unit rows in the current source file.
assert step4_confirmed_bulk_flag.reset_index(
    drop=True
).equals(
    step4_source_transactions[
        "UnitSold"
    ]
    .gt(1)
    .reset_index(drop=True)
), (
    "The confirmed bulk flag does not match the "
    "multi-unit transaction rows."
)


step4_confirmed_bulk_units = int(
    step4_source_transactions.loc[
        step4_confirmed_bulk_flag,
        "UnitSold",
    ].sum()
)


step4_confirmed_extra_units = int(
    (
        step4_source_transactions.loc[
            step4_confirmed_bulk_flag,
            "UnitSold",
        ]
        -
        step4_unit_audit.loc[
            step4_confirmed_bulk_flag,
            "UnitSold_Original",
        ]
    ).sum()
)


assert step4_confirmed_bulk_units == 2_558

assert step4_confirmed_extra_units == 2_497


assert step4_source_transactions.loc[
    step4_confirmed_bulk_flag,
    "PLUCode",
].nunique() == 11


print("Confirmed bulk set validated.")
print()
print(
    "Confirmed bulk lines:",
    f"{int(step4_confirmed_bulk_flag.sum()):,}"
)
print(
    "Confirmed bulk products:",
    step4_source_transactions.loc[
        step4_confirmed_bulk_flag,
        "PLUCode",
    ].nunique()
)
print(
    "Confirmed bulk units:",
    f"{step4_confirmed_bulk_units:,}"
)
print(
    "Additional units identified:",
    f"{step4_confirmed_extra_units:,}"
)

Confirmed bulk set validated.

Confirmed bulk lines: 61
Confirmed bulk products: 11
Confirmed bulk units: 2,558
Additional units identified: 2,497


In [44]:
# ============================================================
# Step 4 - Cell 4
# Create isolated line-level bulk and normal demand fields
# ============================================================

step4_transactions = (
    step4_source_transactions.copy()
)


# Stable source-row reference
step4_transactions[
    "SourceRowNumber_Step4"
] = np.arange(
    1,
    len(step4_transactions) + 1
)


step4_transactions[
    "BulkOrderFlag_Step4"
] = step4_confirmed_bulk_flag.to_numpy()


# Full confirmed bulk line belongs to bulk demand.
step4_transactions[
    "BulkDemandUnits_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    step4_transactions["UnitSold"],
    0
).astype("int64")


step4_transactions[
    "NormalDemandUnits_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    0,
    step4_transactions["UnitSold"]
).astype("int64")


step4_transactions[
    "TotalDemandUnits_Step4"
] = step4_transactions[
    "UnitSold"
].astype("int64")


# This is an audit measure only.
step4_transactions[
    "BulkExtraUnitsAboveOriginalOne_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    (
        step4_transactions["UnitSold"]
        -
        step4_unit_audit[
            "UnitSold_Original"
        ].to_numpy()
    ),
    0
).astype("int64")


step4_transactions[
    "UnitSoldOriginal_Audit_Step4"
] = step4_unit_audit[
    "UnitSold_Original"
].to_numpy()


step4_transactions[
    "BulkUnitPrice_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    step4_unit_audit["UnitPrice"],
    np.nan
)


step4_transactions[
    "BulkAdjustmentMethod_Step4"
] = pd.Series(
    pd.NA,
    index=step4_transactions.index,
    dtype="string"
)


step4_transactions.loc[
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "BulkAdjustmentMethod_Step4",
] = (
    step4_unit_audit.loc[
        step4_confirmed_bulk_flag,
        "UnitSoldAdjustmentMethod",
    ]
    .astype("string")
    .to_numpy()
)


step4_transactions[
    "BulkClassification_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "CONFIRMED_BULK_ORDER_LINE",
    "NORMAL_OR_UNCLASSIFIED_LINE"
)


step4_transactions[
    "BulkClassificationReason_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    (
        "Confirmed multi-unit transaction line from "
        "the UnitSold correction audit."
    ),
    (
        "No confirmed multi-unit bulk evidence in "
        "the UnitSold correction audit."
    )
)


step4_transactions[
    "BulkClassificationVersion_Step4"
] = "STEP_4_CONFIRMED_BULK_V1"


# Create a unique registry ID only for bulk lines
step4_transactions[
    "BulkOrderLineID_Step4"
] = pd.Series(
    pd.NA,
    index=step4_transactions.index,
    dtype="string"
)


bulk_line_ids = [
    f"BULK_LINE_{number:04d}"
    for number in range(
        1,
        int(
            step4_transactions[
                "BulkOrderFlag_Step4"
            ].sum()
        ) + 1
    )
]


step4_transactions.loc[
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "BulkOrderLineID_Step4",
] = bulk_line_ids


# ------------------------------------------------------------
# Validate the split
# ------------------------------------------------------------

assert (
    step4_transactions[
        "NormalDemandUnits_Step4"
    ]
    +
    step4_transactions[
        "BulkDemandUnits_Step4"
    ]
    ==
    step4_transactions[
        "TotalDemandUnits_Step4"
    ]
).all()


assert (
    step4_transactions[
        "TotalDemandUnits_Step4"
    ]
    ==
    step4_transactions["UnitSold"]
).all()


assert int(
    step4_transactions[
        "BulkDemandUnits_Step4"
    ].sum()
) == 2_558


assert int(
    step4_transactions[
        "NormalDemandUnits_Step4"
    ].sum()
) == 138_922


assert int(
    step4_transactions[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 2_497


assert step4_transactions.loc[
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "NormalDemandUnits_Step4",
].eq(0).all()


assert step4_transactions.loc[
    ~step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "BulkDemandUnits_Step4",
].eq(0).all()


print("Line-level demand split created.")
print()
print(
    "Normal demand units:",
    f"{int(step4_transactions['NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "Bulk demand units:",
    f"{int(step4_transactions['BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Total units:",
    f"{int(step4_transactions['TotalDemandUnits_Step4'].sum()):,}"
)

Line-level demand split created.

Normal demand units: 138,922
Bulk demand units: 2,558
Total units: 141,480


In [45]:
# ============================================================
# Step 4 - Cell 5
# Attach final Step 3 product identity
# ============================================================

step4_mapping_columns = [
    "PLUCode",
    "PLUName_Original",
    "PLUName_Corrected",
    "CanonicalProductID_Step3",
    "CanonicalProductName_Step3",
    "MappingStatus",
    "BeverageSeries_Step2",
    "BeverageType_Step2",
    "SupplierLabel_Step2",
    "TierProductFamily_Step3",
    "NominalPriceTier_Step3",
    "MenuGeneration_Step3",
]


missing_step4_mapping_columns = [
    column
    for column in step4_mapping_columns
    if column not in step4_product_mapping.columns
]


if missing_step4_mapping_columns:
    raise ValueError(
        "Step 3 mapping is missing required columns:\n"
        f"{missing_step4_mapping_columns}"
    )


step4_enriched_transactions = (
    step4_transactions
    .merge(
        step4_product_mapping[
            step4_mapping_columns
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values(
        "SourceRowNumber_Step4"
    )
    .reset_index(drop=True)
)


assert len(step4_enriched_transactions) == 138_983


assert step4_enriched_transactions[
    "CanonicalProductID_Step3"
].notna().all()


assert step4_enriched_transactions[
    "CanonicalProductName_Step3"
].notna().all()


# Confirm all original source columns remain unchanged
enriched_source_alignment = prepare_alignment_frame(
    step4_enriched_transactions,
    step4_source_columns
)


pd.testing.assert_frame_equal(
    source_alignment,
    enriched_source_alignment,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000000001,
)


step4_enriched_transactions[
    "ForecastingEligible_Step4"
] = ~(
    step4_enriched_transactions[
        "MappingStatus"
    ]
    .astype("string")
    .eq("EXCLUDE_FROM_FORECASTING")
)


print("Step 3 product mapping attached successfully.")
print()
print(
    "Transaction rows:",
    f"{len(step4_enriched_transactions):,}"
)
print(
    "Canonical products:",
    step4_enriched_transactions[
        "CanonicalProductID_Step3"
    ].nunique()
)

Step 3 product mapping attached successfully.

Transaction rows: 138,983
Canonical products: 228


In [46]:
# ============================================================
# Step 4 - Cell 6
# Validate OPEN UL exclusion and forecastable demand totals
# ============================================================

open_ul_by_name = (
    step4_enriched_transactions[
        "PLUName"
    ]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


open_ul_by_mapping = ~(
    step4_enriched_transactions[
        "ForecastingEligible_Step4"
    ]
)


assert open_ul_by_name.equals(
    open_ul_by_mapping
), (
    "The OPEN UL name flag and mapping status do not agree."
)


assert int(
    open_ul_by_name.sum()
) == 24_736


assert int(
    step4_enriched_transactions.loc[
        open_ul_by_name,
        "UnitSold",
    ].sum()
) == 24_736


assert not step4_enriched_transactions.loc[
    open_ul_by_name,
    "BulkOrderFlag_Step4",
].any(), (
    "An OPEN UL line was classified as bulk."
)


forecastable_step4_mask = (
    ~open_ul_by_name
)


forecastable_normal_units = int(
    step4_enriched_transactions.loc[
        forecastable_step4_mask,
        "NormalDemandUnits_Step4",
    ].sum()
)


forecastable_bulk_units = int(
    step4_enriched_transactions.loc[
        forecastable_step4_mask,
        "BulkDemandUnits_Step4",
    ].sum()
)


forecastable_total_units = int(
    step4_enriched_transactions.loc[
        forecastable_step4_mask,
        "TotalDemandUnits_Step4",
    ].sum()
)


assert int(
    forecastable_step4_mask.sum()
) == 114_247


assert forecastable_normal_units == 114_186

assert forecastable_bulk_units == 2_558

assert forecastable_total_units == 116_744


assert (
    forecastable_normal_units
    +
    forecastable_bulk_units
    ==
    forecastable_total_units
)


assert step4_enriched_transactions.loc[
    forecastable_step4_mask,
    "CanonicalProductID_Step3",
].nunique() == 227


print("Forecastable Step 4 totals validated.")
print()
print(
    "Forecastable transaction rows:",
    f"{int(forecastable_step4_mask.sum()):,}"
)
print(
    "Forecastable canonical products:",
    step4_enriched_transactions.loc[
        forecastable_step4_mask,
        "CanonicalProductID_Step3",
    ].nunique()
)
print(
    "Normal forecastable units:",
    f"{forecastable_normal_units:,}"
)
print(
    "Bulk forecastable units:",
    f"{forecastable_bulk_units:,}"
)
print(
    "Total forecastable units:",
    f"{forecastable_total_units:,}"
)

Forecastable Step 4 totals validated.

Forecastable transaction rows: 114,247
Forecastable canonical products: 227
Normal forecastable units: 114,186
Bulk forecastable units: 2,558
Total forecastable units: 116,744


In [47]:
# ============================================================
# Step 4 - Cell 7
# Create frozen registry of the 61 confirmed bulk lines
# ============================================================

confirmed_bulk_line_registry = (
    step4_enriched_transactions.loc[
        step4_enriched_transactions[
            "BulkOrderFlag_Step4"
        ],
        [
            "SourceRowNumber_Step4",
            "BulkOrderLineID_Step4",
            "Date",
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
            "GroupCode",
            "GroupName",
            "TransValue",
            "BulkUnitPrice_Step4",
            "UnitSoldOriginal_Audit_Step4",
            "UnitSold",
            "NormalDemandUnits_Step4",
            "BulkDemandUnits_Step4",
            "BulkExtraUnitsAboveOriginalOne_Step4",
            "BulkAdjustmentMethod_Step4",
            "BulkClassification_Step4",
            "BulkClassificationReason_Step4",
            "BulkClassificationVersion_Step4",
        ],
    ]
    .sort_values(
        [
            "Date",
            "TransactionID",
            "PLUCode",
            "SourceRowNumber_Step4",
        ]
    )
    .reset_index(drop=True)
)


assert len(confirmed_bulk_line_registry) == 61


assert confirmed_bulk_line_registry[
    "BulkOrderLineID_Step4"
].is_unique


assert int(
    confirmed_bulk_line_registry[
        "BulkDemandUnits_Step4"
    ].sum()
) == 2_558


assert int(
    confirmed_bulk_line_registry[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 2_497


assert confirmed_bulk_line_registry[
    "PLUCode"
].nunique() == 11


assert confirmed_bulk_line_registry[
    "CanonicalProductID_Step3"
].nunique() == 11


assert confirmed_bulk_line_registry[
    "NormalDemandUnits_Step4"
].eq(0).all()


print("Confirmed bulk-line registry created.")
print()
print(
    "Registry lines:",
    len(confirmed_bulk_line_registry)
)
print(
    "Bulk products:",
    confirmed_bulk_line_registry[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Bulk units:",
    f"{int(confirmed_bulk_line_registry['BulkDemandUnits_Step4'].sum()):,}"
)

display(
    confirmed_bulk_line_registry.head(20)
)

Confirmed bulk-line registry created.

Registry lines: 61
Bulk products: 11
Bulk units: 2,558


,SourceRowNumber_Step4,BulkOrderLineID_Step4,Date,TransDate,TransactionID,PLUCode,PLUName,PLUName_Original,PLUName_Corrected,CanonicalProductID_Step3,...,BulkUnitPrice_Step4,UnitSoldOriginal_Audit_Step4,UnitSold,NormalDemandUnits_Step4,BulkDemandUnits_Step4,BulkExtraUnitsAboveOriginalOne_Step4,BulkAdjustmentMethod_Step4,BulkClassification_Step4,BulkClassificationReason_Step4,BulkClassificationVersion_Step4
0,1010,BULK_LINE_0060,2025-05-29,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,FULL FAT CAN,FULL FAT CAN,PLU_2000000019,...,1.8,1,5,0,5,4,Adjusted: historical FULL FAT CAN transaction ...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
1,68,BULK_LINE_0049,2025-07-08,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,3219121,CARTON OF WATER,CARTON OF WATER,CARTON OF WATER,PLU_3219121,...,2.2,1,21,0,21,20,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
2,46,BULK_LINE_0038,2025-07-08,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,4241474,SOUP OF THE DAY,SOUP OF THE DAY,SOUP OF THE DAY,PLU_4241474,...,3.5,1,47,0,47,46,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
3,11,BULK_LINE_0011,2025-07-08,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,4241483,VEGT MAINS 3,VEGT MAINS 3,VEGT MAINS 3,PLU_4241483,...,9.0,1,47,0,47,46,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
4,67,BULK_LINE_0048,2025-07-08,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,2000000019,FULL FAT CAN,FULL FAT CAN,FULL FAT CAN,PLU_2000000019,...,1.8,1,26,0,26,25,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
5,52,BULK_LINE_0043,2025-07-09,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,3219121,CARTON OF WATER,CARTON OF WATER,CARTON OF WATER,PLU_3219121,...,2.2,1,29,0,29,28,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
6,37,BULK_LINE_0030,2025-07-09,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,4241474,SOUP OF THE DAY,SOUP OF THE DAY,SOUP OF THE DAY,PLU_4241474,...,3.5,1,57,0,57,56,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
7,3,BULK_LINE_0003,2025-07-09,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,4241480,MAINS 3,MAINS 3,MAINS 3,PLU_4241480,...,9.0,1,57,0,57,56,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
8,65,BULK_LINE_0047,2025-07-09,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,2000000019,FULL FAT CAN,FULL FAT CAN,FULL FAT CAN,PLU_2000000019,...,1.8,1,27,0,27,26,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1
9,39,BULK_LINE_0032,2025-07-10,2025-07-10 14:51:00,194801_2025-07-10_14-51-00,4241474,SOUP OF THE DAY,SOUP OF THE DAY,SOUP OF THE DAY,PLU_4241474,...,3.5,1,52,0,52,51,Adjusted: TransValue > €9.30 and exact multipl...,CONFIRMED_BULK_ORDER_LINE,Confirmed multi-unit transaction line from the...,STEP_4_CONFIRMED_BULK_V1


In [48]:
# ============================================================
# Step 4 - Cell 8
# Create product-level bulk-demand summary
# ============================================================

bulk_product_summary_step4 = (
    confirmed_bulk_line_registry
    .groupby(
        [
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
        ],
        as_index=False
    )
    .agg(
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        SourcePLUCodes=(
            "PLUCode",
            lambda values: ", ".join(
                str(value)
                for value in sorted(
                    values.unique()
                )
            )
        ),
        SourceProductNames=(
            "PLUName_Corrected",
            lambda values: " | ".join(
                sorted(
                    values.dropna().unique()
                )
            )
        ),
        BulkLineCount=(
            "BulkOrderLineID_Step4",
            "size"
        ),
        UniqueBulkTransactions=(
            "TransactionID",
            "nunique"
        ),
        FirstBulkDate=(
            "Date",
            "min"
        ),
        LastBulkDate=(
            "Date",
            "max"
        ),
        BulkDemandUnits=(
            "BulkDemandUnits_Step4",
            "sum"
        ),
        AdditionalUnitsIdentified=(
            "BulkExtraUnitsAboveOriginalOne_Step4",
            "sum"
        ),
        BulkTransactionValue=(
            "TransValue",
            "sum"
        ),
        MinimumBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "min"
        ),
        MaximumBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "max"
        ),
        MeanBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "mean"
        ),
    )
)


bulk_product_summary_step4[
    "BulkTransactionValue"
] = (
    bulk_product_summary_step4[
        "BulkTransactionValue"
    ]
    .round(2)
)


bulk_product_summary_step4[
    "MeanBulkLineUnits"
] = (
    bulk_product_summary_step4[
        "MeanBulkLineUnits"
    ]
    .round(2)
)


bulk_product_summary_step4 = (
    bulk_product_summary_step4
    .sort_values(
        "BulkDemandUnits",
        ascending=False
    )
    .reset_index(drop=True)
)


assert len(bulk_product_summary_step4) == 11


assert int(
    bulk_product_summary_step4[
        "BulkLineCount"
    ].sum()
) == 61


assert int(
    bulk_product_summary_step4[
        "BulkDemandUnits"
    ].sum()
) == 2_558


assert int(
    bulk_product_summary_step4[
        "AdditionalUnitsIdentified"
    ].sum()
) == 2_497


print("Bulk product summary created.")

display(bulk_product_summary_step4)

Bulk product summary created.


,CanonicalProductID_Step3,CanonicalProductName_Step3,SourcePLUCount,SourcePLUCodes,SourceProductNames,BulkLineCount,UniqueBulkTransactions,FirstBulkDate,LastBulkDate,BulkDemandUnits,AdditionalUnitsIdentified,BulkTransactionValue,MinimumBulkLineUnits,MaximumBulkLineUnits,MeanBulkLineUnits
0,PLU_4241474,SOUP OF THE DAY,1,4241474,SOUP OF THE DAY,15,11,2025-07-08,2025-08-01,719,704,2516.5,3,114,47.93
1,PLU_4241428,HAM & CHEESE SANDWICH CT,1,4241428,HAM & CHEESE SANDWICH CT,10,2,2025-09-24,2025-09-26,610,600,3050.0,61,61,61.00
2,PLU_4241480,MAINS 3,1,4241480,MAINS 3,9,7,2025-07-09,2025-08-01,446,437,4014.0,23,61,49.56
3,PLU_4241483,VEGT MAINS 3,1,4241483,VEGT MAINS 3,4,4,2025-07-08,2025-07-25,258,254,2322.0,45,114,64.50
4,PLU_3219121,CARTON OF WATER,1,3219121,CARTON OF WATER,10,8,2025-07-08,2025-08-01,240,230,528.0,15,30,24.00
5,PLU_2000000019,FULL FAT CAN,1,2000000019,FULL FAT CAN,7,5,2025-05-29,2025-07-25,93,86,167.4,4,27,13.29
6,PLU_42534,€9 KIMBOCK,1,42534,€9 KIMBOCK,2,2,2025-10-29,2025-10-29,82,80,738.0,41,41,41.00
7,PLU_4241430,BOX SALADS,1,4241430,BOX SALADS,1,1,2025-07-28,2025-07-28,49,48,318.5,49,49,49.00
8,PLU_4241476,KIMBOX MAINS 2,1,4241476,KIMBOX MAINS 2,1,1,2025-07-28,2025-07-28,49,48,343.0,49,49,49.00
9,PLU_4241425,CAFFE MOCHA BLISS BALLS,1,4241425,CAFFE MOCHA BLISS BALLS,1,1,2025-07-21,2025-07-21,6,5,21.0,6,6,6.00


In [49]:
# ============================================================
# Step 4 - Cell 9
# Create observed daily normal/bulk demand split
#
# No zero-demand rows are added here.
# Final daily panel creation remains Step 5.
# ============================================================

forecastable_step4_transactions = (
    step4_enriched_transactions.loc[
        forecastable_step4_mask
    ]
    .copy()
)


forecastable_step4_transactions[
    "_BulkLineCount"
] = (
    forecastable_step4_transactions[
        "BulkOrderFlag_Step4"
    ]
    .astype("int64")
)


observed_daily_demand_split_step4 = (
    forecastable_step4_transactions
    .groupby(
        [
            "Date",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
        ],
        as_index=False
    )
    .agg(
        NormalDemandUnits=(
            "NormalDemandUnits_Step4",
            "sum"
        ),
        BulkDemandUnits=(
            "BulkDemandUnits_Step4",
            "sum"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits_Step4",
            "sum"
        ),
        TransactionLineCount=(
            "TransactionID",
            "size"
        ),
        UniqueTransactionCount=(
            "TransactionID",
            "nunique"
        ),
        BulkLineCount=(
            "_BulkLineCount",
            "sum"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
    )
)


observed_daily_demand_split_step4[
    "HasBulkDemand"
] = (
    observed_daily_demand_split_step4[
        "BulkDemandUnits"
    ]
    .gt(0)
)


assert (
    observed_daily_demand_split_step4[
        "NormalDemandUnits"
    ]
    +
    observed_daily_demand_split_step4[
        "BulkDemandUnits"
    ]
    ==
    observed_daily_demand_split_step4[
        "TotalDemandUnits"
    ]
).all()


assert int(
    observed_daily_demand_split_step4[
        "NormalDemandUnits"
    ].sum()
) == 114_186


assert int(
    observed_daily_demand_split_step4[
        "BulkDemandUnits"
    ].sum()
) == 2_558


assert int(
    observed_daily_demand_split_step4[
        "TotalDemandUnits"
    ].sum()
) == 116_744


assert int(
    observed_daily_demand_split_step4[
        "BulkLineCount"
    ].sum()
) == 61


assert not observed_daily_demand_split_step4.duplicated(
    [
        "Date",
        "CanonicalProductID_Step3",
    ]
).any()


print("Observed daily demand split audit created.")
print()
print(
    "Observed canonical product-date rows:",
    f"{len(observed_daily_demand_split_step4):,}"
)
print(
    "Product-date rows containing bulk demand:",
    f"{int(observed_daily_demand_split_step4['HasBulkDemand'].sum()):,}"
)
print(
    "Normal units:",
    f"{int(observed_daily_demand_split_step4['NormalDemandUnits'].sum()):,}"
)
print(
    "Bulk units:",
    f"{int(observed_daily_demand_split_step4['BulkDemandUnits'].sum()):,}"
)
print(
    "Total units:",
    f"{int(observed_daily_demand_split_step4['TotalDemandUnits'].sum()):,}"
)

display(
    observed_daily_demand_split_step4.loc[
        observed_daily_demand_split_step4[
            "HasBulkDemand"
        ]
    ]
    .sort_values(
        "BulkDemandUnits",
        ascending=False
    )
    .head(20)
)

Observed daily demand split audit created.

Observed canonical product-date rows: 15,138
Product-date rows containing bulk demand: 42
Normal units: 114,186
Bulk units: 2,558
Total units: 116,744


,Date,CanonicalProductID_Step3,CanonicalProductName_Step3,NormalDemandUnits,BulkDemandUnits,TotalDemandUnits,TransactionLineCount,UniqueTransactionCount,BulkLineCount,SourcePLUCount,HasBulkDemand
7856,2025-09-26,PLU_4241428,HAM & CHEESE SANDWICH CT,1,305,306,6,2,5,1,True
7761,2025-09-24,PLU_4241428,HAM & CHEESE SANDWICH CT,4,305,309,9,5,5,1,True
4989,2025-07-16,PLU_4241474,SOUP OF THE DAY,2,114,116,3,3,1,1,True
4995,2025-07-16,PLU_4241483,VEGT MAINS 3,0,114,114,1,1,1,1,True
5266,2025-07-22,PLU_4241480,MAINS 3,0,98,98,2,1,2,1,True
5262,2025-07-22,PLU_4241474,SOUP OF THE DAY,0,98,98,2,1,2,1,True
5709,2025-07-31,PLU_4241474,SOUP OF THE DAY,6,83,89,9,6,3,1,True
9046,2025-10-29,PLU_42534,€9 KIMBOCK,0,82,82,2,2,2,1,True
5714,2025-07-31,PLU_4241480,MAINS 3,1,71,72,3,1,2,1,True
5053,2025-07-17,PLU_4241474,SOUP OF THE DAY,2,61,63,3,3,1,1,True


In [50]:
# ============================================================
# Step 4 - Cell 10
# Create complete Step 4 classification summary
# ============================================================

step4_classification_summary = pd.DataFrame([
    {
        "Scope": "COMPLETE_SOURCE",
        "TransactionRows": len(
            step4_enriched_transactions
        ),
        "CanonicalProducts": (
            step4_enriched_transactions[
                "CanonicalProductID_Step3"
            ].nunique()
        ),
        "ConfirmedBulkLines": int(
            step4_enriched_transactions[
                "BulkOrderFlag_Step4"
            ].sum()
        ),
        "NormalDemandUnits": int(
            step4_enriched_transactions[
                "NormalDemandUnits_Step4"
            ].sum()
        ),
        "BulkDemandUnits": int(
            step4_enriched_transactions[
                "BulkDemandUnits_Step4"
            ].sum()
        ),
        "TotalDemandUnits": int(
            step4_enriched_transactions[
                "TotalDemandUnits_Step4"
            ].sum()
        ),
    },
    {
        "Scope": "FORECASTABLE_EXCLUDING_OPEN_UL",
        "TransactionRows": int(
            forecastable_step4_mask.sum()
        ),
        "CanonicalProducts": (
            step4_enriched_transactions.loc[
                forecastable_step4_mask,
                "CanonicalProductID_Step3",
            ].nunique()
        ),
        "ConfirmedBulkLines": int(
            step4_enriched_transactions.loc[
                forecastable_step4_mask,
                "BulkOrderFlag_Step4",
            ].sum()
        ),
        "NormalDemandUnits": (
            forecastable_normal_units
        ),
        "BulkDemandUnits": (
            forecastable_bulk_units
        ),
        "TotalDemandUnits": (
            forecastable_total_units
        ),
    },
    {
        "Scope": "OPEN_UL_EXCLUDED",
        "TransactionRows": int(
            open_ul_by_name.sum()
        ),
        "CanonicalProducts": 1,
        "ConfirmedBulkLines": 0,
        "NormalDemandUnits": int(
            step4_enriched_transactions.loc[
                open_ul_by_name,
                "NormalDemandUnits_Step4",
            ].sum()
        ),
        "BulkDemandUnits": 0,
        "TotalDemandUnits": int(
            step4_enriched_transactions.loc[
                open_ul_by_name,
                "TotalDemandUnits_Step4",
            ].sum()
        ),
    },
])


assert (
    step4_classification_summary[
        "NormalDemandUnits"
    ]
    +
    step4_classification_summary[
        "BulkDemandUnits"
    ]
    ==
    step4_classification_summary[
        "TotalDemandUnits"
    ]
).all()


display(step4_classification_summary)

,Scope,TransactionRows,CanonicalProducts,ConfirmedBulkLines,NormalDemandUnits,BulkDemandUnits,TotalDemandUnits
0,COMPLETE_SOURCE,138983,228,61,138922,2558,141480
1,FORECASTABLE_EXCLUDING_OPEN_UL,114247,227,61,114186,2558,116744
2,OPEN_UL_EXCLUDED,24736,1,0,24736,0,24736


In [51]:
# ============================================================
# Step 4 - Cell 11
# Save Step 4 transaction and audit outputs
# ============================================================

step4_enriched_transactions.to_csv(
    STEP4_ENRICHED_TRANSACTION_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


confirmed_bulk_line_registry.to_csv(
    STEP4_BULK_LINE_REGISTRY_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


bulk_product_summary_step4.to_csv(
    STEP4_BULK_PRODUCT_SUMMARY_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


observed_daily_demand_split_step4.to_csv(
    STEP4_DAILY_SPLIT_AUDIT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


step4_classification_summary.to_csv(
    STEP4_CLASSIFICATION_SUMMARY_FILE,
    index=False
)


print("Step 4 files saved successfully.")
print()

print("1. Enriched bulk-separated transactions:")
print(STEP4_ENRICHED_TRANSACTION_FILE)

print()
print("2. Confirmed bulk-line registry:")
print(STEP4_BULK_LINE_REGISTRY_FILE)

print()
print("3. Bulk product summary:")
print(STEP4_BULK_PRODUCT_SUMMARY_FILE)

print()
print("4. Observed daily split audit:")
print(STEP4_DAILY_SPLIT_AUDIT_FILE)

print()
print("5. Classification summary:")
print(STEP4_CLASSIFICATION_SUMMARY_FILE)

Step 4 files saved successfully.

1. Enriched bulk-separated transactions:
eden_datasets/UL_EDEN_transactions_step4_bulk_separated_enriched.csv

2. Confirmed bulk-line registry:
eden_datasets/UL_EDEN_step4_confirmed_bulk_line_registry.csv

3. Bulk product summary:
eden_datasets/UL_EDEN_step4_bulk_product_summary.csv

4. Observed daily split audit:
eden_datasets/UL_EDEN_step4_observed_daily_demand_split_audit.csv

5. Classification summary:
eden_datasets/UL_EDEN_step4_bulk_classification_summary.csv


In [52]:
# ============================================================
# Step 4 - Cell 12
# Reload and validate saved Step 4 transactions
# ============================================================

saved_step4_transactions = pd.read_csv(
    STEP4_ENRICHED_TRANSACTION_FILE,
    parse_dates=[
        "TransDate",
        "Date",
    ]
)


saved_step4_transactions[
    "PLUCode"
] = (
    pd.to_numeric(
        saved_step4_transactions[
            "PLUCode"
        ],
        errors="raise"
    )
    .astype("int64")
)


for numeric_column in [
    "UnitSold",
    "NormalDemandUnits_Step4",
    "BulkDemandUnits_Step4",
    "TotalDemandUnits_Step4",
    "BulkExtraUnitsAboveOriginalOne_Step4",
]:

    saved_step4_transactions[
        numeric_column
    ] = (
        pd.to_numeric(
            saved_step4_transactions[
                numeric_column
            ],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


saved_step4_transactions[
    "BulkOrderFlag_Step4"
] = convert_csv_boolean(
    saved_step4_transactions[
        "BulkOrderFlag_Step4"
    ]
)


saved_step4_transactions[
    "ForecastingEligible_Step4"
] = convert_csv_boolean(
    saved_step4_transactions[
        "ForecastingEligible_Step4"
    ]
)


saved_bulk_flag = (
    saved_step4_transactions[
        "BulkOrderFlag_Step4"
    ]
)


saved_forecastable_mask = (
    saved_step4_transactions[
        "ForecastingEligible_Step4"
    ]
)


# ------------------------------------------------------------
# Saved-file validation
# ------------------------------------------------------------

assert len(saved_step4_transactions) == 138_983


assert int(
    saved_bulk_flag.sum()
) == 61


assert int(
    saved_step4_transactions[
        "BulkDemandUnits_Step4"
    ].sum()
) == 2_558


assert int(
    saved_step4_transactions[
        "NormalDemandUnits_Step4"
    ].sum()
) == 138_922


assert int(
    saved_step4_transactions[
        "TotalDemandUnits_Step4"
    ].sum()
) == 141_480


assert int(
    saved_step4_transactions[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 2_497


assert (
    saved_step4_transactions[
        "NormalDemandUnits_Step4"
    ]
    +
    saved_step4_transactions[
        "BulkDemandUnits_Step4"
    ]
    ==
    saved_step4_transactions[
        "TotalDemandUnits_Step4"
    ]
).all()


assert (
    saved_step4_transactions[
        "TotalDemandUnits_Step4"
    ]
    ==
    saved_step4_transactions[
        "UnitSold"
    ]
).all()


assert saved_step4_transactions[
    "CanonicalProductID_Step3"
].nunique() == 228


assert saved_step4_transactions.loc[
    saved_forecastable_mask,
    "CanonicalProductID_Step3",
].nunique() == 227


assert int(
    saved_step4_transactions.loc[
        saved_forecastable_mask,
        "NormalDemandUnits_Step4",
    ].sum()
) == 114_186


assert int(
    saved_step4_transactions.loc[
        saved_forecastable_mask,
        "BulkDemandUnits_Step4",
    ].sum()
) == 2_558


assert int(
    saved_step4_transactions.loc[
        saved_forecastable_mask,
        "TotalDemandUnits_Step4",
    ].sum()
) == 116_744


saved_open_ul_mask = (
    saved_step4_transactions[
        "PLUName"
    ]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert not saved_step4_transactions.loc[
    saved_open_ul_mask,
    "BulkOrderFlag_Step4",
].any()


print("=" * 74)
print("STEP 4 BULK-DEMAND SEPARATION COMPLETED")
print("=" * 74)
print()
print(
    "Transaction rows:",
    f"{len(saved_step4_transactions):,}"
)
print(
    "Canonical product identities:",
    saved_step4_transactions[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Confirmed bulk lines:",
    f"{int(saved_bulk_flag.sum()):,}"
)
print(
    "Confirmed bulk units:",
    f"{int(saved_step4_transactions['BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Normal source units:",
    f"{int(saved_step4_transactions['NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "All source units preserved:",
    f"{int(saved_step4_transactions['TotalDemandUnits_Step4'].sum()):,}"
)
print()
print(
    "Forecastable normal units:",
    f"{int(saved_step4_transactions.loc[saved_forecastable_mask, 'NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "Forecastable bulk units:",
    f"{int(saved_step4_transactions.loc[saved_forecastable_mask, 'BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Forecastable total units:",
    f"{int(saved_step4_transactions.loc[saved_forecastable_mask, 'TotalDemandUnits_Step4'].sum()):,}"
)
print()
print("Final Step 4 transaction file:")
print(STEP4_ENRICHED_TRANSACTION_FILE)

STEP 4 BULK-DEMAND SEPARATION COMPLETED

Transaction rows: 138,983
Canonical product identities: 228
Confirmed bulk lines: 61
Confirmed bulk units: 2,558
Normal source units: 138,922
All source units preserved: 141,480

Forecastable normal units: 114,186
Forecastable bulk units: 2,558
Forecastable total units: 116,744

Final Step 4 transaction file:
eden_datasets/UL_EDEN_transactions_step4_bulk_separated_enriched.csv


/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_3618/3432460328.py:6: DtypeWarning: Columns (21,25,31,32,33,34,36) have mixed types. Specify dtype option on import or set low_memory=False.
  saved_step4_transactions = pd.read_csv(


### I just understood that there are some duplicates present in the dataset and that is why I am going to fix that first before finalising on the daily demand dataset

In [53]:
# ============================================================
# Duplicate Fix - Cell 1
# Load corrected transactions and full UnitSold audit
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)


FULL_AUDIT_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_unitsold_corrected_with_audit_columns.csv"
)


DEDUPLICATED_TRANSACTION_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected_deduplicated.csv"
)


DEDUPLICATED_FULL_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_unitsold_corrected_with_audit_columns_deduplicated.csv"
)


DUPLICATE_GROUP_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_confirmed_export_duplicate_group_audit.csv"
)


DUPLICATE_REMOVED_ROWS_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_confirmed_export_duplicate_rows_removed.csv"
)


DUPLICATE_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_confirmed_export_duplicate_summary.csv"
)


for required_file in [
    TRANSACTION_INPUT_FILE,
    FULL_AUDIT_INPUT_FILE,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{required_file}"
        )


duplicate_source = pd.read_csv(
    TRANSACTION_INPUT_FILE,
    low_memory=False
)


duplicate_full_audit = pd.read_csv(
    FULL_AUDIT_INPUT_FILE,
    low_memory=False
)


source_columns = duplicate_source.columns.tolist()


print("Files loaded successfully.")
print()
print(
    "Transaction rows:",
    f"{len(duplicate_source):,}"
)
print(
    "Full-audit rows:",
    f"{len(duplicate_full_audit):,}"
)

Files loaded successfully.

Transaction rows: 138,983
Full-audit rows: 138,983


In [54]:
# ============================================================
# Duplicate Fix - Cell 2
# Convert data types and prove both files are aligned
# ============================================================

for dataframe in [
    duplicate_source,
    duplicate_full_audit,
]:

    dataframe["TransDate"] = pd.to_datetime(
        dataframe["TransDate"],
        errors="raise"
    )

    dataframe["Date"] = (
        pd.to_datetime(
            dataframe["Date"],
            errors="raise"
        )
        .dt.normalize()
    )

    dataframe["PLUCode"] = (
        pd.to_numeric(
            dataframe["PLUCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["GroupCode"] = (
        pd.to_numeric(
            dataframe["GroupCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["TransValue"] = pd.to_numeric(
        dataframe["TransValue"],
        errors="raise"
    )

    dataframe["UnitSold"] = (
        pd.to_numeric(
            dataframe["UnitSold"],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


duplicate_full_audit["UnitSold_Original"] = (
    pd.to_numeric(
        duplicate_full_audit["UnitSold_Original"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


missing_source_columns = [
    column
    for column in source_columns
    if column not in duplicate_full_audit.columns
]


if missing_source_columns:
    raise ValueError(
        "The full audit is missing source columns:\n"
        f"{missing_source_columns}"
    )


def create_comparison_frame(
    dataframe,
    columns
):
    comparison = pd.DataFrame(
        index=dataframe.index
    )

    datetime_columns = {
        "TransDate",
        "Date",
    }

    numeric_columns = {
        "TransValue",
        "GroupCode",
        "PLUCode",
        "Hour",
        "Month",
        "WeekOfYear",
        "UnitSold",
    }

    for column in columns:

        if column in datetime_columns:
            comparison[column] = pd.to_datetime(
                dataframe[column],
                errors="raise"
            )

        elif column in numeric_columns:
            comparison[column] = pd.to_numeric(
                dataframe[column],
                errors="raise"
            )

        else:
            comparison[column] = (
                dataframe[column]
                .astype("string")
                .str.strip()
                .fillna("<MISSING>")
            )

    return comparison


source_comparison = create_comparison_frame(
    duplicate_source,
    source_columns
)


audit_comparison = create_comparison_frame(
    duplicate_full_audit,
    source_columns
)


pd.testing.assert_frame_equal(
    source_comparison,
    audit_comparison,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000000001,
)


assert len(duplicate_source) == 138_983

assert int(
    duplicate_source["UnitSold"].sum()
) == 141_480

assert int(
    duplicate_source["UnitSold"].gt(1).sum()
) == 61


original_transaction_value_cents = int(
    (
        duplicate_source["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert original_transaction_value_cents == 56_739_534


print("Source and full-audit files match row by row.")
print()
print(
    "Rows:",
    f"{len(duplicate_source):,}"
)
print(
    "Units:",
    f"{int(duplicate_source['UnitSold'].sum()):,}"
)
print(
    "Transaction value:",
    f"€{original_transaction_value_cents / 100:,.2f}"
)

Source and full-audit files match row by row.

Rows: 138,983
Units: 141,480
Transaction value: €567,395.34


In [55]:
# ============================================================
# Duplicate Fix - Cell 3
# Define only the confirmed export-duplicate groups
# ============================================================

confirmed_duplicate_groups = [
    {
        "DuplicateGroupID": "DUP_EXPORT_001",
        "TransactionID": "194852_2025-07-22_13-21-00",
        "PLUCode": 4241480,
        "ExpectedPLUName": "MAINS 3",
        "TransValueCents": 44_100,
        "UnitSold": 49,
        "ExpectedCopies": 2,
        "CopiesToKeep": 1,
    },
    {
        "DuplicateGroupID": "DUP_EXPORT_002",
        "TransactionID": "194852_2025-07-22_13-21-00",
        "PLUCode": 4241474,
        "ExpectedPLUName": "SOUP OF THE DAY",
        "TransValueCents": 17_150,
        "UnitSold": 49,
        "ExpectedCopies": 2,
        "CopiesToKeep": 1,
    },
    {
        "DuplicateGroupID": "DUP_EXPORT_003",
        "TransactionID": "207641_2025-09-24_13-21-00",
        "PLUCode": 4241428,
        "ExpectedPLUName": "HAM & CHEESE SANDWICH CT",
        "TransValueCents": 30_500,
        "UnitSold": 61,
        "ExpectedCopies": 5,
        "CopiesToKeep": 1,
    },
    {
        "DuplicateGroupID": "DUP_EXPORT_004",
        "TransactionID": "357119_2025-09-26_13-20-00",
        "PLUCode": 4241428,
        "ExpectedPLUName": "HAM & CHEESE SANDWICH CT",
        "TransValueCents": 30_500,
        "UnitSold": 61,
        "ExpectedCopies": 5,
        "CopiesToKeep": 1,
    },
]


confirmed_duplicate_groups_df = pd.DataFrame(
    confirmed_duplicate_groups
)


assert confirmed_duplicate_groups_df[
    "DuplicateGroupID"
].is_unique


duplicate_working = duplicate_source.copy()


duplicate_working[
    "_SourceRowNumber"
] = np.arange(
    1,
    len(duplicate_working) + 1
)


duplicate_working[
    "_TransValueCents"
] = (
    duplicate_working["TransValue"]
    * 100
).round().astype("int64")


for group in confirmed_duplicate_groups:

    group_mask = (
        duplicate_working["TransactionID"]
        .astype("string")
        .str.strip()
        .eq(group["TransactionID"])

        & duplicate_working["PLUCode"]
        .eq(group["PLUCode"])

        & duplicate_working["_TransValueCents"]
        .eq(group["TransValueCents"])

        & duplicate_working["UnitSold"]
        .eq(group["UnitSold"])
    )


    matching_rows = duplicate_working.loc[
        group_mask
    ]


    assert len(matching_rows) == (
        group["ExpectedCopies"]
    ), (
        f"Unexpected copy count for "
        f"{group['DuplicateGroupID']}.\n"
        f"Expected: {group['ExpectedCopies']}\n"
        f"Found: {len(matching_rows)}"
    )


    assert matching_rows[
        "PLUName"
    ].astype("string").str.strip().eq(
        group["ExpectedPLUName"]
    ).all()


    # Every original source field must be identical
    # inside the confirmed duplicate group.
    for column in source_columns:

        assert matching_rows[
            column
        ].nunique(
            dropna=False
        ) == 1, (
            f"{group['DuplicateGroupID']} is not exactly "
            f"duplicated in column {column}."
        )


print("All four confirmed duplicate groups validated.")
print()
print(
    "Original copies in confirmed groups:",
    sum(
        group["ExpectedCopies"]
        for group in confirmed_duplicate_groups
    )
)
print(
    "Rows that must be removed:",
    sum(
        group["ExpectedCopies"]
        - group["CopiesToKeep"]
        for group in confirmed_duplicate_groups
    )
)

All four confirmed duplicate groups validated.

Original copies in confirmed groups: 14
Rows that must be removed: 10


In [56]:
# ============================================================
# Duplicate Fix - Cell 4
# Mark confirmed duplicates while keeping one representative
# ============================================================

duplicate_working[
    "ConfirmedDuplicateGroupID"
] = pd.Series(
    pd.NA,
    index=duplicate_working.index,
    dtype="string"
)


duplicate_working[
    "DuplicateOccurrenceNumber"
] = pd.Series(
    pd.NA,
    index=duplicate_working.index,
    dtype="Int64"
)


duplicate_working[
    "DuplicateCorrectionAction"
] = pd.Series(
    pd.NA,
    index=duplicate_working.index,
    dtype="string"
)


for group in confirmed_duplicate_groups:

    group_mask = (
        duplicate_working["TransactionID"]
        .astype("string")
        .str.strip()
        .eq(group["TransactionID"])

        & duplicate_working["PLUCode"]
        .eq(group["PLUCode"])

        & duplicate_working["_TransValueCents"]
        .eq(group["TransValueCents"])

        & duplicate_working["UnitSold"]
        .eq(group["UnitSold"])
    )


    group_indices = (
        duplicate_working.loc[
            group_mask
        ]
        .sort_values(
            "_SourceRowNumber"
        )
        .index
        .tolist()
    )


    assert len(group_indices) == (
        group["ExpectedCopies"]
    )


    occurrence_numbers = list(
        range(
            1,
            len(group_indices) + 1
        )
    )


    duplicate_working.loc[
        group_indices,
        "ConfirmedDuplicateGroupID"
    ] = group["DuplicateGroupID"]


    duplicate_working.loc[
        group_indices,
        "DuplicateOccurrenceNumber"
    ] = occurrence_numbers


    keep_count = group["CopiesToKeep"]


    keep_indices = group_indices[
        :keep_count
    ]


    remove_indices = group_indices[
        keep_count:
    ]


    duplicate_working.loc[
        keep_indices,
        "DuplicateCorrectionAction"
    ] = "KEEP_ONE_CONFIRMED_ROW"


    duplicate_working.loc[
        remove_indices,
        "DuplicateCorrectionAction"
    ] = "REMOVE_CONFIRMED_EXPORT_DUPLICATE"


duplicate_remove_mask = (
    duplicate_working[
        "DuplicateCorrectionAction"
    ].eq(
        "REMOVE_CONFIRMED_EXPORT_DUPLICATE"
    )
)


duplicate_keep_representative_mask = (
    duplicate_working[
        "DuplicateCorrectionAction"
    ].eq(
        "KEEP_ONE_CONFIRMED_ROW"
    )
)


assert int(
    duplicate_remove_mask.sum()
) == 10


assert int(
    duplicate_keep_representative_mask.sum()
) == 4


removed_unit_total = int(
    duplicate_working.loc[
        duplicate_remove_mask,
        "UnitSold",
    ].sum()
)


removed_value_cents = int(
    duplicate_working.loc[
        duplicate_remove_mask,
        "_TransValueCents",
    ].sum()
)


assert removed_unit_total == 586

assert removed_value_cents == 305_250


print("Confirmed duplicate rows marked safely.")
print()
print(
    "Representative rows retained:",
    int(
        duplicate_keep_representative_mask.sum()
    )
)
print(
    "Duplicate rows marked for removal:",
    int(
        duplicate_remove_mask.sum()
    )
)
print(
    "Units to remove:",
    f"{removed_unit_total:,}"
)
print(
    "Transaction value to remove:",
    f"€{removed_value_cents / 100:,.2f}"
)

Confirmed duplicate rows marked safely.

Representative rows retained: 4
Duplicate rows marked for removal: 10
Units to remove: 586
Transaction value to remove: €3,052.50


In [60]:
# ============================================================
# Duplicate Fix - Cell 5
# Remove only the 10 confirmed duplicate rows
#
# Corrected:
# Converts the nullable pandas mask into a strict NumPy
# Boolean array before applying it to both aligned datasets.
# ============================================================

# ------------------------------------------------------------
# Create a strict Boolean removal mask
# ------------------------------------------------------------

duplicate_remove_mask_strict = (
    duplicate_working[
        "DuplicateCorrectionAction"
    ]
    .eq(
        "REMOVE_CONFIRMED_EXPORT_DUPLICATE"
    )
    .fillna(False)
    .astype(bool)
)


# Convert to a positional NumPy Boolean array.
# Both source and audit files were already proven to be
# row-aligned, so the same positional mask can be used safely.
duplicate_remove_mask_array = (
    duplicate_remove_mask_strict
    .to_numpy(dtype=bool)
)


assert len(
    duplicate_remove_mask_array
) == len(duplicate_source)


assert len(
    duplicate_remove_mask_array
) == len(duplicate_full_audit)


assert int(
    duplicate_remove_mask_array.sum()
) == 10, (
    "Exactly 10 confirmed duplicate rows should be removed."
)


# ------------------------------------------------------------
# Remove the same 10 row positions from both aligned files
# ------------------------------------------------------------

deduplicated_transactions = (
    duplicate_working.iloc[
        ~duplicate_remove_mask_array
    ][
        source_columns
    ]
    .reset_index(drop=True)
)


deduplicated_full_audit = (
    duplicate_full_audit.iloc[
        ~duplicate_remove_mask_array
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate final source row counts
# ------------------------------------------------------------

assert len(
    deduplicated_transactions
) == 138_973, (
    "Expected 138,973 transaction rows after removing "
    "10 confirmed duplicate rows."
)


assert len(
    deduplicated_full_audit
) == 138_973, (
    "Expected 138,973 rows in the deduplicated full audit."
)


# ------------------------------------------------------------
# Validate final corrected UnitSold totals
# ------------------------------------------------------------

assert int(
    deduplicated_transactions[
        "UnitSold"
    ].sum()
) == 140_894, (
    "Expected final UnitSold total of 140,894."
)


assert int(
    deduplicated_transactions[
        "UnitSold"
    ].gt(1).sum()
) == 51, (
    "Expected 51 remaining multi-unit transaction rows."
)


# ------------------------------------------------------------
# Validate transaction value
# ------------------------------------------------------------

deduplicated_value_cents = int(
    (
        deduplicated_transactions[
            "TransValue"
        ]
        * 100
    )
    .round()
    .sum()
)


assert deduplicated_value_cents == 56_434_284, (
    "Expected final transaction value of €564,342.84."
)


assert (
    original_transaction_value_cents
    - deduplicated_value_cents
) == 305_250, (
    "Expected removed transaction value of €3,052.50."
)


# ------------------------------------------------------------
# Validate the deduplicated full audit
# ------------------------------------------------------------

assert int(
    deduplicated_full_audit[
        "UnitSold_Original"
    ].sum()
) == 138_973, (
    "The deduplicated original UnitSold total is incorrect."
)


assert int(
    deduplicated_full_audit[
        "UnitSold"
    ].sum()
) == 140_894, (
    "The deduplicated corrected UnitSold total is incorrect."
)


deduplicated_extra_units = int(
    (
        deduplicated_full_audit[
            "UnitSold"
        ]
        -
        deduplicated_full_audit[
            "UnitSold_Original"
        ]
    ).sum()
)


assert deduplicated_extra_units == 1_921, (
    "Expected 1,921 remaining additional inferred units."
)


# ------------------------------------------------------------
# Confirm one genuine copy remains in every approved group
# ------------------------------------------------------------

deduplicated_transactions[
    "_TransValueCents"
] = (
    deduplicated_transactions[
        "TransValue"
    ]
    * 100
).round().astype("int64")


for group in confirmed_duplicate_groups:

    remaining_mask = (
        deduplicated_transactions[
            "TransactionID"
        ]
        .astype("string")
        .str.strip()
        .eq(group["TransactionID"])

        & deduplicated_transactions[
            "PLUCode"
        ].eq(group["PLUCode"])

        & deduplicated_transactions[
            "_TransValueCents"
        ].eq(group["TransValueCents"])

        & deduplicated_transactions[
            "UnitSold"
        ].eq(group["UnitSold"])
    )


    remaining_count = int(
        remaining_mask
        .fillna(False)
        .sum()
    )


    assert remaining_count == 1, (
        f"Expected exactly one retained row for "
        f"{group['DuplicateGroupID']}, "
        f"but found {remaining_count}."
    )


# Remove temporary validation column
deduplicated_transactions.drop(
    columns="_TransValueCents",
    inplace=True
)


# ------------------------------------------------------------
# Final Cell 5 output
# ------------------------------------------------------------

print("Confirmed duplicate rows removed successfully.")
print()
print(
    "Rows removed:",
    f"{int(duplicate_remove_mask_array.sum()):,}"
)
print(
    "Final rows:",
    f"{len(deduplicated_transactions):,}"
)
print(
    "Final UnitSold total:",
    f"{int(deduplicated_transactions['UnitSold'].sum()):,}"
)
print(
    "Remaining multi-unit rows:",
    f"{int(deduplicated_transactions['UnitSold'].gt(1).sum()):,}"
)
print(
    "Final transaction value:",
    f"€{deduplicated_value_cents / 100:,.2f}"
)
print(
    "Remaining additional inferred units:",
    f"{deduplicated_extra_units:,}"
)

Confirmed duplicate rows removed successfully.

Rows removed: 10
Final rows: 138,973
Final UnitSold total: 140,894
Remaining multi-unit rows: 51
Final transaction value: €564,342.84
Remaining additional inferred units: 1,921


In [61]:
# ============================================================
# Duplicate Fix - Cell 6
# Create group-level and removed-row audit files
# ============================================================

duplicate_group_audit_columns = [
    "_SourceRowNumber",
    "ConfirmedDuplicateGroupID",
    "DuplicateOccurrenceNumber",
    "DuplicateCorrectionAction",
] + source_columns


confirmed_duplicate_group_audit = (
    duplicate_working.loc[
        duplicate_working[
            "ConfirmedDuplicateGroupID"
        ].notna(),
        duplicate_group_audit_columns,
    ]
    .sort_values(
        [
            "ConfirmedDuplicateGroupID",
            "DuplicateOccurrenceNumber",
        ]
    )
    .reset_index(drop=True)
)


confirmed_duplicate_rows_removed = (
    confirmed_duplicate_group_audit.loc[
        confirmed_duplicate_group_audit[
            "DuplicateCorrectionAction"
        ].eq(
            "REMOVE_CONFIRMED_EXPORT_DUPLICATE"
        )
    ]
    .reset_index(drop=True)
)


duplicate_summary_rows = []


for group in confirmed_duplicate_groups:

    group_audit = (
        confirmed_duplicate_group_audit.loc[
            confirmed_duplicate_group_audit[
                "ConfirmedDuplicateGroupID"
            ].eq(group["DuplicateGroupID"])
        ]
    )


    removed_group_rows = group_audit.loc[
        group_audit[
            "DuplicateCorrectionAction"
        ].eq(
            "REMOVE_CONFIRMED_EXPORT_DUPLICATE"
        )
    ]


    duplicate_summary_rows.append({
        "DuplicateGroupID":
            group["DuplicateGroupID"],

        "TransactionID":
            group["TransactionID"],

        "PLUCode":
            group["PLUCode"],

        "PLUName":
            group["ExpectedPLUName"],

        "TransValuePerRow":
            group["TransValueCents"] / 100,

        "UnitSoldPerRow":
            group["UnitSold"],

        "OriginalCopies":
            len(group_audit),

        "CopiesRetained":
            int(
                group_audit[
                    "DuplicateCorrectionAction"
                ].eq(
                    "KEEP_ONE_CONFIRMED_ROW"
                ).sum()
            ),

        "CopiesRemoved":
            len(removed_group_rows),

        "UnitsRemoved":
            int(
                removed_group_rows[
                    "UnitSold"
                ].sum()
            ),

        "TransactionValueRemoved":
            float(
                removed_group_rows[
                    "TransValue"
                ].sum()
            ),
    })


confirmed_duplicate_summary = pd.DataFrame(
    duplicate_summary_rows
)


assert len(
    confirmed_duplicate_group_audit
) == 14


assert len(
    confirmed_duplicate_rows_removed
) == 10


assert int(
    confirmed_duplicate_summary[
        "CopiesRemoved"
    ].sum()
) == 10


assert int(
    confirmed_duplicate_summary[
        "UnitsRemoved"
    ].sum()
) == 586


assert np.isclose(
    confirmed_duplicate_summary[
        "TransactionValueRemoved"
    ].sum(),
    3_052.50,
)


print("Duplicate-correction audits created.")
print()

display(confirmed_duplicate_summary)

print()
print("Removed rows:")

display(confirmed_duplicate_rows_removed)

Duplicate-correction audits created.



,DuplicateGroupID,TransactionID,PLUCode,PLUName,TransValuePerRow,UnitSoldPerRow,OriginalCopies,CopiesRetained,CopiesRemoved,UnitsRemoved,TransactionValueRemoved
0,DUP_EXPORT_001,194852_2025-07-22_13-21-00,4241480,MAINS 3,441.0,49,2,1,1,49,441.0
1,DUP_EXPORT_002,194852_2025-07-22_13-21-00,4241474,SOUP OF THE DAY,171.5,49,2,1,1,49,171.5
2,DUP_EXPORT_003,207641_2025-09-24_13-21-00,4241428,HAM & CHEESE SANDWICH CT,305.0,61,5,1,4,244,1220.0
3,DUP_EXPORT_004,357119_2025-09-26_13-20-00,4241428,HAM & CHEESE SANDWICH CT,305.0,61,5,1,4,244,1220.0



Removed rows:


,_SourceRowNumber,ConfirmedDuplicateGroupID,DuplicateOccurrenceNumber,DuplicateCorrectionAction,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,TransactionID,UnitSold
0,9,DUP_EXPORT_001,2,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-07-22 13:21:00,441.0,MAINS 3,7,DINNER,4241480,2025-07-22,13,Tuesday,7,30,194852_2025-07-22_13-21-00,49
1,43,DUP_EXPORT_002,2,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-07-22 13:21:00,171.5,SOUP OF THE DAY,8,PIZZA,4241474,2025-07-22,13,Tuesday,7,30,194852_2025-07-22_13-21-00,49
2,24,DUP_EXPORT_003,2,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-24 13:21:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-24,13,Wednesday,9,39,207641_2025-09-24_13-21-00,61
3,25,DUP_EXPORT_003,3,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-24 13:21:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-24,13,Wednesday,9,39,207641_2025-09-24_13-21-00,61
4,26,DUP_EXPORT_003,4,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-24 13:21:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-24,13,Wednesday,9,39,207641_2025-09-24_13-21-00,61
5,27,DUP_EXPORT_003,5,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-24 13:21:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-24,13,Wednesday,9,39,207641_2025-09-24_13-21-00,61
6,29,DUP_EXPORT_004,2,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-26 13:20:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-26,13,Friday,9,39,357119_2025-09-26_13-20-00,61
7,30,DUP_EXPORT_004,3,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-26 13:20:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-26,13,Friday,9,39,357119_2025-09-26_13-20-00,61
8,31,DUP_EXPORT_004,4,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-26 13:20:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-26,13,Friday,9,39,357119_2025-09-26_13-20-00,61
9,32,DUP_EXPORT_004,5,REMOVE_CONFIRMED_EXPORT_DUPLICATE,2025-09-26 13:20:00,305.0,HAM & CHEESE SANDWICH CT,5,SANDWICHES,4241428,2025-09-26,13,Friday,9,39,357119_2025-09-26_13-20-00,61


In [62]:
# ============================================================
# Duplicate Fix - Cell 7
# Save deduplicated transaction and audit files
# ============================================================

deduplicated_transactions.to_csv(
    DEDUPLICATED_TRANSACTION_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


deduplicated_full_audit.to_csv(
    DEDUPLICATED_FULL_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


confirmed_duplicate_group_audit.to_csv(
    DUPLICATE_GROUP_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


confirmed_duplicate_rows_removed.to_csv(
    DUPLICATE_REMOVED_ROWS_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


confirmed_duplicate_summary.to_csv(
    DUPLICATE_SUMMARY_OUTPUT_FILE,
    index=False
)


print("Duplicate-correction files saved successfully.")
print()

print("1. Deduplicated corrected transactions:")
print(DEDUPLICATED_TRANSACTION_OUTPUT_FILE)

print()
print("2. Deduplicated full UnitSold audit:")
print(DEDUPLICATED_FULL_AUDIT_OUTPUT_FILE)

print()
print("3. Complete confirmed-group audit:")
print(DUPLICATE_GROUP_AUDIT_OUTPUT_FILE)

print()
print("4. Removed-row audit:")
print(DUPLICATE_REMOVED_ROWS_OUTPUT_FILE)

print()
print("5. Duplicate correction summary:")
print(DUPLICATE_SUMMARY_OUTPUT_FILE)

Duplicate-correction files saved successfully.

1. Deduplicated corrected transactions:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected_deduplicated.csv

2. Deduplicated full UnitSold audit:
eden_datasets/UL_EDEN_transactions_unitsold_corrected_with_audit_columns_deduplicated.csv

3. Complete confirmed-group audit:
eden_datasets/UL_EDEN_confirmed_export_duplicate_group_audit.csv

4. Removed-row audit:
eden_datasets/UL_EDEN_confirmed_export_duplicate_rows_removed.csv

5. Duplicate correction summary:
eden_datasets/UL_EDEN_confirmed_export_duplicate_summary.csv


In [63]:
# ============================================================
# Duplicate Fix - Cell 8
# Reload and validate saved deduplicated source
# ============================================================

saved_deduplicated_transactions = pd.read_csv(
    DEDUPLICATED_TRANSACTION_OUTPUT_FILE,
    low_memory=False,
    parse_dates=[
        "TransDate",
        "Date",
    ]
)


saved_deduplicated_full_audit = pd.read_csv(
    DEDUPLICATED_FULL_AUDIT_OUTPUT_FILE,
    low_memory=False,
    parse_dates=[
        "TransDate",
        "Date",
    ]
)


for dataframe in [
    saved_deduplicated_transactions,
    saved_deduplicated_full_audit,
]:

    dataframe["PLUCode"] = (
        pd.to_numeric(
            dataframe["PLUCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["TransValue"] = pd.to_numeric(
        dataframe["TransValue"],
        errors="raise"
    )

    dataframe["UnitSold"] = (
        pd.to_numeric(
            dataframe["UnitSold"],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


saved_deduplicated_full_audit[
    "UnitSold_Original"
] = (
    pd.to_numeric(
        saved_deduplicated_full_audit[
            "UnitSold_Original"
        ],
        errors="raise"
    )
    .round()
    .astype("int64")
)


assert len(
    saved_deduplicated_transactions
) == 138_973


assert len(
    saved_deduplicated_full_audit
) == 138_973


assert int(
    saved_deduplicated_transactions[
        "UnitSold"
    ].sum()
) == 140_894


assert int(
    saved_deduplicated_transactions[
        "UnitSold"
    ].gt(1).sum()
) == 51


saved_value_cents = int(
    (
        saved_deduplicated_transactions[
            "TransValue"
        ]
        * 100
    )
    .round()
    .sum()
)


assert saved_value_cents == 56_434_284


assert int(
    saved_deduplicated_full_audit[
        "UnitSold_Original"
    ].sum()
) == 138_973


assert int(
    (
        saved_deduplicated_full_audit[
            "UnitSold"
        ]
        -
        saved_deduplicated_full_audit[
            "UnitSold_Original"
        ]
    ).sum()
) == 1_921


# Recheck the four confirmed groups
saved_deduplicated_transactions[
    "_TransValueCents"
] = (
    saved_deduplicated_transactions[
        "TransValue"
    ]
    * 100
).round().astype("int64")


for group in confirmed_duplicate_groups:

    saved_group_mask = (
        saved_deduplicated_transactions[
            "TransactionID"
        ]
        .astype("string")
        .str.strip()
        .eq(group["TransactionID"])

        & saved_deduplicated_transactions[
            "PLUCode"
        ].eq(group["PLUCode"])

        & saved_deduplicated_transactions[
            "_TransValueCents"
        ].eq(group["TransValueCents"])

        & saved_deduplicated_transactions[
            "UnitSold"
        ].eq(group["UnitSold"])
    )


    assert int(
        saved_group_mask.sum()
    ) == 1


saved_deduplicated_transactions.drop(
    columns="_TransValueCents",
    inplace=True
)


print("=" * 74)
print("CONFIRMED EXPORT-DUPLICATE CORRECTION COMPLETED")
print("=" * 74)
print()
print(
    "Rows before:",
    f"{138_983:,}"
)
print(
    "Rows removed:",
    f"{10:,}"
)
print(
    "Rows after:",
    f"{len(saved_deduplicated_transactions):,}"
)
print()
print(
    "Units before:",
    f"{141_480:,}"
)
print(
    "Units removed:",
    f"{586:,}"
)
print(
    "Units after:",
    f"{int(saved_deduplicated_transactions['UnitSold'].sum()):,}"
)
print()
print(
    "Multi-unit rows after:",
    f"{int(saved_deduplicated_transactions['UnitSold'].gt(1).sum()):,}"
)
print(
    "Remaining additional inferred units:",
    f"{1_921:,}"
)
print(
    "Final transaction value:",
    f"€{saved_value_cents / 100:,.2f}"
)
print()
print("Final deduplicated source:")
print(DEDUPLICATED_TRANSACTION_OUTPUT_FILE)

CONFIRMED EXPORT-DUPLICATE CORRECTION COMPLETED

Rows before: 138,983
Rows removed: 10
Rows after: 138,973

Units before: 141,480
Units removed: 586
Units after: 140,894

Multi-unit rows after: 51
Remaining additional inferred units: 1,921
Final transaction value: €564,342.84

Final deduplicated source:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected_deduplicated.csv


### Duplicates have been dealt with now we are going to finalise the dataset

In [64]:
# ============================================================
# Mapping Refresh - Cell 1
# Load deduplicated transactions and approved Step 3 mapping
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


DEDUPLICATED_TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected_deduplicated.csv"
)


STEP3_MAPPING_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata.csv"
)


REFRESHED_STEP3_MAPPING_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata_deduplicated.csv"
)


MAPPING_REFRESH_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_mapping_statistics_refresh_after_dedup_audit.csv"
)


for required_file in [
    DEDUPLICATED_TRANSACTION_INPUT_FILE,
    STEP3_MAPPING_INPUT_FILE,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{required_file}"
        )


deduplicated_source = pd.read_csv(
    DEDUPLICATED_TRANSACTION_INPUT_FILE,
    low_memory=False
)


existing_step3_mapping = pd.read_csv(
    STEP3_MAPPING_INPUT_FILE,
    low_memory=False,
    parse_dates=[
        "FirstObservedDate",
        "LastObservedDate",
        "PriceChangeDate_Step3",
    ]
)


print("Files loaded successfully.")
print()
print(
    "Deduplicated source rows:",
    f"{len(deduplicated_source):,}"
)
print(
    "Step 3 mapping rows:",
    f"{len(existing_step3_mapping):,}"
)

Files loaded successfully.

Deduplicated source rows: 138,973
Step 3 mapping rows: 236


In [65]:
# ============================================================
# Mapping Refresh - Cell 2
# Convert types and validate source state
# ============================================================

deduplicated_source["TransDate"] = pd.to_datetime(
    deduplicated_source["TransDate"],
    errors="raise"
)


deduplicated_source["Date"] = (
    pd.to_datetime(
        deduplicated_source["Date"],
        errors="raise"
    )
    .dt.normalize()
)


deduplicated_source["PLUCode"] = (
    pd.to_numeric(
        deduplicated_source["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


deduplicated_source["GroupCode"] = (
    pd.to_numeric(
        deduplicated_source["GroupCode"],
        errors="raise"
    )
    .astype("int64")
)


deduplicated_source["TransValue"] = pd.to_numeric(
    deduplicated_source["TransValue"],
    errors="raise"
)


deduplicated_source["UnitSold"] = (
    pd.to_numeric(
        deduplicated_source["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


existing_step3_mapping["PLUCode"] = (
    pd.to_numeric(
        existing_step3_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


existing_step3_mapping["TransactionRows"] = (
    pd.to_numeric(
        existing_step3_mapping["TransactionRows"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


existing_step3_mapping["TotalUnits"] = (
    pd.to_numeric(
        existing_step3_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


existing_step3_mapping[
    "TotalTransactionValue"
] = pd.to_numeric(
    existing_step3_mapping[
        "TotalTransactionValue"
    ],
    errors="raise"
)


# ------------------------------------------------------------
# Validate deduplicated source
# ------------------------------------------------------------

assert len(deduplicated_source) == 138_973


assert int(
    deduplicated_source["UnitSold"].sum()
) == 140_894


assert deduplicated_source[
    "PLUCode"
].nunique() == 236


deduplicated_value_cents = int(
    (
        deduplicated_source["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert deduplicated_value_cents == 56_434_284


# ------------------------------------------------------------
# Validate existing mapping decisions
# ------------------------------------------------------------

assert len(existing_step3_mapping) == 236


assert existing_step3_mapping[
    "PLUCode"
].is_unique


assert existing_step3_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    existing_step3_mapping[
        "TotalUnits"
    ].sum()
) == 141_480


print("Input validation passed.")
print()
print(
    "Deduplicated products:",
    deduplicated_source["PLUCode"].nunique()
)
print(
    "Deduplicated units:",
    f"{int(deduplicated_source['UnitSold'].sum()):,}"
)
print(
    "Deduplicated value:",
    f"€{deduplicated_value_cents / 100:,.2f}"
)

Input validation passed.

Deduplicated products: 236
Deduplicated units: 140,894
Deduplicated value: €564,342.84


In [66]:
# ============================================================
# Mapping Refresh - Cell 3
# Recalculate statistics from the deduplicated source
# ============================================================

source_plu_consistency = (
    deduplicated_source
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        ProductNameCount=(
            "PLUName",
            "nunique"
        ),
        GroupCodeCount=(
            "GroupCode",
            "nunique"
        ),
        GroupNameCount=(
            "GroupName",
            "nunique"
        ),
    )
)


assert source_plu_consistency[
    "ProductNameCount"
].eq(1).all()


assert source_plu_consistency[
    "GroupCodeCount"
].eq(1).all()


assert source_plu_consistency[
    "GroupNameCount"
].eq(1).all()


refreshed_plu_statistics = (
    deduplicated_source
    .groupby(
        "PLUCode",
        as_index=False
    )
    .agg(
        PLUName_Source=(
            "PLUName",
            "first"
        ),
        GroupCode_Source=(
            "GroupCode",
            "first"
        ),
        GroupName_Source=(
            "GroupName",
            "first"
        ),
        FirstObservedDate_New=(
            "Date",
            "min"
        ),
        LastObservedDate_New=(
            "Date",
            "max"
        ),
        TransactionRows_New=(
            "PLUCode",
            "size"
        ),
        TotalUnits_New=(
            "UnitSold",
            "sum"
        ),
        TotalTransactionValue_New=(
            "TransValue",
            "sum"
        ),
    )
)


refreshed_plu_statistics[
    "TotalTransactionValue_New"
] = (
    refreshed_plu_statistics[
        "TotalTransactionValue_New"
    ]
    .round(2)
)


assert len(refreshed_plu_statistics) == 236


assert int(
    refreshed_plu_statistics[
        "TransactionRows_New"
    ].sum()
) == 138_973


assert int(
    refreshed_plu_statistics[
        "TotalUnits_New"
    ].sum()
) == 140_894


assert np.isclose(
    refreshed_plu_statistics[
        "TotalTransactionValue_New"
    ].sum(),
    564_342.84,
    atol=0.001,
)


print("Deduplicated PLU statistics calculated.")

Deduplicated PLU statistics calculated.


In [67]:
# ============================================================
# Mapping Refresh - Cell 4
# Compare old and refreshed statistics
# ============================================================

mapping_statistics_comparison = (
    existing_step3_mapping[
        [
            "PLUCode",
            "PLUName_Original",
            "PLUName_Corrected",
            "TransactionRows",
            "TotalUnits",
            "TotalTransactionValue",
        ]
    ]
    .merge(
        refreshed_plu_statistics,
        on="PLUCode",
        how="outer",
        validate="one_to_one"
    )
)


assert len(mapping_statistics_comparison) == 236


# Confirm the original name still agrees with the source
name_matches = (
    mapping_statistics_comparison[
        "PLUName_Original"
    ]
    .astype("string")
    .str.strip()
    .eq(
        mapping_statistics_comparison[
            "PLUName_Source"
        ]
        .astype("string")
        .str.strip()
    )
)


assert name_matches.all(), (
    "At least one original PLU name does not match "
    "the deduplicated transaction source."
)


mapping_statistics_comparison[
    "TransactionRowsChange"
] = (
    mapping_statistics_comparison[
        "TransactionRows_New"
    ]
    -
    mapping_statistics_comparison[
        "TransactionRows"
    ]
)


mapping_statistics_comparison[
    "TotalUnitsChange"
] = (
    mapping_statistics_comparison[
        "TotalUnits_New"
    ]
    -
    mapping_statistics_comparison[
        "TotalUnits"
    ]
)


mapping_statistics_comparison[
    "TransactionValueChange"
] = (
    mapping_statistics_comparison[
        "TotalTransactionValue_New"
    ]
    -
    mapping_statistics_comparison[
        "TotalTransactionValue"
    ]
).round(2)


mapping_refresh_audit = (
    mapping_statistics_comparison.loc[
        (
            mapping_statistics_comparison[
                "TransactionRowsChange"
            ].ne(0)
            |
            mapping_statistics_comparison[
                "TotalUnitsChange"
            ].ne(0)
            |
            ~np.isclose(
                mapping_statistics_comparison[
                    "TransactionValueChange"
                ],
                0,
                atol=0.001
            )
        )
    ]
    .copy()
    .sort_values("PLUCode")
    .reset_index(drop=True)
)


# Only three PLUs should be affected
assert len(mapping_refresh_audit) == 3


expected_changes = {
    4241428: {
        "Rows": -8,
        "Units": -488,
        "Value": -2440.00,
    },
    4241474: {
        "Rows": -1,
        "Units": -49,
        "Value": -171.50,
    },
    4241480: {
        "Rows": -1,
        "Units": -49,
        "Value": -441.00,
    },
}


for plu_code, expected in expected_changes.items():

    affected_row = mapping_refresh_audit.loc[
        mapping_refresh_audit[
            "PLUCode"
        ].eq(plu_code)
    ]


    assert len(affected_row) == 1


    assert int(
        affected_row[
            "TransactionRowsChange"
        ].iloc[0]
    ) == expected["Rows"]


    assert int(
        affected_row[
            "TotalUnitsChange"
        ].iloc[0]
    ) == expected["Units"]


    assert np.isclose(
        affected_row[
            "TransactionValueChange"
        ].iloc[0],
        expected["Value"],
        atol=0.001,
    )


assert int(
    mapping_refresh_audit[
        "TransactionRowsChange"
    ].sum()
) == -10


assert int(
    mapping_refresh_audit[
        "TotalUnitsChange"
    ].sum()
) == -586


assert np.isclose(
    mapping_refresh_audit[
        "TransactionValueChange"
    ].sum(),
    -3052.50,
    atol=0.001,
)


print("Mapping-statistics change audit validated.")
print()

display(
    mapping_refresh_audit[
        [
            "PLUCode",
            "PLUName_Corrected",
            "TransactionRows",
            "TransactionRows_New",
            "TransactionRowsChange",
            "TotalUnits",
            "TotalUnits_New",
            "TotalUnitsChange",
            "TotalTransactionValue",
            "TotalTransactionValue_New",
            "TransactionValueChange",
        ]
    ]
)

Mapping-statistics change audit validated.



,PLUCode,PLUName_Corrected,TransactionRows,TransactionRows_New,TransactionRowsChange,TotalUnits,TotalUnits_New,TotalUnitsChange,TotalTransactionValue,TotalTransactionValue_New,TransactionValueChange
0,4241428,HAM & CHEESE SANDWICH CT,1766,1758,-8,2366,1878,-488,11915.6,9475.6,-2440.0
1,4241474,SOUP OF THE DAY,329,328,-1,1033,984,-49,3615.5,3444.0,-171.5
2,4241480,MAINS 3,37,36,-1,474,425,-49,4266.0,3825.0,-441.0


In [68]:
# ============================================================
# Mapping Refresh - Cell 5
# Refresh statistical fields while preserving all decisions
# ============================================================

refreshed_step3_mapping = (
    existing_step3_mapping.copy()
)


statistics_lookup = (
    refreshed_plu_statistics
    .set_index("PLUCode")
)


refreshed_step3_mapping[
    "FirstObservedDate"
] = refreshed_step3_mapping[
    "PLUCode"
].map(
    statistics_lookup[
        "FirstObservedDate_New"
    ]
)


refreshed_step3_mapping[
    "LastObservedDate"
] = refreshed_step3_mapping[
    "PLUCode"
].map(
    statistics_lookup[
        "LastObservedDate_New"
    ]
)


refreshed_step3_mapping[
    "TransactionRows"
] = (
    refreshed_step3_mapping[
        "PLUCode"
    ]
    .map(
        statistics_lookup[
            "TransactionRows_New"
        ]
    )
    .astype("int64")
)


refreshed_step3_mapping[
    "TotalUnits"
] = (
    refreshed_step3_mapping[
        "PLUCode"
    ]
    .map(
        statistics_lookup[
            "TotalUnits_New"
        ]
    )
    .astype("int64")
)


refreshed_step3_mapping[
    "TotalTransactionValue"
] = (
    refreshed_step3_mapping[
        "PLUCode"
    ]
    .map(
        statistics_lookup[
            "TotalTransactionValue_New"
        ]
    )
    .astype(float)
    .round(2)
)


refreshed_step3_mapping[
    "MappingStatisticsSourceVersion"
] = (
    "DEDUPLICATED_SOURCE_V1"
)


refreshed_step3_mapping[
    "MappingStatisticsRefreshReason"
] = (
    "PLU statistics refreshed after removing "
    "10 confirmed duplicated export rows."
)


# ------------------------------------------------------------
# Validate that mapping decisions did not change
# ------------------------------------------------------------

decision_columns_to_preserve = [
    "PLUName_Original",
    "PLUName_Corrected",
    "CanonicalProductID_Step1",
    "CanonicalProductName_Step1",
    "CanonicalProductID_Step2",
    "CanonicalProductName_Step2",
    "CanonicalProductID_Step3",
    "CanonicalProductName_Step3",
    "MappingType",
    "MappingStatus",
    "BeverageSeries_Step2",
    "BeverageType_Step2",
    "SupplierLabel_Step2",
    "TierProductFamily_Step3",
    "NominalPriceTier_Step3",
    "MenuGeneration_Step3",
]


for column in decision_columns_to_preserve:

    left_values = (
        refreshed_step3_mapping[column]
        .astype("string")
        .fillna("<MISSING>")
    )

    right_values = (
        existing_step3_mapping[column]
        .astype("string")
        .fillna("<MISSING>")
    )

    assert left_values.eq(
        right_values
    ).all(), (
        f"Mapping decision column changed: {column}"
    )


assert len(refreshed_step3_mapping) == 236


assert refreshed_step3_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    refreshed_step3_mapping[
        "TransactionRows"
    ].sum()
) == 138_973


assert int(
    refreshed_step3_mapping[
        "TotalUnits"
    ].sum()
) == 140_894


assert np.isclose(
    refreshed_step3_mapping[
        "TotalTransactionValue"
    ].sum(),
    564_342.84,
    atol=0.001,
)


print("Step 3 mapping statistics refreshed safely.")
print()
print(
    "Mapping decisions changed:",
    0
)
print(
    "Mapping rows:",
    len(refreshed_step3_mapping)
)
print(
    "Canonical identities:",
    refreshed_step3_mapping[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Refreshed transaction rows:",
    f"{int(refreshed_step3_mapping['TransactionRows'].sum()):,}"
)
print(
    "Refreshed units:",
    f"{int(refreshed_step3_mapping['TotalUnits'].sum()):,}"
)

Step 3 mapping statistics refreshed safely.

Mapping decisions changed: 0
Mapping rows: 236
Canonical identities: 228
Refreshed transaction rows: 138,973
Refreshed units: 140,894


In [69]:
# ============================================================
# Mapping Refresh - Cell 6
# Save refreshed mapping and audit
# ============================================================

refreshed_step3_mapping.to_csv(
    REFRESHED_STEP3_MAPPING_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


mapping_refresh_audit.to_csv(
    MAPPING_REFRESH_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


saved_refreshed_mapping = pd.read_csv(
    REFRESHED_STEP3_MAPPING_OUTPUT_FILE,
    low_memory=False
)


saved_refreshed_mapping["TotalUnits"] = (
    pd.to_numeric(
        saved_refreshed_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


saved_refreshed_mapping["TransactionRows"] = (
    pd.to_numeric(
        saved_refreshed_mapping["TransactionRows"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


saved_refreshed_mapping[
    "TotalTransactionValue"
] = pd.to_numeric(
    saved_refreshed_mapping[
        "TotalTransactionValue"
    ],
    errors="raise"
)


assert len(saved_refreshed_mapping) == 236


assert saved_refreshed_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    saved_refreshed_mapping[
        "TransactionRows"
    ].sum()
) == 138_973


assert int(
    saved_refreshed_mapping[
        "TotalUnits"
    ].sum()
) == 140_894


assert np.isclose(
    saved_refreshed_mapping[
        "TotalTransactionValue"
    ].sum(),
    564_342.84,
    atol=0.001,
)


print("=" * 74)
print("STEP 1-3 MAPPING STATISTICS REFRESH COMPLETED")
print("=" * 74)
print()
print(
    "Mapping rows:",
    len(saved_refreshed_mapping)
)
print(
    "Canonical identities:",
    saved_refreshed_mapping[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Refreshed source rows:",
    f"{int(saved_refreshed_mapping['TransactionRows'].sum()):,}"
)
print(
    "Refreshed source units:",
    f"{int(saved_refreshed_mapping['TotalUnits'].sum()):,}"
)
print(
    "Products affected by deduplication:",
    len(mapping_refresh_audit)
)
print()
print("Refreshed Step 3 mapping:")
print(REFRESHED_STEP3_MAPPING_OUTPUT_FILE)

STEP 1-3 MAPPING STATISTICS REFRESH COMPLETED

Mapping rows: 236
Canonical identities: 228
Refreshed source rows: 138,973
Refreshed source units: 140,894
Products affected by deduplication: 3

Refreshed Step 3 mapping:
eden_datasets/UL_EDEN_master_product_mapping_step3_price_tier_metadata_deduplicated.csv


### I have to do the bulk seperation again as the the duplicate exports were from the bulk exports

In [70]:
# ============================================================
# Deduplicated Step 4 - Cell 1
# Load deduplicated transactions, deduplicated audit,
# and refreshed Step 3 product mapping
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


DEDUPLICATED_TRANSACTION_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected_deduplicated.csv"
)


DEDUPLICATED_FULL_AUDIT_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_unitsold_corrected_with_audit_columns_deduplicated.csv"
)


REFRESHED_STEP3_MAPPING_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_master_product_mapping_step3_price_tier_metadata_deduplicated.csv"
)


STEP4_FINAL_TRANSACTION_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_step4_bulk_separated_enriched_deduplicated.csv"
)


STEP4_FINAL_BULK_REGISTRY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_confirmed_bulk_line_registry_deduplicated.csv"
)


STEP4_FINAL_BULK_PRODUCT_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_bulk_product_summary_deduplicated.csv"
)


STEP4_FINAL_DAILY_SPLIT_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_observed_daily_demand_split_audit_deduplicated.csv"
)


STEP4_FINAL_CLASSIFICATION_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step4_bulk_classification_summary_deduplicated.csv"
)


for required_file in [
    DEDUPLICATED_TRANSACTION_INPUT_FILE,
    DEDUPLICATED_FULL_AUDIT_INPUT_FILE,
    REFRESHED_STEP3_MAPPING_INPUT_FILE,
]:

    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file was not found:\n{required_file}"
        )


step4_source = pd.read_csv(
    DEDUPLICATED_TRANSACTION_INPUT_FILE,
    low_memory=False
)


step4_audit = pd.read_csv(
    DEDUPLICATED_FULL_AUDIT_INPUT_FILE,
    low_memory=False
)


step4_mapping = pd.read_csv(
    REFRESHED_STEP3_MAPPING_INPUT_FILE,
    low_memory=False,
)


step4_source_columns = step4_source.columns.tolist()


print("Deduplicated Step 4 input files loaded.")
print()
print(
    "Transaction rows:",
    f"{len(step4_source):,}"
)
print(
    "Audit rows:",
    f"{len(step4_audit):,}"
)
print(
    "Mapping rows:",
    f"{len(step4_mapping):,}"
)

Deduplicated Step 4 input files loaded.

Transaction rows: 138,973
Audit rows: 138,973
Mapping rows: 236


In [71]:
# ============================================================
# Deduplicated Step 4 - Cell 2
# Convert data types and validate input totals
# ============================================================

for dataframe in [
    step4_source,
    step4_audit,
]:

    dataframe["TransDate"] = pd.to_datetime(
        dataframe["TransDate"],
        errors="raise"
    )

    dataframe["Date"] = (
        pd.to_datetime(
            dataframe["Date"],
            errors="raise"
        )
        .dt.normalize()
    )

    dataframe["PLUCode"] = (
        pd.to_numeric(
            dataframe["PLUCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["GroupCode"] = (
        pd.to_numeric(
            dataframe["GroupCode"],
            errors="raise"
        )
        .astype("int64")
    )

    dataframe["TransValue"] = pd.to_numeric(
        dataframe["TransValue"],
        errors="raise"
    )

    dataframe["UnitSold"] = (
        pd.to_numeric(
            dataframe["UnitSold"],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


step4_audit["UnitSold_Original"] = (
    pd.to_numeric(
        step4_audit["UnitSold_Original"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step4_audit["UnitPrice"] = pd.to_numeric(
    step4_audit["UnitPrice"],
    errors="coerce"
)


step4_mapping["PLUCode"] = (
    pd.to_numeric(
        step4_mapping["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


step4_mapping["TransactionRows"] = (
    pd.to_numeric(
        step4_mapping["TransactionRows"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


step4_mapping["TotalUnits"] = (
    pd.to_numeric(
        step4_mapping["TotalUnits"],
        errors="raise"
    )
    .round()
    .astype("int64")
)


# ------------------------------------------------------------
# Robust CSV Boolean converter
# ------------------------------------------------------------

def convert_csv_boolean(
    series,
    allow_missing=False
):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    converted = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })

    missing_text_mask = (
        normalized.isna()
        | normalized.isin([
            "",
            "<na>",
            "nan",
            "none",
        ])
    )

    invalid_mask = (
        converted.isna()
        & ~missing_text_mask
    )

    if invalid_mask.any():

        invalid_values = (
            series.loc[
                invalid_mask
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unexpected Boolean values found:\n"
            f"{invalid_values}"
        )

    if allow_missing:
        return converted.astype("boolean")

    if converted.isna().any():
        raise ValueError(
            "A required Boolean column contains "
            "missing values."
        )

    return converted.astype(bool)


step4_audit[
    "MultiUnitAdjustedFlag_Parsed"
] = convert_csv_boolean(
    step4_audit[
        "MultiUnitAdjustedFlag"
    ]
)


step4_audit[
    "IsExactPriceMultiple_Parsed"
] = convert_csv_boolean(
    step4_audit[
        "IsExactPriceMultiple"
    ],
    allow_missing=True
)


# ------------------------------------------------------------
# Validate source state
# ------------------------------------------------------------

assert len(step4_source) == 138_973

assert len(step4_audit) == 138_973

assert len(step4_mapping) == 236


assert step4_source[
    "PLUCode"
].nunique() == 236


assert step4_mapping[
    "PLUCode"
].is_unique


assert step4_mapping[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    step4_source["UnitSold"].sum()
) == 140_894


assert int(
    step4_source["UnitSold"].gt(1).sum()
) == 51


assert int(
    step4_mapping["TransactionRows"].sum()
) == 138_973


assert int(
    step4_mapping["TotalUnits"].sum()
) == 140_894


step4_source_value_cents = int(
    (
        step4_source["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert step4_source_value_cents == 56_434_284


print("Deduplicated Step 4 inputs validated.")
print()
print(
    "Rows:",
    f"{len(step4_source):,}"
)
print(
    "Units:",
    f"{int(step4_source['UnitSold'].sum()):,}"
)
print(
    "Multi-unit rows:",
    f"{int(step4_source['UnitSold'].gt(1).sum()):,}"
)
print(
    "Transaction value:",
    f"€{step4_source_value_cents / 100:,.2f}"
)

Deduplicated Step 4 inputs validated.

Rows: 138,973
Units: 140,894
Multi-unit rows: 51
Transaction value: €564,342.84


In [72]:
# ============================================================
# Deduplicated Step 4 - Cell 3
# Confirm source and audit match row by row
# ============================================================

missing_source_columns = [
    column
    for column in step4_source_columns
    if column not in step4_audit.columns
]


if missing_source_columns:
    raise ValueError(
        "The deduplicated audit is missing source columns:\n"
        f"{missing_source_columns}"
    )


def prepare_alignment_frame(
    dataframe,
    columns
):
    aligned = pd.DataFrame(
        index=dataframe.index
    )

    date_columns = {
        "TransDate",
        "Date",
    }

    numeric_columns = {
        "TransValue",
        "GroupCode",
        "PLUCode",
        "Hour",
        "Month",
        "WeekOfYear",
        "UnitSold",
    }

    for column in columns:

        if column in date_columns:

            aligned[column] = pd.to_datetime(
                dataframe[column],
                errors="raise"
            )

        elif column in numeric_columns:

            aligned[column] = pd.to_numeric(
                dataframe[column],
                errors="raise"
            )

        else:

            aligned[column] = (
                dataframe[column]
                .astype("string")
                .str.strip()
                .fillna("<MISSING>")
            )

    return aligned


step4_source_alignment = prepare_alignment_frame(
    step4_source,
    step4_source_columns
)


step4_audit_alignment = prepare_alignment_frame(
    step4_audit,
    step4_source_columns
)


pd.testing.assert_frame_equal(
    step4_source_alignment,
    step4_audit_alignment,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000000001,
)


print(
    "Deduplicated source and audit match row by row."
)

Deduplicated source and audit match row by row.


In [73]:
# ============================================================
# Deduplicated Step 4 - Cell 4
# Validate the remaining 51 confirmed bulk lines
# ============================================================

confirmed_bulk_mask = (
    step4_audit[
        "MultiUnitAdjustedFlag_Parsed"
    ]
    .fillna(False)
    .astype(bool)
)


assert int(
    confirmed_bulk_mask.sum()
) == 51, (
    "Expected exactly 51 confirmed bulk rows."
)


assert confirmed_bulk_mask.reset_index(
    drop=True
).equals(
    step4_source[
        "UnitSold"
    ]
    .gt(1)
    .reset_index(drop=True)
), (
    "The confirmed bulk flag does not match "
    "the remaining multi-unit transaction rows."
)


assert step4_source.loc[
    confirmed_bulk_mask,
    "UnitSold",
].gt(1).all()


assert step4_audit.loc[
    confirmed_bulk_mask,
    "UnitSold_Original",
].eq(1).all()


assert step4_audit.loc[
    confirmed_bulk_mask,
    "IsExactPriceMultiple_Parsed",
].eq(True).all()


assert step4_audit.loc[
    confirmed_bulk_mask,
    "UnitPrice",
].notna().all()


confirmed_bulk_units = int(
    step4_source.loc[
        confirmed_bulk_mask,
        "UnitSold",
    ].sum()
)


confirmed_additional_units = int(
    (
        step4_source.loc[
            confirmed_bulk_mask,
            "UnitSold",
        ]
        -
        step4_audit.loc[
            confirmed_bulk_mask,
            "UnitSold_Original",
        ]
    ).sum()
)


assert confirmed_bulk_units == 1_972

assert confirmed_additional_units == 1_921


assert step4_source.loc[
    confirmed_bulk_mask,
    "PLUCode",
].nunique() == 11


print("Remaining confirmed bulk set validated.")
print()
print(
    "Bulk lines:",
    f"{int(confirmed_bulk_mask.sum()):,}"
)
print(
    "Bulk products:",
    step4_source.loc[
        confirmed_bulk_mask,
        "PLUCode",
    ].nunique()
)
print(
    "Bulk units:",
    f"{confirmed_bulk_units:,}"
)
print(
    "Additional inferred units:",
    f"{confirmed_additional_units:,}"
)

Remaining confirmed bulk set validated.

Bulk lines: 51
Bulk products: 11
Bulk units: 1,972
Additional inferred units: 1,921


In [74]:
# ============================================================
# Deduplicated Step 4 - Cell 5
# Create isolated line-level demand split
# ============================================================

step4_transactions = step4_source.copy()


step4_transactions[
    "SourceRowNumber_Step4"
] = np.arange(
    1,
    len(step4_transactions) + 1
)


step4_transactions[
    "BulkOrderFlag_Step4"
] = confirmed_bulk_mask.to_numpy(
    dtype=bool
)


step4_transactions[
    "NormalDemandUnits_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    0,
    step4_transactions["UnitSold"]
).astype("int64")


step4_transactions[
    "BulkDemandUnits_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    step4_transactions["UnitSold"],
    0
).astype("int64")


step4_transactions[
    "TotalDemandUnits_Step4"
] = step4_transactions[
    "UnitSold"
].astype("int64")


step4_transactions[
    "UnitSoldOriginal_Audit_Step4"
] = step4_audit[
    "UnitSold_Original"
].to_numpy()


step4_transactions[
    "BulkExtraUnitsAboveOriginalOne_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    (
        step4_transactions["UnitSold"]
        -
        step4_transactions[
            "UnitSoldOriginal_Audit_Step4"
        ]
    ),
    0
).astype("int64")


step4_transactions[
    "BulkUnitPrice_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    step4_audit["UnitPrice"],
    np.nan
)


step4_transactions[
    "BulkAdjustmentMethod_Step4"
] = pd.Series(
    pd.NA,
    index=step4_transactions.index,
    dtype="string"
)


step4_transactions.loc[
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "BulkAdjustmentMethod_Step4",
] = (
    step4_audit.loc[
        confirmed_bulk_mask,
        "UnitSoldAdjustmentMethod",
    ]
    .astype("string")
    .to_numpy()
)


step4_transactions[
    "BulkClassification_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "CONFIRMED_BULK_ORDER_LINE",
    "NORMAL_OR_UNCLASSIFIED_LINE"
)


step4_transactions[
    "BulkClassificationReason_Step4"
] = np.where(
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    (
        "Confirmed multi-unit transaction line from "
        "the corrected and deduplicated UnitSold audit."
    ),
    (
        "No confirmed multi-unit bulk evidence in "
        "the UnitSold correction audit."
    )
)


step4_transactions[
    "BulkClassificationVersion_Step4"
] = (
    "STEP_4_DEDUPLICATED_CONFIRMED_BULK_V1"
)


step4_transactions[
    "BulkOrderLineID_Step4"
] = pd.Series(
    pd.NA,
    index=step4_transactions.index,
    dtype="string"
)


bulk_line_ids = [
    f"BULK_LINE_{number:04d}"
    for number in range(
        1,
        int(
            step4_transactions[
                "BulkOrderFlag_Step4"
            ].sum()
        ) + 1
    )
]


step4_transactions.loc[
    step4_transactions[
        "BulkOrderFlag_Step4"
    ],
    "BulkOrderLineID_Step4",
] = bulk_line_ids


# ------------------------------------------------------------
# Validate demand split
# ------------------------------------------------------------

assert (
    step4_transactions[
        "NormalDemandUnits_Step4"
    ]
    +
    step4_transactions[
        "BulkDemandUnits_Step4"
    ]
    ==
    step4_transactions[
        "TotalDemandUnits_Step4"
    ]
).all()


assert (
    step4_transactions[
        "TotalDemandUnits_Step4"
    ]
    ==
    step4_transactions["UnitSold"]
).all()


assert int(
    step4_transactions[
        "NormalDemandUnits_Step4"
    ].sum()
) == 138_922


assert int(
    step4_transactions[
        "BulkDemandUnits_Step4"
    ].sum()
) == 1_972


assert int(
    step4_transactions[
        "TotalDemandUnits_Step4"
    ].sum()
) == 140_894


assert int(
    step4_transactions[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 1_921


print("Deduplicated line-level demand split created.")
print()
print(
    "Normal source units:",
    f"{int(step4_transactions['NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "Bulk source units:",
    f"{int(step4_transactions['BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Total source units:",
    f"{int(step4_transactions['TotalDemandUnits_Step4'].sum()):,}"
)

Deduplicated line-level demand split created.

Normal source units: 138,922
Bulk source units: 1,972
Total source units: 140,894


In [75]:
# ============================================================
# Deduplicated Step 4 - Cell 6
# Attach final refreshed Step 3 product identity
# ============================================================

mapping_columns_required = [
    "PLUCode",
    "PLUName_Original",
    "PLUName_Corrected",
    "CanonicalProductID_Step3",
    "CanonicalProductName_Step3",
    "MappingStatus",
    "BeverageSeries_Step2",
    "BeverageType_Step2",
    "SupplierLabel_Step2",
    "TierProductFamily_Step3",
    "NominalPriceTier_Step3",
    "MenuGeneration_Step3",
]


missing_mapping_columns = [
    column
    for column in mapping_columns_required
    if column not in step4_mapping.columns
]


if missing_mapping_columns:
    raise ValueError(
        "Refreshed Step 3 mapping is missing columns:\n"
        f"{missing_mapping_columns}"
    )


step4_enriched = (
    step4_transactions
    .merge(
        step4_mapping[
            mapping_columns_required
        ],
        on="PLUCode",
        how="left",
        validate="many_to_one",
        sort=False
    )
    .sort_values(
        "SourceRowNumber_Step4"
    )
    .reset_index(drop=True)
)


assert len(step4_enriched) == 138_973


assert step4_enriched[
    "CanonicalProductID_Step3"
].notna().all()


assert step4_enriched[
    "CanonicalProductName_Step3"
].notna().all()


step4_enriched[
    "ForecastingEligible_Step4"
] = ~(
    step4_enriched[
        "MappingStatus"
    ]
    .astype("string")
    .str.strip()
    .eq("EXCLUDE_FROM_FORECASTING")
)


# Confirm original source fields remain unchanged
enriched_source_alignment = prepare_alignment_frame(
    step4_enriched,
    step4_source_columns
)


pd.testing.assert_frame_equal(
    step4_source_alignment,
    enriched_source_alignment,
    check_dtype=False,
    check_exact=False,
    rtol=0,
    atol=0.000000001,
)


print("Refreshed Step 3 mapping attached successfully.")
print()
print(
    "Transaction rows:",
    f"{len(step4_enriched):,}"
)
print(
    "Canonical products including OPEN UL:",
    step4_enriched[
        "CanonicalProductID_Step3"
    ].nunique()
)

Refreshed Step 3 mapping attached successfully.

Transaction rows: 138,973
Canonical products including OPEN UL: 228


In [76]:
# ============================================================
# Deduplicated Step 4 - Cell 7
# Validate OPEN UL and forecastable totals
# ============================================================

open_ul_name_mask = (
    step4_enriched[
        "PLUName"
    ]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


open_ul_mapping_mask = ~(
    step4_enriched[
        "ForecastingEligible_Step4"
    ]
)


assert open_ul_name_mask.equals(
    open_ul_mapping_mask
)


assert int(
    open_ul_name_mask.sum()
) == 24_736


assert int(
    step4_enriched.loc[
        open_ul_name_mask,
        "TotalDemandUnits_Step4",
    ].sum()
) == 24_736


assert not step4_enriched.loc[
    open_ul_name_mask,
    "BulkOrderFlag_Step4",
].any()


forecastable_mask = (
    step4_enriched[
        "ForecastingEligible_Step4"
    ]
)


forecastable_rows = int(
    forecastable_mask.sum()
)


forecastable_normal_units = int(
    step4_enriched.loc[
        forecastable_mask,
        "NormalDemandUnits_Step4",
    ].sum()
)


forecastable_bulk_units = int(
    step4_enriched.loc[
        forecastable_mask,
        "BulkDemandUnits_Step4",
    ].sum()
)


forecastable_total_units = int(
    step4_enriched.loc[
        forecastable_mask,
        "TotalDemandUnits_Step4",
    ].sum()
)


assert forecastable_rows == 114_237

assert forecastable_normal_units == 114_186

assert forecastable_bulk_units == 1_972

assert forecastable_total_units == 116_158


assert (
    forecastable_normal_units
    +
    forecastable_bulk_units
    ==
    forecastable_total_units
)


assert step4_enriched.loc[
    forecastable_mask,
    "CanonicalProductID_Step3",
].nunique() == 227


print("Forecastable demand totals validated.")
print()
print(
    "Forecastable transaction rows:",
    f"{forecastable_rows:,}"
)
print(
    "Forecastable canonical products:",
    step4_enriched.loc[
        forecastable_mask,
        "CanonicalProductID_Step3",
    ].nunique()
)
print(
    "Forecastable normal units:",
    f"{forecastable_normal_units:,}"
)
print(
    "Forecastable bulk units:",
    f"{forecastable_bulk_units:,}"
)
print(
    "Forecastable total units:",
    f"{forecastable_total_units:,}"
)

Forecastable demand totals validated.

Forecastable transaction rows: 114,237
Forecastable canonical products: 227
Forecastable normal units: 114,186
Forecastable bulk units: 1,972
Forecastable total units: 116,158


In [77]:
# ============================================================
# Deduplicated Step 4 - Cell 8
# Create bulk registry, product summary and daily split audit
# ============================================================

confirmed_bulk_registry = (
    step4_enriched.loc[
        step4_enriched[
            "BulkOrderFlag_Step4"
        ],
        [
            "SourceRowNumber_Step4",
            "BulkOrderLineID_Step4",
            "Date",
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "PLUName_Original",
            "PLUName_Corrected",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
            "GroupCode",
            "GroupName",
            "TransValue",
            "BulkUnitPrice_Step4",
            "UnitSoldOriginal_Audit_Step4",
            "UnitSold",
            "NormalDemandUnits_Step4",
            "BulkDemandUnits_Step4",
            "BulkExtraUnitsAboveOriginalOne_Step4",
            "BulkAdjustmentMethod_Step4",
            "BulkClassification_Step4",
            "BulkClassificationReason_Step4",
            "BulkClassificationVersion_Step4",
        ],
    ]
    .sort_values(
        [
            "Date",
            "TransactionID",
            "PLUCode",
            "SourceRowNumber_Step4",
        ]
    )
    .reset_index(drop=True)
)


assert len(confirmed_bulk_registry) == 51

assert confirmed_bulk_registry[
    "BulkOrderLineID_Step4"
].is_unique

assert int(
    confirmed_bulk_registry[
        "BulkDemandUnits_Step4"
    ].sum()
) == 1_972

assert int(
    confirmed_bulk_registry[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 1_921

assert confirmed_bulk_registry[
    "CanonicalProductID_Step3"
].nunique() == 11


bulk_product_summary = (
    confirmed_bulk_registry
    .groupby(
        [
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
        ],
        as_index=False
    )
    .agg(
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
        BulkLineCount=(
            "BulkOrderLineID_Step4",
            "size"
        ),
        UniqueBulkTransactions=(
            "TransactionID",
            "nunique"
        ),
        FirstBulkDate=(
            "Date",
            "min"
        ),
        LastBulkDate=(
            "Date",
            "max"
        ),
        BulkDemandUnits=(
            "BulkDemandUnits_Step4",
            "sum"
        ),
        AdditionalUnitsIdentified=(
            "BulkExtraUnitsAboveOriginalOne_Step4",
            "sum"
        ),
        BulkTransactionValue=(
            "TransValue",
            "sum"
        ),
        MinimumBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "min"
        ),
        MaximumBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "max"
        ),
        MeanBulkLineUnits=(
            "BulkDemandUnits_Step4",
            "mean"
        ),
    )
)


bulk_product_summary[
    "BulkTransactionValue"
] = bulk_product_summary[
    "BulkTransactionValue"
].round(2)


bulk_product_summary[
    "MeanBulkLineUnits"
] = bulk_product_summary[
    "MeanBulkLineUnits"
].round(2)


bulk_product_summary = (
    bulk_product_summary
    .sort_values(
        "BulkDemandUnits",
        ascending=False
    )
    .reset_index(drop=True)
)


assert len(bulk_product_summary) == 11

assert int(
    bulk_product_summary[
        "BulkLineCount"
    ].sum()
) == 51

assert int(
    bulk_product_summary[
        "BulkDemandUnits"
    ].sum()
) == 1_972


forecastable_transactions = (
    step4_enriched.loc[
        forecastable_mask
    ]
    .copy()
)


forecastable_transactions[
    "_BulkLineCount"
] = forecastable_transactions[
    "BulkOrderFlag_Step4"
].astype("int64")


observed_daily_split_audit = (
    forecastable_transactions
    .groupby(
        [
            "Date",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
        ],
        as_index=False
    )
    .agg(
        NormalDemandUnits=(
            "NormalDemandUnits_Step4",
            "sum"
        ),
        BulkDemandUnits=(
            "BulkDemandUnits_Step4",
            "sum"
        ),
        TotalDemandUnits=(
            "TotalDemandUnits_Step4",
            "sum"
        ),
        TransactionLineCount=(
            "TransactionID",
            "size"
        ),
        UniqueTransactionCount=(
            "TransactionID",
            "nunique"
        ),
        BulkLineCount=(
            "_BulkLineCount",
            "sum"
        ),
        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),
    )
)


observed_daily_split_audit[
    "HasBulkDemand"
] = (
    observed_daily_split_audit[
        "BulkDemandUnits"
    ].gt(0)
)


assert (
    observed_daily_split_audit[
        "NormalDemandUnits"
    ]
    +
    observed_daily_split_audit[
        "BulkDemandUnits"
    ]
    ==
    observed_daily_split_audit[
        "TotalDemandUnits"
    ]
).all()


assert int(
    observed_daily_split_audit[
        "NormalDemandUnits"
    ].sum()
) == 114_186


assert int(
    observed_daily_split_audit[
        "BulkDemandUnits"
    ].sum()
) == 1_972


assert int(
    observed_daily_split_audit[
        "TotalDemandUnits"
    ].sum()
) == 116_158


assert int(
    observed_daily_split_audit[
        "BulkLineCount"
    ].sum()
) == 51


assert not observed_daily_split_audit.duplicated(
    [
        "Date",
        "CanonicalProductID_Step3",
    ]
).any()


classification_summary = pd.DataFrame([
    {
        "Scope": "COMPLETE_SOURCE",
        "TransactionRows": len(step4_enriched),
        "CanonicalProducts": (
            step4_enriched[
                "CanonicalProductID_Step3"
            ].nunique()
        ),
        "ConfirmedBulkLines": int(
            step4_enriched[
                "BulkOrderFlag_Step4"
            ].sum()
        ),
        "NormalDemandUnits": int(
            step4_enriched[
                "NormalDemandUnits_Step4"
            ].sum()
        ),
        "BulkDemandUnits": int(
            step4_enriched[
                "BulkDemandUnits_Step4"
            ].sum()
        ),
        "TotalDemandUnits": int(
            step4_enriched[
                "TotalDemandUnits_Step4"
            ].sum()
        ),
    },
    {
        "Scope": "FORECASTABLE_EXCLUDING_OPEN_UL",
        "TransactionRows": forecastable_rows,
        "CanonicalProducts": (
            step4_enriched.loc[
                forecastable_mask,
                "CanonicalProductID_Step3",
            ].nunique()
        ),
        "ConfirmedBulkLines": int(
            step4_enriched.loc[
                forecastable_mask,
                "BulkOrderFlag_Step4",
            ].sum()
        ),
        "NormalDemandUnits": forecastable_normal_units,
        "BulkDemandUnits": forecastable_bulk_units,
        "TotalDemandUnits": forecastable_total_units,
    },
    {
        "Scope": "OPEN_UL_EXCLUDED",
        "TransactionRows": int(
            open_ul_name_mask.sum()
        ),
        "CanonicalProducts": 1,
        "ConfirmedBulkLines": 0,
        "NormalDemandUnits": int(
            step4_enriched.loc[
                open_ul_name_mask,
                "NormalDemandUnits_Step4",
            ].sum()
        ),
        "BulkDemandUnits": 0,
        "TotalDemandUnits": int(
            step4_enriched.loc[
                open_ul_name_mask,
                "TotalDemandUnits_Step4",
            ].sum()
        ),
    },
])


display(bulk_product_summary)

display(classification_summary)

,CanonicalProductID_Step3,CanonicalProductName_Step3,SourcePLUCount,BulkLineCount,UniqueBulkTransactions,FirstBulkDate,LastBulkDate,BulkDemandUnits,AdditionalUnitsIdentified,BulkTransactionValue,MinimumBulkLineUnits,MaximumBulkLineUnits,MeanBulkLineUnits
0,PLU_4241474,SOUP OF THE DAY,1,14,11,2025-07-08,2025-08-01,670,656,2345.0,3,114,47.86
1,PLU_4241480,MAINS 3,1,8,7,2025-07-09,2025-08-01,397,389,3573.0,23,61,49.62
2,PLU_4241483,VEGT MAINS 3,1,4,4,2025-07-08,2025-07-25,258,254,2322.0,45,114,64.50
3,PLU_3219121,CARTON OF WATER,1,10,8,2025-07-08,2025-08-01,240,230,528.0,15,30,24.00
4,PLU_4241428,HAM & CHEESE SANDWICH CT,1,2,2,2025-09-24,2025-09-26,122,120,610.0,61,61,61.00
5,PLU_2000000019,FULL FAT CAN,1,7,5,2025-05-29,2025-07-25,93,86,167.4,4,27,13.29
6,PLU_42534,€9 KIMBOCK,1,2,2,2025-10-29,2025-10-29,82,80,738.0,41,41,41.00
7,PLU_4241430,BOX SALADS,1,1,1,2025-07-28,2025-07-28,49,48,318.5,49,49,49.00
8,PLU_4241476,KIMBOX MAINS 2,1,1,1,2025-07-28,2025-07-28,49,48,343.0,49,49,49.00
9,PLU_4241425,CAFFE MOCHA BLISS BALLS,1,1,1,2025-07-21,2025-07-21,6,5,21.0,6,6,6.00


,Scope,TransactionRows,CanonicalProducts,ConfirmedBulkLines,NormalDemandUnits,BulkDemandUnits,TotalDemandUnits
0,COMPLETE_SOURCE,138973,228,51,138922,1972,140894
1,FORECASTABLE_EXCLUDING_OPEN_UL,114237,227,51,114186,1972,116158
2,OPEN_UL_EXCLUDED,24736,1,0,24736,0,24736


In [78]:
# ============================================================
# Deduplicated Step 4 - Cell 9
# Save the corrected Step 4 outputs
# ============================================================

step4_enriched.to_csv(
    STEP4_FINAL_TRANSACTION_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


confirmed_bulk_registry.to_csv(
    STEP4_FINAL_BULK_REGISTRY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


bulk_product_summary.to_csv(
    STEP4_FINAL_BULK_PRODUCT_SUMMARY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


observed_daily_split_audit.to_csv(
    STEP4_FINAL_DAILY_SPLIT_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


classification_summary.to_csv(
    STEP4_FINAL_CLASSIFICATION_SUMMARY_OUTPUT_FILE,
    index=False
)


print("Corrected Step 4 files saved successfully.")
print()

print("1. Final enriched transaction dataset:")
print(STEP4_FINAL_TRANSACTION_OUTPUT_FILE)

print()
print("2. Final confirmed bulk registry:")
print(STEP4_FINAL_BULK_REGISTRY_OUTPUT_FILE)

print()
print("3. Final bulk product summary:")
print(STEP4_FINAL_BULK_PRODUCT_SUMMARY_OUTPUT_FILE)

print()
print("4. Final observed daily split audit:")
print(STEP4_FINAL_DAILY_SPLIT_AUDIT_OUTPUT_FILE)

print()
print("5. Final classification summary:")
print(STEP4_FINAL_CLASSIFICATION_SUMMARY_OUTPUT_FILE)

Corrected Step 4 files saved successfully.

1. Final enriched transaction dataset:
eden_datasets/UL_EDEN_transactions_step4_bulk_separated_enriched_deduplicated.csv

2. Final confirmed bulk registry:
eden_datasets/UL_EDEN_step4_confirmed_bulk_line_registry_deduplicated.csv

3. Final bulk product summary:
eden_datasets/UL_EDEN_step4_bulk_product_summary_deduplicated.csv

4. Final observed daily split audit:
eden_datasets/UL_EDEN_step4_observed_daily_demand_split_audit_deduplicated.csv

5. Final classification summary:
eden_datasets/UL_EDEN_step4_bulk_classification_summary_deduplicated.csv


In [79]:
# ============================================================
# Deduplicated Step 4 - Cell 10
# Reload and validate the final saved Step 4 dataset
# ============================================================

saved_step4_final = pd.read_csv(
    STEP4_FINAL_TRANSACTION_OUTPUT_FILE,
    low_memory=False,
    parse_dates=[
        "TransDate",
        "Date",
    ]
)


for numeric_column in [
    "PLUCode",
    "UnitSold",
    "NormalDemandUnits_Step4",
    "BulkDemandUnits_Step4",
    "TotalDemandUnits_Step4",
    "BulkExtraUnitsAboveOriginalOne_Step4",
]:

    saved_step4_final[
        numeric_column
    ] = pd.to_numeric(
        saved_step4_final[
            numeric_column
        ],
        errors="raise"
    )


saved_step4_final["PLUCode"] = (
    saved_step4_final["PLUCode"]
    .astype("int64")
)


for integer_column in [
    "UnitSold",
    "NormalDemandUnits_Step4",
    "BulkDemandUnits_Step4",
    "TotalDemandUnits_Step4",
    "BulkExtraUnitsAboveOriginalOne_Step4",
]:

    saved_step4_final[
        integer_column
    ] = (
        saved_step4_final[
            integer_column
        ]
        .round()
        .astype("int64")
    )


saved_step4_final[
    "BulkOrderFlag_Step4"
] = convert_csv_boolean(
    saved_step4_final[
        "BulkOrderFlag_Step4"
    ]
)


saved_step4_final[
    "ForecastingEligible_Step4"
] = convert_csv_boolean(
    saved_step4_final[
        "ForecastingEligible_Step4"
    ]
)


saved_bulk_mask = (
    saved_step4_final[
        "BulkOrderFlag_Step4"
    ]
)


saved_forecastable_mask = (
    saved_step4_final[
        "ForecastingEligible_Step4"
    ]
)


assert len(saved_step4_final) == 138_973


assert saved_step4_final[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    saved_bulk_mask.sum()
) == 51


assert int(
    saved_step4_final[
        "NormalDemandUnits_Step4"
    ].sum()
) == 138_922


assert int(
    saved_step4_final[
        "BulkDemandUnits_Step4"
    ].sum()
) == 1_972


assert int(
    saved_step4_final[
        "TotalDemandUnits_Step4"
    ].sum()
) == 140_894


assert int(
    saved_step4_final[
        "BulkExtraUnitsAboveOriginalOne_Step4"
    ].sum()
) == 1_921


assert (
    saved_step4_final[
        "NormalDemandUnits_Step4"
    ]
    +
    saved_step4_final[
        "BulkDemandUnits_Step4"
    ]
    ==
    saved_step4_final[
        "TotalDemandUnits_Step4"
    ]
).all()


assert int(
    saved_forecastable_mask.sum()
) == 114_237


assert saved_step4_final.loc[
    saved_forecastable_mask,
    "CanonicalProductID_Step3",
].nunique() == 227


assert int(
    saved_step4_final.loc[
        saved_forecastable_mask,
        "NormalDemandUnits_Step4",
    ].sum()
) == 114_186


assert int(
    saved_step4_final.loc[
        saved_forecastable_mask,
        "BulkDemandUnits_Step4",
    ].sum()
) == 1_972


assert int(
    saved_step4_final.loc[
        saved_forecastable_mask,
        "TotalDemandUnits_Step4",
    ].sum()
) == 116_158


print("=" * 74)
print("DEDUPLICATED STEP 4 BULK SEPARATION COMPLETED")
print("=" * 74)
print()
print(
    "Transaction rows:",
    f"{len(saved_step4_final):,}"
)
print(
    "Canonical product identities:",
    saved_step4_final[
        "CanonicalProductID_Step3"
    ].nunique()
)
print(
    "Confirmed bulk lines:",
    f"{int(saved_bulk_mask.sum()):,}"
)
print(
    "Confirmed bulk units:",
    f"{int(saved_step4_final['BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Normal source units:",
    f"{int(saved_step4_final['NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "All source units:",
    f"{int(saved_step4_final['TotalDemandUnits_Step4'].sum()):,}"
)
print()
print(
    "Forecastable normal units:",
    f"{int(saved_step4_final.loc[saved_forecastable_mask, 'NormalDemandUnits_Step4'].sum()):,}"
)
print(
    "Forecastable bulk units:",
    f"{int(saved_step4_final.loc[saved_forecastable_mask, 'BulkDemandUnits_Step4'].sum()):,}"
)
print(
    "Forecastable total units:",
    f"{int(saved_step4_final.loc[saved_forecastable_mask, 'TotalDemandUnits_Step4'].sum()):,}"
)
print()
print("Final Step 4 file:")
print(STEP4_FINAL_TRANSACTION_OUTPUT_FILE)

DEDUPLICATED STEP 4 BULK SEPARATION COMPLETED

Transaction rows: 138,973
Canonical product identities: 228
Confirmed bulk lines: 51
Confirmed bulk units: 1,972
Normal source units: 138,922
All source units: 140,894

Forecastable normal units: 114,186
Forecastable bulk units: 1,972
Forecastable total units: 116,158

Final Step 4 file:
eden_datasets/UL_EDEN_transactions_step4_bulk_separated_enriched_deduplicated.csv


## I am going to create the final product-level daily forecasting dataset

In [80]:
# ============================================================
# Step 5 - Cell 1
# Load and validate the final deduplicated Step 4 source
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


DATA_FOLDER = Path("eden_datasets")


# Final approved input from deduplicated Step 4
STEP5_INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_step4_bulk_separated_enriched_deduplicated.csv"
)


# Planned Step 5 outputs
STEP5_FINAL_DAILY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_canonical_product_daily_demand_forecasting_final.csv"
)


STEP5_PRODUCT_METADATA_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step5_final_product_metadata.csv"
)


STEP5_OPERATING_CALENDAR_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step5_operating_calendar.csv"
)


STEP5_OBSERVED_DAILY_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step5_observed_daily_demand_audit.csv"
)


STEP5_VALIDATION_SUMMARY_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_step5_final_validation_summary.csv"
)


if not STEP5_INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Final Step 4 input file was not found:\n"
        f"{STEP5_INPUT_FILE}"
    )


step5_source = pd.read_csv(
    STEP5_INPUT_FILE,
    low_memory=False
)


# ------------------------------------------------------------
# Robust Boolean conversion
# ------------------------------------------------------------

def convert_csv_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    converted = normalized.map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    })

    missing_mask = (
        normalized.isna()
        | normalized.isin([
            "",
            "<na>",
            "nan",
            "none",
        ])
    )

    invalid_mask = (
        converted.isna()
        & ~missing_mask
    )

    if invalid_mask.any():
        invalid_values = (
            series.loc[invalid_mask]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Unexpected Boolean values found:\n"
            f"{invalid_values}"
        )

    if converted.isna().any():
        raise ValueError(
            "A required Boolean column contains missing values."
        )

    return converted.astype(bool)


# ------------------------------------------------------------
# Confirm required columns exist
# ------------------------------------------------------------

required_step5_columns = [
    "Date",
    "TransDate",
    "TransactionID",
    "PLUCode",
    "PLUName",
    "CanonicalProductID_Step3",
    "CanonicalProductName_Step3",
    "MappingStatus",
    "UnitSold",
    "NormalDemandUnits_Step4",
    "BulkDemandUnits_Step4",
    "TotalDemandUnits_Step4",
    "BulkOrderFlag_Step4",
    "ForecastingEligible_Step4",
]


missing_step5_columns = [
    column
    for column in required_step5_columns
    if column not in step5_source.columns
]


if missing_step5_columns:
    raise ValueError(
        "Step 5 input is missing required columns:\n"
        f"{missing_step5_columns}"
    )


# ------------------------------------------------------------
# Convert data types
# ------------------------------------------------------------

step5_source["TransDate"] = pd.to_datetime(
    step5_source["TransDate"],
    errors="raise"
)


step5_source["Date"] = (
    pd.to_datetime(
        step5_source["Date"],
        errors="raise"
    )
    .dt.normalize()
)


step5_source["PLUCode"] = (
    pd.to_numeric(
        step5_source["PLUCode"],
        errors="raise"
    )
    .astype("int64")
)


for integer_column in [
    "UnitSold",
    "NormalDemandUnits_Step4",
    "BulkDemandUnits_Step4",
    "TotalDemandUnits_Step4",
]:

    step5_source[integer_column] = (
        pd.to_numeric(
            step5_source[integer_column],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


step5_source[
    "BulkOrderFlag_Step4"
] = convert_csv_boolean(
    step5_source[
        "BulkOrderFlag_Step4"
    ]
)


step5_source[
    "ForecastingEligible_Step4"
] = convert_csv_boolean(
    step5_source[
        "ForecastingEligible_Step4"
    ]
)


# ------------------------------------------------------------
# Validate complete source
# ------------------------------------------------------------

assert len(step5_source) == 138_973


assert step5_source[
    "CanonicalProductID_Step3"
].nunique() == 228


assert int(
    step5_source[
        "UnitSold"
    ].sum()
) == 140_894


assert int(
    step5_source[
        "NormalDemandUnits_Step4"
    ].sum()
) == 138_922


assert int(
    step5_source[
        "BulkDemandUnits_Step4"
    ].sum()
) == 1_972


assert int(
    step5_source[
        "TotalDemandUnits_Step4"
    ].sum()
) == 140_894


assert int(
    step5_source[
        "BulkOrderFlag_Step4"
    ].sum()
) == 51


assert (
    step5_source[
        "NormalDemandUnits_Step4"
    ]
    +
    step5_source[
        "BulkDemandUnits_Step4"
    ]
    ==
    step5_source[
        "TotalDemandUnits_Step4"
    ]
).all()


assert (
    step5_source[
        "TotalDemandUnits_Step4"
    ]
    ==
    step5_source["UnitSold"]
).all()


# ------------------------------------------------------------
# Create forecastable working source
# ------------------------------------------------------------

forecastable_mask_step5 = (
    step5_source[
        "ForecastingEligible_Step4"
    ]
)


step5_forecastable_source = (
    step5_source.loc[
        forecastable_mask_step5
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(
    step5_forecastable_source
) == 114_237


assert step5_forecastable_source[
    "CanonicalProductID_Step3"
].nunique() == 227


assert step5_forecastable_source[
    "CanonicalProductID_Step3"
].notna().all()


assert step5_forecastable_source[
    "CanonicalProductName_Step3"
].notna().all()


assert int(
    step5_forecastable_source[
        "NormalDemandUnits_Step4"
    ].sum()
) == 114_186


assert int(
    step5_forecastable_source[
        "BulkDemandUnits_Step4"
    ].sum()
) == 1_972


assert int(
    step5_forecastable_source[
        "TotalDemandUnits_Step4"
    ].sum()
) == 116_158


# ------------------------------------------------------------
# Audit operating-date sets
# ------------------------------------------------------------

all_source_operating_dates = pd.DatetimeIndex(
    sorted(
        step5_source["Date"].unique()
    )
)


forecastable_operating_dates = pd.DatetimeIndex(
    sorted(
        step5_forecastable_source[
            "Date"
        ].unique()
    )
)


open_ul_only_dates = (
    all_source_operating_dates.difference(
        forecastable_operating_dates
    )
)


print("=" * 74)
print("STEP 5 INPUT VALIDATION PASSED")
print("=" * 74)
print()

print(
    "Complete source rows:",
    f"{len(step5_source):,}"
)

print(
    "Complete source units:",
    f"{int(step5_source['TotalDemandUnits_Step4'].sum()):,}"
)

print()

print(
    "Forecastable transaction rows:",
    f"{len(step5_forecastable_source):,}"
)

print(
    "Forecastable canonical products:",
    step5_forecastable_source[
        "CanonicalProductID_Step3"
    ].nunique()
)

print(
    "Forecastable normal units:",
    f"{int(step5_forecastable_source['NormalDemandUnits_Step4'].sum()):,}"
)

print(
    "Forecastable bulk units:",
    f"{int(step5_forecastable_source['BulkDemandUnits_Step4'].sum()):,}"
)

print(
    "Forecastable total units:",
    f"{int(step5_forecastable_source['TotalDemandUnits_Step4'].sum()):,}"
)

print()

print(
    "Complete-source operating dates:",
    len(all_source_operating_dates)
)

print(
    "Forecastable operating dates:",
    len(forecastable_operating_dates)
)

print(
    "Dates containing only excluded OPEN UL transactions:",
    len(open_ul_only_dates)
)

print(
    "Forecastable date range:",
    forecastable_operating_dates.min().date(),
    "to",
    forecastable_operating_dates.max().date()
)


if len(open_ul_only_dates) > 0:
    print()
    print("OPEN UL-only dates:")
    print(
        [
            date.strftime("%Y-%m-%d")
            for date in open_ul_only_dates
        ]
    )

STEP 5 INPUT VALIDATION PASSED

Complete source rows: 138,973
Complete source units: 140,894

Forecastable transaction rows: 114,237
Forecastable canonical products: 227
Forecastable normal units: 114,186
Forecastable bulk units: 1,972
Forecastable total units: 116,158

Complete-source operating dates: 245
Forecastable operating dates: 245
Dates containing only excluded OPEN UL transactions: 0
Forecastable date range: 2025-04-01 to 2026-03-30


In [81]:
# ============================================================
# Step 5 - Cell 2
# Create the official observed operating-date calendar
#
# Only dates on which Eden recorded transactions are included.
# Closed or unobserved calendar dates are not introduced.
# ============================================================

step5_operating_calendar = pd.DataFrame({
    "Date": forecastable_operating_dates
})


step5_operating_calendar = (
    step5_operating_calendar
    .sort_values("Date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Create stable calendar features
# ------------------------------------------------------------

step5_operating_calendar[
    "OperatingDaySequence"
] = np.arange(
    1,
    len(step5_operating_calendar) + 1
)


step5_operating_calendar[
    "Year"
] = step5_operating_calendar[
    "Date"
].dt.year.astype("int64")


step5_operating_calendar[
    "Month"
] = step5_operating_calendar[
    "Date"
].dt.month.astype("int64")


step5_operating_calendar[
    "MonthName"
] = step5_operating_calendar[
    "Date"
].dt.month_name()


step5_operating_calendar[
    "Quarter"
] = step5_operating_calendar[
    "Date"
].dt.quarter.astype("int64")


step5_operating_calendar[
    "DayOfWeekNumber"
] = step5_operating_calendar[
    "Date"
].dt.dayofweek.astype("int64")
# Monday = 0, Sunday = 6


step5_operating_calendar[
    "DayOfWeek"
] = step5_operating_calendar[
    "Date"
].dt.day_name()


iso_calendar = (
    step5_operating_calendar[
        "Date"
    ]
    .dt.isocalendar()
)


step5_operating_calendar[
    "ISOYear"
] = iso_calendar[
    "year"
].astype("int64")


step5_operating_calendar[
    "ISOWeek"
] = iso_calendar[
    "week"
].astype("int64")


step5_operating_calendar[
    "DayOfYear"
] = step5_operating_calendar[
    "Date"
].dt.dayofyear.astype("int64")


step5_operating_calendar[
    "IsWeekend"
] = (
    step5_operating_calendar[
        "DayOfWeekNumber"
    ]
    .ge(5)
)


# ------------------------------------------------------------
# Record gaps between observed operating dates
# ------------------------------------------------------------

step5_operating_calendar[
    "DaysSincePreviousOperatingDate"
] = (
    step5_operating_calendar[
        "Date"
    ]
    .diff()
    .dt.days
    .fillna(0)
    .astype("int64")
)


step5_operating_calendar[
    "IsConsecutiveCalendarDay"
] = (
    step5_operating_calendar[
        "DaysSincePreviousOperatingDate"
    ]
    .eq(1)
)


# First date has no previous operating date
step5_operating_calendar.loc[
    0,
    "IsConsecutiveCalendarDay"
] = False


# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert len(
    step5_operating_calendar
) == 245


assert step5_operating_calendar[
    "Date"
].is_unique


assert step5_operating_calendar[
    "Date"
].is_monotonic_increasing


assert step5_operating_calendar[
    "Date"
].min() == pd.Timestamp(
    "2025-04-01"
)


assert step5_operating_calendar[
    "Date"
].max() == pd.Timestamp(
    "2026-03-30"
)


assert step5_operating_calendar[
    "OperatingDaySequence"
].tolist() == list(
    range(1, 246)
)


assert step5_operating_calendar[
    "DayOfWeekNumber"
].between(
    0,
    6
).all()


assert step5_operating_calendar[
    "Month"
].between(
    1,
    12
).all()


assert set(
    step5_forecastable_source[
        "Date"
    ].unique()
) == set(
    step5_operating_calendar[
        "Date"
    ].unique()
)


# ------------------------------------------------------------
# Operating-day summary
# ------------------------------------------------------------

operating_day_weekday_summary = (
    step5_operating_calendar
    .groupby(
        [
            "DayOfWeekNumber",
            "DayOfWeek",
        ],
        as_index=False
    )
    .agg(
        OperatingDateCount=(
            "Date",
            "size"
        )
    )
    .sort_values(
        "DayOfWeekNumber"
    )
    .reset_index(drop=True)
)


print("=" * 74)
print("STEP 5 OPERATING CALENDAR CREATED")
print("=" * 74)
print()

print(
    "Observed operating dates:",
    f"{len(step5_operating_calendar):,}"
)

print(
    "First operating date:",
    step5_operating_calendar[
        "Date"
    ].min().date()
)

print(
    "Last operating date:",
    step5_operating_calendar[
        "Date"
    ].max().date()
)

print(
    "Weekend operating dates:",
    f"{int(step5_operating_calendar['IsWeekend'].sum()):,}"
)

print(
    "Largest gap between operating dates:",
    f"{int(step5_operating_calendar['DaysSincePreviousOperatingDate'].max()):,}",
    "calendar days"
)

print()
display(operating_day_weekday_summary)

print()
display(step5_operating_calendar.head(10))

STEP 5 OPERATING CALENDAR CREATED

Observed operating dates: 245
First operating date: 2025-04-01
Last operating date: 2026-03-30
Weekend operating dates: 1
Largest gap between operating dates: 13 calendar days



,DayOfWeekNumber,DayOfWeek,OperatingDateCount
0,0,Monday,45
1,1,Tuesday,50
2,2,Wednesday,50
3,3,Thursday,50
4,4,Friday,49
5,5,Saturday,1


,Date,OperatingDaySequence,Year,Month,MonthName,Quarter,DayOfWeekNumber,DayOfWeek,ISOYear,ISOWeek,DayOfYear,IsWeekend,DaysSincePreviousOperatingDate,IsConsecutiveCalendarDay
0,2025-04-01,1,2025,4,April,2,1,Tuesday,2025,14,91,False,0,False
1,2025-04-02,2,2025,4,April,2,2,Wednesday,2025,14,92,False,1,True
2,2025-04-03,3,2025,4,April,2,3,Thursday,2025,14,93,False,1,True
3,2025-04-04,4,2025,4,April,2,4,Friday,2025,14,94,False,1,True
4,2025-04-07,5,2025,4,April,2,0,Monday,2025,15,97,False,3,False
5,2025-04-08,6,2025,4,April,2,1,Tuesday,2025,15,98,False,1,True
6,2025-04-09,7,2025,4,April,2,2,Wednesday,2025,15,99,False,1,True
7,2025-04-10,8,2025,4,April,2,3,Thursday,2025,15,100,False,1,True
8,2025-04-11,9,2025,4,April,2,4,Friday,2025,15,101,False,1,True
9,2025-04-14,10,2025,4,April,2,0,Monday,2025,16,104,False,3,False


In [82]:
# ============================================================
# Step 5 - Cell 3
# Create one controlled metadata record for each forecastable
# canonical product
# ============================================================


# ------------------------------------------------------------
# Helper for consolidated text metadata
# ------------------------------------------------------------

def join_unique_text(values):
    cleaned = (
        values
        .astype("string")
        .str.strip()
        .dropna()
    )

    cleaned = cleaned.loc[
        cleaned.ne("")
    ]

    unique_values = sorted(
        cleaned.unique().tolist()
    )

    if not unique_values:
        return pd.NA

    return " | ".join(unique_values)


def join_unique_integer_codes(values):
    cleaned = pd.to_numeric(
        values,
        errors="raise"
    ).astype("int64")

    unique_values = sorted(
        cleaned.unique().tolist()
    )

    return ", ".join(
        str(value)
        for value in unique_values
    )


# ------------------------------------------------------------
# Validate canonical identity consistency
# ------------------------------------------------------------

canonical_identity_check = (
    step5_forecastable_source
    .groupby(
        "CanonicalProductID_Step3",
        as_index=False
    )
    .agg(
        CanonicalNameCount=(
            "CanonicalProductName_Step3",
            "nunique"
        )
    )
)


assert canonical_identity_check[
    "CanonicalNameCount"
].eq(1).all(), (
    "At least one canonical product ID has more than "
    "one canonical product name."
)


# ------------------------------------------------------------
# Create canonical product metadata
# ------------------------------------------------------------

step5_product_metadata = (
    step5_forecastable_source
    .groupby(
        "CanonicalProductID_Step3",
        as_index=False
    )
    .agg(
        CanonicalProductName=(
            "CanonicalProductName_Step3",
            "first"
        ),

        SourcePLUCount=(
            "PLUCode",
            "nunique"
        ),

        SourcePLUCodes=(
            "PLUCode",
            join_unique_integer_codes
        ),

        SourcePLUNames=(
            "PLUName_Corrected",
            join_unique_text
        ),

        SourceGroupCodes=(
            "GroupCode",
            join_unique_integer_codes
        ),

        SourceGroupNames=(
            "GroupName",
            join_unique_text
        ),

        MappingStatuses=(
            "MappingStatus",
            join_unique_text
        ),

        BeverageSeries=(
            "BeverageSeries_Step2",
            join_unique_text
        ),

        BeverageType=(
            "BeverageType_Step2",
            join_unique_text
        ),

        SupplierLabelsObserved=(
            "SupplierLabel_Step2",
            join_unique_text
        ),

        TierProductFamily=(
            "TierProductFamily_Step3",
            join_unique_text
        ),

        NominalPriceTier=(
            "NominalPriceTier_Step3",
            join_unique_text
        ),

        MenuGeneration=(
            "MenuGeneration_Step3",
            join_unique_text
        ),

        FirstObservedDate=(
            "Date",
            "min"
        ),

        LastObservedDate=(
            "Date",
            "max"
        ),

        ObservedOperatingDateCount=(
            "Date",
            "nunique"
        ),

        TransactionLineCount=(
            "TransactionID",
            "size"
        ),

        UniqueTransactionCount=(
            "TransactionID",
            "nunique"
        ),

        NormalDemandUnitsObserved=(
            "NormalDemandUnits_Step4",
            "sum"
        ),

        BulkDemandUnitsObserved=(
            "BulkDemandUnits_Step4",
            "sum"
        ),

        TotalDemandUnitsObserved=(
            "TotalDemandUnits_Step4",
            "sum"
        ),

        ConfirmedBulkLineCount=(
            "BulkOrderFlag_Step4",
            "sum"
        ),
    )
    .rename(
        columns={
            "CanonicalProductID_Step3":
                "CanonicalProductID"
        }
    )
)


# ------------------------------------------------------------
# Attach operating-day sequence for active ranges
# ------------------------------------------------------------

operating_sequence_lookup = (
    step5_operating_calendar
    .set_index("Date")[
        "OperatingDaySequence"
    ]
)


step5_product_metadata[
    "FirstOperatingDaySequence"
] = (
    step5_product_metadata[
        "FirstObservedDate"
    ]
    .map(operating_sequence_lookup)
)


step5_product_metadata[
    "LastOperatingDaySequence"
] = (
    step5_product_metadata[
        "LastObservedDate"
    ]
    .map(operating_sequence_lookup)
)


assert step5_product_metadata[
    "FirstOperatingDaySequence"
].notna().all()


assert step5_product_metadata[
    "LastOperatingDaySequence"
].notna().all()


step5_product_metadata[
    "FirstOperatingDaySequence"
] = (
    step5_product_metadata[
        "FirstOperatingDaySequence"
    ]
    .astype("int64")
)


step5_product_metadata[
    "LastOperatingDaySequence"
] = (
    step5_product_metadata[
        "LastOperatingDaySequence"
    ]
    .astype("int64")
)


# Number of confirmed operating dates between the product's
# first and last observed sale, inclusive.
step5_product_metadata[
    "ActiveOperatingDateCount"
] = (
    step5_product_metadata[
        "LastOperatingDaySequence"
    ]
    -
    step5_product_metadata[
        "FirstOperatingDaySequence"
    ]
    + 1
).astype("int64")


step5_product_metadata[
    "ZeroDemandOperatingDatesToAdd"
] = (
    step5_product_metadata[
        "ActiveOperatingDateCount"
    ]
    -
    step5_product_metadata[
        "ObservedOperatingDateCount"
    ]
).astype("int64")


step5_product_metadata[
    "IsMultiPLUCanonicalProduct"
] = (
    step5_product_metadata[
        "SourcePLUCount"
    ]
    .gt(1)
)


step5_product_metadata[
    "HasConfirmedBulkDemand"
] = (
    step5_product_metadata[
        "BulkDemandUnitsObserved"
    ]
    .gt(0)
)


step5_product_metadata[
    "ProductMetadataVersion"
] = (
    "STEP_5_CANONICAL_PRODUCT_METADATA_V1"
)


# ------------------------------------------------------------
# Sort metadata consistently
# ------------------------------------------------------------

step5_product_metadata = (
    step5_product_metadata
    .sort_values(
        [
            "CanonicalProductName",
            "CanonicalProductID",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert len(step5_product_metadata) == 227


assert step5_product_metadata[
    "CanonicalProductID"
].is_unique


assert step5_product_metadata[
    "CanonicalProductName"
].notna().all()


# 235 source PLUs remain after excluding OPEN UL.
assert int(
    step5_product_metadata[
        "SourcePLUCount"
    ].sum()
) == 235


assert int(
    step5_product_metadata[
        "TransactionLineCount"
    ].sum()
) == 114_237


assert int(
    step5_product_metadata[
        "NormalDemandUnitsObserved"
    ].sum()
) == 114_186


assert int(
    step5_product_metadata[
        "BulkDemandUnitsObserved"
    ].sum()
) == 1_972


assert int(
    step5_product_metadata[
        "TotalDemandUnitsObserved"
    ].sum()
) == 116_158


assert int(
    step5_product_metadata[
        "ConfirmedBulkLineCount"
    ].sum()
) == 51


assert int(
    step5_product_metadata[
        "HasConfirmedBulkDemand"
    ].sum()
) == 11


assert (
    step5_product_metadata[
        "NormalDemandUnitsObserved"
    ]
    +
    step5_product_metadata[
        "BulkDemandUnitsObserved"
    ]
    ==
    step5_product_metadata[
        "TotalDemandUnitsObserved"
    ]
).all()


assert (
    step5_product_metadata[
        "FirstObservedDate"
    ]
    <=
    step5_product_metadata[
        "LastObservedDate"
    ]
).all()


assert (
    step5_product_metadata[
        "ObservedOperatingDateCount"
    ]
    <=
    step5_product_metadata[
        "ActiveOperatingDateCount"
    ]
).all()


assert step5_product_metadata[
    "ZeroDemandOperatingDatesToAdd"
].ge(0).all()


expected_final_panel_rows = int(
    step5_product_metadata[
        "ActiveOperatingDateCount"
    ].sum()
)


expected_zero_demand_rows = int(
    step5_product_metadata[
        "ZeroDemandOperatingDatesToAdd"
    ].sum()
)


observed_product_date_rows = int(
    step5_product_metadata[
        "ObservedOperatingDateCount"
    ].sum()
)


assert (
    observed_product_date_rows
    +
    expected_zero_demand_rows
    ==
    expected_final_panel_rows
)


print("=" * 74)
print("STEP 5 CANONICAL PRODUCT METADATA CREATED")
print("=" * 74)
print()

print(
    "Forecastable canonical products:",
    f"{len(step5_product_metadata):,}"
)

print(
    "Forecastable source PLUs:",
    f"{int(step5_product_metadata['SourcePLUCount'].sum()):,}"
)

print(
    "Multi-PLU canonical products:",
    f"{int(step5_product_metadata['IsMultiPLUCanonicalProduct'].sum()):,}"
)

print(
    "Products with confirmed bulk demand:",
    f"{int(step5_product_metadata['HasConfirmedBulkDemand'].sum()):,}"
)

print()

print(
    "Observed product-date rows:",
    f"{observed_product_date_rows:,}"
)

print(
    "Zero-demand rows to add:",
    f"{expected_zero_demand_rows:,}"
)

print(
    "Expected final daily panel rows:",
    f"{expected_final_panel_rows:,}"
)

print()

print(
    "Normal demand units:",
    f"{int(step5_product_metadata['NormalDemandUnitsObserved'].sum()):,}"
)

print(
    "Bulk demand units:",
    f"{int(step5_product_metadata['BulkDemandUnitsObserved'].sum()):,}"
)

print(
    "Total demand units:",
    f"{int(step5_product_metadata['TotalDemandUnitsObserved'].sum()):,}"
)

print()

display(
    step5_product_metadata.head(15)
)

STEP 5 CANONICAL PRODUCT METADATA CREATED

Forecastable canonical products: 227
Forecastable source PLUs: 235
Multi-PLU canonical products: 7
Products with confirmed bulk demand: 11

Observed product-date rows: 15,138
Zero-demand rows to add: 10,267
Expected final daily panel rows: 25,405

Normal demand units: 114,186
Bulk demand units: 1,972
Total demand units: 116,158



,CanonicalProductID,CanonicalProductName,SourcePLUCount,SourcePLUCodes,SourcePLUNames,SourceGroupCodes,SourceGroupNames,MappingStatuses,BeverageSeries,BeverageType,...,BulkDemandUnitsObserved,TotalDemandUnitsObserved,ConfirmedBulkLineCount,FirstOperatingDaySequence,LastOperatingDaySequence,ActiveOperatingDateCount,ZeroDemandOperatingDatesToAdd,IsMultiPLUCanonicalProduct,HasConfirmedBulkDemand,ProductMetadataVersion
0,PLU_200000001,7UP CAN,1,200000001,7UP CAN,2,COLD BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,134,0,2,130,129,57,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
1,PLU_4241472,ADDITIONAL SYRUPS,1,4241472,ADDITIONAL SYRUPS,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,54,0,3,138,136,103,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
2,PLU_4241437,ALMOND CROISSANT,1,4241437,ALMOND CROISSANT,10,CAKES/PASTRIES,APPROVED_STEP_1,<NA>,<NA>,...,0,56,0,17,133,117,91,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
3,PLU_42552,ALTERNATIVE MILK,1,42552,ALTERNATIVE MILK,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,839,0,136,245,110,2,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
4,PLU_4241473,ALTERNATIVE MILK CT,1,4241473,ALTERNATIVE MILK CT,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,505,0,1,131,131,28,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
5,PLU_42597,AMERICANO,1,42597,AMERICANO,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,1135,0,137,245,109,0,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
6,PLU_4241452,AMERICANO 12 OZ,1,4241452,AMERICANO 12 OZ,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,1651,0,1,134,134,6,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
7,PLU_4241453,AMERICANO 16OZ,1,4241453,AMERICANO 16OZ,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,398,0,1,105,105,16,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
8,PLU_4251503,AMERICANO LG,1,4251503,AMERICANO LG,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,163,0,1,104,104,20,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1
9,PLU_4251502,AMERICANO MED,1,4251502,AMERICANO MED,1,HOT BEVS,APPROVED_STEP_1,<NA>,<NA>,...,0,310,0,1,107,107,17,False,False,STEP_5_CANONICAL_PRODUCT_METADATA_V1


In [83]:
# ============================================================
# Step 5 - Cell 4
# Aggregate transaction lines into observed daily demand
#
# One row per:
# Date + CanonicalProductID
#
# Zero-demand rows are not added in this cell.
# ============================================================

# Count confirmed bulk transaction lines during aggregation
step5_forecastable_source[
    "_ConfirmedBulkLineCount"
] = (
    step5_forecastable_source[
        "BulkOrderFlag_Step4"
    ]
    .astype("int64")
)


# ------------------------------------------------------------
# Aggregate observed demand
# ------------------------------------------------------------

step5_observed_daily = (
    step5_forecastable_source
    .groupby(
        [
            "Date",
            "CanonicalProductID_Step3",
            "CanonicalProductName_Step3",
        ],
        as_index=False
    )
    .agg(
        NormalDemand=(
            "NormalDemandUnits_Step4",
            "sum"
        ),

        BulkDemand=(
            "BulkDemandUnits_Step4",
            "sum"
        ),

        TotalDemand=(
            "TotalDemandUnits_Step4",
            "sum"
        ),

        TransactionLineCount=(
            "TransactionID",
            "size"
        ),

        UniqueTransactionCount=(
            "TransactionID",
            "nunique"
        ),

        SourcePLUCountObserved=(
            "PLUCode",
            "nunique"
        ),

        ConfirmedBulkLineCount=(
            "_ConfirmedBulkLineCount",
            "sum"
        ),
    )
    .rename(
        columns={
            "CanonicalProductID_Step3":
                "CanonicalProductID",

            "CanonicalProductName_Step3":
                "CanonicalProductName",
        }
    )
)


# ------------------------------------------------------------
# Convert demand and count columns to integers
# ------------------------------------------------------------

observed_integer_columns = [
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
    "TransactionLineCount",
    "UniqueTransactionCount",
    "SourcePLUCountObserved",
    "ConfirmedBulkLineCount",
]


for column in observed_integer_columns:

    step5_observed_daily[column] = (
        pd.to_numeric(
            step5_observed_daily[column],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


# ------------------------------------------------------------
# Add observed-row indicators
# ------------------------------------------------------------

step5_observed_daily[
    "IsObservedProductDate"
] = True


step5_observed_daily[
    "IsZeroDemandRow"
] = False


step5_observed_daily[
    "HasBulkDemand"
] = (
    step5_observed_daily[
        "BulkDemand"
    ]
    .gt(0)
)


step5_observed_daily[
    "DemandRecordSource"
] = "OBSERVED_TRANSACTION_AGGREGATION"


# ------------------------------------------------------------
# Sort consistently
# ------------------------------------------------------------

step5_observed_daily = (
    step5_observed_daily
    .sort_values(
        [
            "Date",
            "CanonicalProductID",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Strict validation
# ------------------------------------------------------------

assert len(step5_observed_daily) == 15_138


assert not step5_observed_daily.duplicated(
    [
        "Date",
        "CanonicalProductID",
    ]
).any(), (
    "Duplicate canonical product-date rows were created."
)


assert step5_observed_daily[
    "CanonicalProductID"
].nunique() == 227


assert step5_observed_daily[
    "Date"
].nunique() == 245


assert step5_observed_daily[
    "CanonicalProductID"
].notna().all()


assert step5_observed_daily[
    "CanonicalProductName"
].notna().all()


assert step5_observed_daily[
    "TotalDemand"
].gt(0).all(), (
    "An observed product-date row has zero total demand."
)


assert (
    step5_observed_daily[
        "NormalDemand"
    ]
    +
    step5_observed_daily[
        "BulkDemand"
    ]
    ==
    step5_observed_daily[
        "TotalDemand"
    ]
).all()


assert int(
    step5_observed_daily[
        "NormalDemand"
    ].sum()
) == 114_186


assert int(
    step5_observed_daily[
        "BulkDemand"
    ].sum()
) == 1_972


assert int(
    step5_observed_daily[
        "TotalDemand"
    ].sum()
) == 116_158


assert int(
    step5_observed_daily[
        "TransactionLineCount"
    ].sum()
) == 114_237


assert int(
    step5_observed_daily[
        "ConfirmedBulkLineCount"
    ].sum()
) == 51


assert int(
    step5_observed_daily[
        "HasBulkDemand"
    ].sum()
) <= 51


assert step5_observed_daily[
    "IsObservedProductDate"
].all()


assert not step5_observed_daily[
    "IsZeroDemandRow"
].any()


# Every observed date must be an approved operating date
assert set(
    step5_observed_daily[
        "Date"
    ].unique()
) == set(
    step5_operating_calendar[
        "Date"
    ].unique()
)


# Reconcile against product metadata
assert len(step5_observed_daily) == int(
    step5_product_metadata[
        "ObservedOperatingDateCount"
    ].sum()
)


observed_product_totals = (
    step5_observed_daily
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        ObservedDateCountFromDaily=(
            "Date",
            "nunique"
        ),

        NormalDemandFromDaily=(
            "NormalDemand",
            "sum"
        ),

        BulkDemandFromDaily=(
            "BulkDemand",
            "sum"
        ),

        TotalDemandFromDaily=(
            "TotalDemand",
            "sum"
        ),
    )
)


metadata_reconciliation = (
    step5_product_metadata[
        [
            "CanonicalProductID",
            "ObservedOperatingDateCount",
            "NormalDemandUnitsObserved",
            "BulkDemandUnitsObserved",
            "TotalDemandUnitsObserved",
        ]
    ]
    .merge(
        observed_product_totals,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)


assert metadata_reconciliation[
    "ObservedOperatingDateCount"
].eq(
    metadata_reconciliation[
        "ObservedDateCountFromDaily"
    ]
).all()


assert metadata_reconciliation[
    "NormalDemandUnitsObserved"
].eq(
    metadata_reconciliation[
        "NormalDemandFromDaily"
    ]
).all()


assert metadata_reconciliation[
    "BulkDemandUnitsObserved"
].eq(
    metadata_reconciliation[
        "BulkDemandFromDaily"
    ]
).all()


assert metadata_reconciliation[
    "TotalDemandUnitsObserved"
].eq(
    metadata_reconciliation[
        "TotalDemandFromDaily"
    ]
).all()


print("=" * 74)
print("STEP 5 OBSERVED DAILY DEMAND CREATED")
print("=" * 74)
print()

print(
    "Observed product-date rows:",
    f"{len(step5_observed_daily):,}"
)

print(
    "Canonical products:",
    step5_observed_daily[
        "CanonicalProductID"
    ].nunique()
)

print(
    "Operating dates represented:",
    step5_observed_daily[
        "Date"
    ].nunique()
)

print()

print(
    "Normal demand units:",
    f"{int(step5_observed_daily['NormalDemand'].sum()):,}"
)

print(
    "Bulk demand units:",
    f"{int(step5_observed_daily['BulkDemand'].sum()):,}"
)

print(
    "Total demand units:",
    f"{int(step5_observed_daily['TotalDemand'].sum()):,}"
)

print(
    "Confirmed bulk lines:",
    f"{int(step5_observed_daily['ConfirmedBulkLineCount'].sum()):,}"
)

print()

display(
    step5_observed_daily.head(15)
)

STEP 5 OBSERVED DAILY DEMAND CREATED

Observed product-date rows: 15,138
Canonical products: 227
Operating dates represented: 245

Normal demand units: 114,186
Bulk demand units: 1,972
Total demand units: 116,158
Confirmed bulk lines: 51



,Date,CanonicalProductID,CanonicalProductName,NormalDemand,BulkDemand,TotalDemand,TransactionLineCount,UniqueTransactionCount,SourcePLUCountObserved,ConfirmedBulkLineCount,IsObservedProductDate,IsZeroDemandRow,HasBulkDemand,DemandRecordSource
0,2025-04-01,BEV_BRANDED_AMERICANO,B-AMERICANO,13,0,13,13,13,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
1,2025-04-01,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,1,0,1,1,1,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
2,2025-04-01,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,1,0,1,1,1,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
3,2025-04-01,BEV_BRANDED_LATTE,B-LATTE,1,0,1,1,1,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
4,2025-04-01,BEV_BRANDED_MOCHA,B-MOCHA,1,0,1,1,1,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
5,2025-04-01,PLU_12304,KOMBUCHA,1,0,1,1,1,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
6,2025-04-01,PLU_125038,FILTER COFFEE SM,5,0,5,5,5,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
7,2025-04-01,PLU_2000000019,FULL FAT CAN,15,0,15,15,15,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
8,2025-04-01,PLU_2000000023,COKE ZERO 330ML,21,0,21,21,21,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION
9,2025-04-01,PLU_2000000027,DIET/ZERO COKE 330ML CAN,4,0,4,4,4,1,0,True,False,False,OBSERVED_TRANSACTION_AGGREGATION


In [84]:
# ============================================================
# Step 5 - Cell 5
# Create the final active product-date skeleton and merge
# observed demand into it
#
# Zero-demand rows are created only:
# 1. On confirmed operating dates
# 2. Between each product's first and last observed sale
# ============================================================


# ------------------------------------------------------------
# Select controlled product metadata for panel construction
# ------------------------------------------------------------

step5_panel_product_metadata = (
    step5_product_metadata[
        [
            "CanonicalProductID",
            "CanonicalProductName",
            "FirstObservedDate",
            "LastObservedDate",
            "FirstOperatingDaySequence",
            "LastOperatingDaySequence",
            "ActiveOperatingDateCount",
        ]
    ]
    .copy()
)


assert len(step5_panel_product_metadata) == 227

assert step5_panel_product_metadata[
    "CanonicalProductID"
].is_unique


# ------------------------------------------------------------
# Cross product metadata with the 245 operating dates
# ------------------------------------------------------------

step5_product_date_candidates = (
    step5_panel_product_metadata
    .merge(
        step5_operating_calendar,
        how="cross"
    )
)


assert len(
    step5_product_date_candidates
) == 227 * 245


# ------------------------------------------------------------
# Retain only each product's active operating-date range
# ------------------------------------------------------------

active_date_mask = (
    step5_product_date_candidates[
        "OperatingDaySequence"
    ]
    .ge(
        step5_product_date_candidates[
            "FirstOperatingDaySequence"
        ]
    )
    &
    step5_product_date_candidates[
        "OperatingDaySequence"
    ]
    .le(
        step5_product_date_candidates[
            "LastOperatingDaySequence"
        ]
    )
)


step5_active_product_date_skeleton = (
    step5_product_date_candidates.loc[
        active_date_mask
    ]
    .copy()
    .sort_values(
        [
            "CanonicalProductID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Validate active skeleton before adding demand
# ------------------------------------------------------------

assert len(
    step5_active_product_date_skeleton
) == 25_405


assert not step5_active_product_date_skeleton.duplicated(
    [
        "Date",
        "CanonicalProductID",
    ]
).any()


assert step5_active_product_date_skeleton[
    "CanonicalProductID"
].nunique() == 227


assert step5_active_product_date_skeleton[
    "Date"
].nunique() == 245


assert (
    step5_active_product_date_skeleton[
        "Date"
    ]
    >=
    step5_active_product_date_skeleton[
        "FirstObservedDate"
    ]
).all()


assert (
    step5_active_product_date_skeleton[
        "Date"
    ]
    <=
    step5_active_product_date_skeleton[
        "LastObservedDate"
    ]
).all()


# Confirm each product receives exactly its expected
# number of active operating dates.
skeleton_product_counts = (
    step5_active_product_date_skeleton
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        SkeletonRowCount=(
            "Date",
            "size"
        )
    )
)


skeleton_count_validation = (
    step5_product_metadata[
        [
            "CanonicalProductID",
            "ActiveOperatingDateCount",
        ]
    ]
    .merge(
        skeleton_product_counts,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)


assert skeleton_count_validation[
    "ActiveOperatingDateCount"
].eq(
    skeleton_count_validation[
        "SkeletonRowCount"
    ]
).all()


# ------------------------------------------------------------
# Prepare observed daily fields for merging
# ------------------------------------------------------------

observed_daily_merge_columns = [
    "Date",
    "CanonicalProductID",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
    "TransactionLineCount",
    "UniqueTransactionCount",
    "SourcePLUCountObserved",
    "ConfirmedBulkLineCount",
    "IsObservedProductDate",
    "HasBulkDemand",
    "DemandRecordSource",
]


step5_observed_daily_for_merge = (
    step5_observed_daily[
        observed_daily_merge_columns
    ]
    .copy()
)


# ------------------------------------------------------------
# Merge observed demand into the active skeleton
# ------------------------------------------------------------

step5_daily_panel = (
    step5_active_product_date_skeleton
    .merge(
        step5_observed_daily_for_merge,
        on=[
            "Date",
            "CanonicalProductID",
        ],
        how="left",
        validate="one_to_one",
        sort=False
    )
)


assert len(step5_daily_panel) == 25_405


assert not step5_daily_panel.duplicated(
    [
        "Date",
        "CanonicalProductID",
    ]
).any()


# A populated IsObservedProductDate value means that an
# observed transaction aggregation matched the skeleton row.
observed_match_mask = (
    step5_daily_panel[
        "IsObservedProductDate"
    ]
    .notna()
)


assert int(
    observed_match_mask.sum()
) == 15_138


assert int(
    (~observed_match_mask).sum()
) == 10_267


# ------------------------------------------------------------
# Fill demand/count fields for genuine zero-demand rows
# ------------------------------------------------------------

daily_integer_columns = [
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
    "TransactionLineCount",
    "UniqueTransactionCount",
    "SourcePLUCountObserved",
    "ConfirmedBulkLineCount",
]


for column in daily_integer_columns:

    step5_daily_panel[column] = (
        pd.to_numeric(
            step5_daily_panel[column],
            errors="coerce"
        )
        .fillna(0)
        .round()
        .astype("int64")
    )


step5_daily_panel[
    "IsObservedProductDate"
] = (
    observed_match_mask
    .astype(bool)
)


step5_daily_panel[
    "IsZeroDemandRow"
] = (
    ~observed_match_mask
).astype(bool)


step5_daily_panel[
    "HasBulkDemand"
] = (
    step5_daily_panel[
        "BulkDemand"
    ]
    .gt(0)
)


step5_daily_panel[
    "DemandRecordSource"
] = (
    step5_daily_panel[
        "DemandRecordSource"
    ]
    .astype("string")
    .fillna(
        "ZERO_FILLED_ACTIVE_OPERATING_DATE"
    )
)


# Product age measured in confirmed operating days.
# First observed sale date has age zero.
step5_daily_panel[
    "ProductAgeOperatingDays"
] = (
    step5_daily_panel[
        "OperatingDaySequence"
    ]
    -
    step5_daily_panel[
        "FirstOperatingDaySequence"
    ]
).astype("int64")


step5_daily_panel[
    "DailyPanelVersion"
] = (
    "STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1"
)


# ------------------------------------------------------------
# Sort into final product-date order
# ------------------------------------------------------------

step5_daily_panel = (
    step5_daily_panel
    .sort_values(
        [
            "CanonicalProductID",
            "Date",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Strict demand validation
# ------------------------------------------------------------

assert (
    step5_daily_panel[
        "NormalDemand"
    ]
    +
    step5_daily_panel[
        "BulkDemand"
    ]
    ==
    step5_daily_panel[
        "TotalDemand"
    ]
).all()


assert int(
    step5_daily_panel[
        "NormalDemand"
    ].sum()
) == 114_186


assert int(
    step5_daily_panel[
        "BulkDemand"
    ].sum()
) == 1_972


assert int(
    step5_daily_panel[
        "TotalDemand"
    ].sum()
) == 116_158


assert int(
    step5_daily_panel[
        "IsObservedProductDate"
    ].sum()
) == 15_138


assert int(
    step5_daily_panel[
        "IsZeroDemandRow"
    ].sum()
) == 10_267


assert (
    step5_daily_panel[
        "IsObservedProductDate"
    ]
    ^
    step5_daily_panel[
        "IsZeroDemandRow"
    ]
).all(), (
    "Every row must be either observed or zero-filled, "
    "but never both."
)


assert step5_daily_panel.loc[
    step5_daily_panel[
        "IsZeroDemandRow"
    ],
    [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
        "TransactionLineCount",
        "UniqueTransactionCount",
        "SourcePLUCountObserved",
        "ConfirmedBulkLineCount",
    ],
].eq(0).all().all()


assert step5_daily_panel.loc[
    step5_daily_panel[
        "IsObservedProductDate"
    ],
    "TotalDemand",
].gt(0).all()


assert step5_daily_panel.loc[
    step5_daily_panel[
        "IsZeroDemandRow"
    ],
    "DemandRecordSource",
].eq(
    "ZERO_FILLED_ACTIVE_OPERATING_DATE"
).all()


assert step5_daily_panel[
    "ProductAgeOperatingDays"
].ge(0).all()


# ------------------------------------------------------------
# Reconcile row counts per product with metadata
# ------------------------------------------------------------

final_product_panel_summary = (
    step5_daily_panel
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        FinalPanelRows=(
            "Date",
            "size"
        ),

        ObservedRows=(
            "IsObservedProductDate",
            "sum"
        ),

        ZeroDemandRows=(
            "IsZeroDemandRow",
            "sum"
        ),

        NormalDemandFromPanel=(
            "NormalDemand",
            "sum"
        ),

        BulkDemandFromPanel=(
            "BulkDemand",
            "sum"
        ),

        TotalDemandFromPanel=(
            "TotalDemand",
            "sum"
        ),
    )
)


final_product_reconciliation = (
    step5_product_metadata[
        [
            "CanonicalProductID",
            "ActiveOperatingDateCount",
            "ObservedOperatingDateCount",
            "ZeroDemandOperatingDatesToAdd",
            "NormalDemandUnitsObserved",
            "BulkDemandUnitsObserved",
            "TotalDemandUnitsObserved",
        ]
    ]
    .merge(
        final_product_panel_summary,
        on="CanonicalProductID",
        how="left",
        validate="one_to_one"
    )
)


assert final_product_reconciliation[
    "ActiveOperatingDateCount"
].eq(
    final_product_reconciliation[
        "FinalPanelRows"
    ]
).all()


assert final_product_reconciliation[
    "ObservedOperatingDateCount"
].eq(
    final_product_reconciliation[
        "ObservedRows"
    ]
).all()


assert final_product_reconciliation[
    "ZeroDemandOperatingDatesToAdd"
].eq(
    final_product_reconciliation[
        "ZeroDemandRows"
    ]
).all()


assert final_product_reconciliation[
    "NormalDemandUnitsObserved"
].eq(
    final_product_reconciliation[
        "NormalDemandFromPanel"
    ]
).all()


assert final_product_reconciliation[
    "BulkDemandUnitsObserved"
].eq(
    final_product_reconciliation[
        "BulkDemandFromPanel"
    ]
).all()


assert final_product_reconciliation[
    "TotalDemandUnitsObserved"
].eq(
    final_product_reconciliation[
        "TotalDemandFromPanel"
    ]
).all()


print("=" * 74)
print("STEP 5 ZERO-FILLED ACTIVE DAILY PANEL CREATED")
print("=" * 74)
print()

print(
    "Final product-date rows:",
    f"{len(step5_daily_panel):,}"
)

print(
    "Canonical products:",
    step5_daily_panel[
        "CanonicalProductID"
    ].nunique()
)

print(
    "Operating dates represented:",
    step5_daily_panel[
        "Date"
    ].nunique()
)

print()

print(
    "Observed product-date rows:",
    f"{int(step5_daily_panel['IsObservedProductDate'].sum()):,}"
)

print(
    "Zero-demand rows added:",
    f"{int(step5_daily_panel['IsZeroDemandRow'].sum()):,}"
)

print()

print(
    "Normal demand units:",
    f"{int(step5_daily_panel['NormalDemand'].sum()):,}"
)

print(
    "Bulk demand units:",
    f"{int(step5_daily_panel['BulkDemand'].sum()):,}"
)

print(
    "Total demand units:",
    f"{int(step5_daily_panel['TotalDemand'].sum()):,}"
)

print()

display(
    step5_daily_panel.head(20)
)

STEP 5 ZERO-FILLED ACTIVE DAILY PANEL CREATED

Final product-date rows: 25,405
Canonical products: 227
Operating dates represented: 245

Observed product-date rows: 15,138
Zero-demand rows added: 10,267

Normal demand units: 114,186
Bulk demand units: 1,972
Total demand units: 116,158



,CanonicalProductID,CanonicalProductName,FirstObservedDate,LastObservedDate,FirstOperatingDaySequence,LastOperatingDaySequence,ActiveOperatingDateCount,Date,OperatingDaySequence,Year,...,TransactionLineCount,UniqueTransactionCount,SourcePLUCountObserved,ConfirmedBulkLineCount,IsObservedProductDate,HasBulkDemand,DemandRecordSource,IsZeroDemandRow,ProductAgeOperatingDays,DailyPanelVersion
0,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-01,1,2025,...,13,13,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,0,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
1,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-02,2,2025,...,14,13,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,1,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
2,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-03,3,2025,...,14,14,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,2,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
3,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-04,4,2025,...,7,7,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,3,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
4,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-07,5,2025,...,8,8,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,4,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
5,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-08,6,2025,...,16,15,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,5,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
6,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-09,7,2025,...,9,9,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,6,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
7,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-10,8,2025,...,5,5,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,7,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
8,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-11,9,2025,...,5,5,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,8,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1
9,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,2026-03-30,1,245,245,2025-04-14,10,2025,...,9,9,1,0,True,False,OBSERVED_TRANSACTION_AGGREGATION,False,9,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1


In [85]:
# ============================================================
# Step 5 - Cell 6
# Create the final forecasting-safe daily dataset
#
# Excludes same-day observational fields that could cause
# leakage, such as transaction counts and HasBulkDemand.
# ============================================================


# ------------------------------------------------------------
# Select static canonical-product metadata
# ------------------------------------------------------------

final_static_metadata_columns = [
    "CanonicalProductID",
    "SourcePLUCount",
    "SourcePLUCodes",
    "SourcePLUNames",
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
    "IsMultiPLUCanonicalProduct",
    "ProductMetadataVersion",
]


step5_static_product_metadata = (
    step5_product_metadata[
        final_static_metadata_columns
    ]
    .copy()
)


assert len(step5_static_product_metadata) == 227

assert step5_static_product_metadata[
    "CanonicalProductID"
].is_unique


# ------------------------------------------------------------
# Attach static metadata to the completed daily panel
# ------------------------------------------------------------

step5_final_daily_working = (
    step5_daily_panel
    .rename(
        columns={
            "FirstObservedDate":
                "ProductFirstObservedDate"
        }
    )
    .merge(
        step5_static_product_metadata,
        on="CanonicalProductID",
        how="left",
        validate="many_to_one",
        sort=False
    )
)


assert len(step5_final_daily_working) == 25_405


# ------------------------------------------------------------
# Define the final controlled column order
# ------------------------------------------------------------

step5_final_columns = [
    # Identity
    "Date",
    "CanonicalProductID",
    "CanonicalProductName",

    # Product lifecycle
    "ProductFirstObservedDate",
    "ProductAgeOperatingDays",

    # Product-source metadata
    "SourcePLUCount",
    "SourcePLUCodes",
    "SourcePLUNames",
    "SourceGroupCodes",
    "SourceGroupNames",
    "BeverageSeries",
    "BeverageType",
    "SupplierLabelsObserved",
    "TierProductFamily",
    "NominalPriceTier",
    "MenuGeneration",
    "IsMultiPLUCanonicalProduct",

    # Operating-date and calendar features
    "OperatingDaySequence",
    "Year",
    "Month",
    "MonthName",
    "Quarter",
    "DayOfWeekNumber",
    "DayOfWeek",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "IsWeekend",
    "DaysSincePreviousOperatingDate",
    "IsConsecutiveCalendarDay",

    # Forecasting targets
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",

    # Construction/audit indicators
    "IsObservedProductDate",
    "IsZeroDemandRow",
    "DemandRecordSource",

    # Dataset versions
    "DailyPanelVersion",
    "ProductMetadataVersion",
]


missing_final_columns = [
    column
    for column in step5_final_columns
    if column not in step5_final_daily_working.columns
]


if missing_final_columns:
    raise ValueError(
        "Final daily dataset is missing required columns:\n"
        f"{missing_final_columns}"
    )


step5_final_daily = (
    step5_final_daily_working[
        step5_final_columns
    ]
    .copy()
    .sort_values(
        [
            "Date",
            "CanonicalProductID",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Confirm leakage-prone same-day fields were excluded
# ------------------------------------------------------------

excluded_same_day_fields = [
    "TransactionLineCount",
    "UniqueTransactionCount",
    "SourcePLUCountObserved",
    "ConfirmedBulkLineCount",
    "HasBulkDemand",
    "LastObservedDate",
    "LastOperatingDaySequence",
]


unexpected_fields = [
    column
    for column in excluded_same_day_fields
    if column in step5_final_daily.columns
]


assert not unexpected_fields, (
    "Leakage-prone or retrospective fields remain in the "
    f"final daily dataset: {unexpected_fields}"
)


# ------------------------------------------------------------
# Standardise final data types
# ------------------------------------------------------------

step5_final_daily["Date"] = pd.to_datetime(
    step5_final_daily["Date"],
    errors="raise"
)


step5_final_daily[
    "ProductFirstObservedDate"
] = pd.to_datetime(
    step5_final_daily[
        "ProductFirstObservedDate"
    ],
    errors="raise"
)


final_integer_columns = [
    "ProductAgeOperatingDays",
    "SourcePLUCount",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "DaysSincePreviousOperatingDate",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
]


for column in final_integer_columns:

    step5_final_daily[column] = (
        pd.to_numeric(
            step5_final_daily[column],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


final_boolean_columns = [
    "IsMultiPLUCanonicalProduct",
    "IsWeekend",
    "IsConsecutiveCalendarDay",
    "IsObservedProductDate",
    "IsZeroDemandRow",
]


for column in final_boolean_columns:

    step5_final_daily[column] = (
        step5_final_daily[column]
        .astype(bool)
    )


# ------------------------------------------------------------
# Core structural validation
# ------------------------------------------------------------

assert len(step5_final_daily) == 25_405


assert not step5_final_daily.duplicated(
    [
        "Date",
        "CanonicalProductID",
    ]
).any()


assert step5_final_daily[
    "CanonicalProductID"
].nunique() == 227


assert step5_final_daily[
    "Date"
].nunique() == 245


assert step5_final_daily[
    "Date"
].min() == pd.Timestamp(
    "2025-04-01"
)


assert step5_final_daily[
    "Date"
].max() == pd.Timestamp(
    "2026-03-30"
)


assert step5_final_daily[
    "CanonicalProductID"
].notna().all()


assert step5_final_daily[
    "CanonicalProductName"
].notna().all()


assert step5_final_daily[
    "ProductFirstObservedDate"
].notna().all()


# OPEN UL must not appear
assert not step5_final_daily[
    "CanonicalProductName"
].astype("string").str.strip().str.fullmatch(
    r"OPEN\s+UL",
    case=False,
    na=False
).any()


# ------------------------------------------------------------
# Demand validation
# ------------------------------------------------------------

assert step5_final_daily[
    [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    ]
].ge(0).all().all()


assert (
    step5_final_daily[
        "NormalDemand"
    ]
    +
    step5_final_daily[
        "BulkDemand"
    ]
    ==
    step5_final_daily[
        "TotalDemand"
    ]
).all()


assert int(
    step5_final_daily[
        "NormalDemand"
    ].sum()
) == 114_186


assert int(
    step5_final_daily[
        "BulkDemand"
    ].sum()
) == 1_972


assert int(
    step5_final_daily[
        "TotalDemand"
    ].sum()
) == 116_158


assert int(
    step5_final_daily[
        "IsObservedProductDate"
    ].sum()
) == 15_138


assert int(
    step5_final_daily[
        "IsZeroDemandRow"
    ].sum()
) == 10_267


assert (
    step5_final_daily[
        "IsObservedProductDate"
    ]
    ^
    step5_final_daily[
        "IsZeroDemandRow"
    ]
).all()


assert step5_final_daily.loc[
    step5_final_daily[
        "IsZeroDemandRow"
    ],
    [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    ],
].eq(0).all().all()


assert step5_final_daily.loc[
    step5_final_daily[
        "IsObservedProductDate"
    ],
    "TotalDemand",
].gt(0).all()


# ------------------------------------------------------------
# Product lifecycle validation
# ------------------------------------------------------------

product_age_validation = (
    step5_final_daily
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        FirstPanelDate=(
            "Date",
            "min"
        ),
        ProductFirstObservedDate=(
            "ProductFirstObservedDate",
            "first"
        ),
        MinimumProductAge=(
            "ProductAgeOperatingDays",
            "min"
        ),
        MaximumProductAge=(
            "ProductAgeOperatingDays",
            "max"
        ),
        UniqueProductAgeCount=(
            "ProductAgeOperatingDays",
            "nunique"
        ),
        ProductPanelRows=(
            "Date",
            "size"
        ),
    )
)


assert product_age_validation[
    "FirstPanelDate"
].eq(
    product_age_validation[
        "ProductFirstObservedDate"
    ]
).all()


assert product_age_validation[
    "MinimumProductAge"
].eq(0).all()


assert product_age_validation[
    "MaximumProductAge"
].eq(
    product_age_validation[
        "ProductPanelRows"
    ]
    - 1
).all()


assert product_age_validation[
    "UniqueProductAgeCount"
].eq(
    product_age_validation[
        "ProductPanelRows"
    ]
).all()


# ------------------------------------------------------------
# Static metadata validation
# ------------------------------------------------------------

final_unique_product_metadata = (
    step5_final_daily[
        [
            "CanonicalProductID",
            "SourcePLUCount",
            "IsMultiPLUCanonicalProduct",
        ]
    ]
    .drop_duplicates()
)


assert len(final_unique_product_metadata) == 227


assert int(
    final_unique_product_metadata[
        "SourcePLUCount"
    ].sum()
) == 235


assert int(
    final_unique_product_metadata[
        "IsMultiPLUCanonicalProduct"
    ].sum()
) == 7


print("=" * 74)
print("STEP 5 FINAL FORECASTING SCHEMA CREATED")
print("=" * 74)
print()

print(
    "Final rows:",
    f"{len(step5_final_daily):,}"
)

print(
    "Final columns:",
    f"{len(step5_final_daily.columns):,}"
)

print(
    "Canonical products:",
    step5_final_daily[
        "CanonicalProductID"
    ].nunique()
)

print(
    "Operating dates:",
    step5_final_daily[
        "Date"
    ].nunique()
)

print()

print(
    "Observed rows:",
    f"{int(step5_final_daily['IsObservedProductDate'].sum()):,}"
)

print(
    "Zero-demand rows:",
    f"{int(step5_final_daily['IsZeroDemandRow'].sum()):,}"
)

print()

print(
    "Normal demand:",
    f"{int(step5_final_daily['NormalDemand'].sum()):,}"
)

print(
    "Bulk demand:",
    f"{int(step5_final_daily['BulkDemand'].sum()):,}"
)

print(
    "Total demand:",
    f"{int(step5_final_daily['TotalDemand'].sum()):,}"
)

print()

print("Final column order:")
print(step5_final_daily.columns.tolist())

print()

display(
    step5_final_daily.head(15)
)

STEP 5 FINAL FORECASTING SCHEMA CREATED

Final rows: 25,405
Final columns: 38
Canonical products: 227
Operating dates: 245

Observed rows: 15,138
Zero-demand rows: 10,267

Normal demand: 114,186
Bulk demand: 1,972
Total demand: 116,158

Final column order:
['Date', 'CanonicalProductID', 'CanonicalProductName', 'ProductFirstObservedDate', 'ProductAgeOperatingDays', 'SourcePLUCount', 'SourcePLUCodes', 'SourcePLUNames', 'SourceGroupCodes', 'SourceGroupNames', 'BeverageSeries', 'BeverageType', 'SupplierLabelsObserved', 'TierProductFamily', 'NominalPriceTier', 'MenuGeneration', 'IsMultiPLUCanonicalProduct', 'OperatingDaySequence', 'Year', 'Month', 'MonthName', 'Quarter', 'DayOfWeekNumber', 'DayOfWeek', 'ISOYear', 'ISOWeek', 'DayOfYear', 'IsWeekend', 'DaysSincePreviousOperatingDate', 'IsConsecutiveCalendarDay', 'NormalDemand', 'BulkDemand', 'TotalDemand', 'IsObservedProductDate', 'IsZeroDemandRow', 'DemandRecordSource', 'DailyPanelVersion', 'ProductMetadataVersion']



,Date,CanonicalProductID,CanonicalProductName,ProductFirstObservedDate,ProductAgeOperatingDays,SourcePLUCount,SourcePLUCodes,SourcePLUNames,SourceGroupCodes,SourceGroupNames,...,DaysSincePreviousOperatingDate,IsConsecutiveCalendarDay,NormalDemand,BulkDemand,TotalDemand,IsObservedProductDate,IsZeroDemandRow,DemandRecordSource,DailyPanelVersion,ProductMetadataVersion
0,2025-04-01,BEV_BRANDED_AMERICANO,B-AMERICANO,2025-04-01,0,3,"44381, 425531, 42527101",BEWLEYS AMERICANO | RN AMERICANO | ROASTED NOT...,1,HOT BEVS,...,0,False,13,0,13,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
1,2025-04-01,BEV_BRANDED_CAPPUCCINO,B-CAPPUCCINO,2025-04-01,0,2,"44382, 42527103",BEWLEYS CAPPUCCINO | RN CAPPUCCINO,1,HOT BEVS,...,0,False,1,0,1,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
2,2025-04-01,BEV_BRANDED_FLAT_WHITE,B-FLAT WHITE,2025-04-01,0,2,"417173, 42527104",BEWLEYS FLAT WHITE | RN FLAT WHITE,1,HOT BEVS,...,0,False,1,0,1,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
3,2025-04-01,BEV_BRANDED_LATTE,B-LATTE,2025-04-01,0,2,"44384, 42527102",BEWLEYS LATTE | RN LATTE,1,HOT BEVS,...,0,False,1,0,1,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
4,2025-04-01,BEV_BRANDED_MOCHA,B-MOCHA,2025-04-01,0,2,"44385, 42527105",BEWLEYS MOCHA | RN MOCHA,1,HOT BEVS,...,0,False,1,0,1,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
5,2025-04-01,PLU_12304,KOMBUCHA,2025-04-01,0,1,12304,KOMBUCHA,2,COLD BEVS,...,0,False,1,0,1,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
6,2025-04-01,PLU_125038,FILTER COFFEE SM,2025-04-01,0,1,125038,FILTER COFFEE SM,1,HOT BEVS,...,0,False,5,0,5,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
7,2025-04-01,PLU_2000000019,FULL FAT CAN,2025-04-01,0,1,2000000019,FULL FAT CAN,2,COLD BEVS,...,0,False,15,0,15,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
8,2025-04-01,PLU_2000000023,COKE ZERO 330ML,2025-04-01,0,1,2000000023,COKE ZERO 330ML,2,COLD BEVS,...,0,False,21,0,21,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1
9,2025-04-01,PLU_2000000027,DIET/ZERO COKE 330ML CAN,2025-04-01,0,1,2000000027,DIET/ZERO COKE 330ML CAN,2,COLD BEVS,...,0,False,4,0,4,True,False,OBSERVED_TRANSACTION_AGGREGATION,STEP_5_ACTIVE_OPERATING_DATE_PANEL_V1,STEP_5_CANONICAL_PRODUCT_METADATA_V1


In [86]:
# ============================================================
# Step 5 - Cell 7
# Save all final Step 5 outputs, reload the principal
# forecasting dataset, and perform permanent validation
# ============================================================


# ------------------------------------------------------------
# Create final validation summary
# ------------------------------------------------------------

step5_validation_summary = pd.DataFrame([
    {
        "DatasetVersion":
            "UL_EDEN_CANONICAL_PRODUCT_DAILY_FINAL_V1",

        "FinalRows":
            len(step5_final_daily),

        "FinalColumns":
            len(step5_final_daily.columns),

        "CanonicalProducts":
            step5_final_daily[
                "CanonicalProductID"
            ].nunique(),

        "ForecastableSourcePLUs":
            int(
                step5_product_metadata[
                    "SourcePLUCount"
                ].sum()
            ),

        "OperatingDates":
            step5_final_daily[
                "Date"
            ].nunique(),

        "FirstOperatingDate":
            step5_final_daily[
                "Date"
            ].min(),

        "LastOperatingDate":
            step5_final_daily[
                "Date"
            ].max(),

        "ObservedProductDateRows":
            int(
                step5_final_daily[
                    "IsObservedProductDate"
                ].sum()
            ),

        "ZeroDemandRows":
            int(
                step5_final_daily[
                    "IsZeroDemandRow"
                ].sum()
            ),

        "NormalDemandUnits":
            int(
                step5_final_daily[
                    "NormalDemand"
                ].sum()
            ),

        "BulkDemandUnits":
            int(
                step5_final_daily[
                    "BulkDemand"
                ].sum()
            ),

        "TotalDemandUnits":
            int(
                step5_final_daily[
                    "TotalDemand"
                ].sum()
            ),

        "ProductsWithConfirmedBulkDemand":
            int(
                step5_product_metadata[
                    "HasConfirmedBulkDemand"
                ].sum()
            ),

        "MultiPLUCanonicalProducts":
            int(
                step5_product_metadata[
                    "IsMultiPLUCanonicalProduct"
                ].sum()
            ),

        "WeekendOperatingDates":
            int(
                step5_operating_calendar[
                    "IsWeekend"
                ].sum()
            ),

        "LargestOperatingDateGap":
            int(
                step5_operating_calendar[
                    "DaysSincePreviousOperatingDate"
                ].max()
            ),

        "ExcludedProduct":
            "OPEN UL",

        "PanelConstructionRule":
            (
                "Zero demand added only on observed Eden "
                "operating dates between each product's "
                "first and last observed sale."
            ),
    }
])


# ------------------------------------------------------------
# Validate summary before saving
# ------------------------------------------------------------

assert int(
    step5_validation_summary[
        "FinalRows"
    ].iloc[0]
) == 25_405


assert int(
    step5_validation_summary[
        "FinalColumns"
    ].iloc[0]
) == 38


assert int(
    step5_validation_summary[
        "CanonicalProducts"
    ].iloc[0]
) == 227


assert int(
    step5_validation_summary[
        "OperatingDates"
    ].iloc[0]
) == 245


assert int(
    step5_validation_summary[
        "NormalDemandUnits"
    ].iloc[0]
) == 114_186


assert int(
    step5_validation_summary[
        "BulkDemandUnits"
    ].iloc[0]
) == 1_972


assert int(
    step5_validation_summary[
        "TotalDemandUnits"
    ].iloc[0]
) == 116_158


# ------------------------------------------------------------
# Save all five Step 5 outputs
# ------------------------------------------------------------

step5_final_daily.to_csv(
    STEP5_FINAL_DAILY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


step5_product_metadata.to_csv(
    STEP5_PRODUCT_METADATA_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


step5_operating_calendar.to_csv(
    STEP5_OPERATING_CALENDAR_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


step5_observed_daily.to_csv(
    STEP5_OBSERVED_DAILY_AUDIT_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


step5_validation_summary.to_csv(
    STEP5_VALIDATION_SUMMARY_OUTPUT_FILE,
    index=False,
    date_format="%Y-%m-%d"
)


for saved_file in [
    STEP5_FINAL_DAILY_OUTPUT_FILE,
    STEP5_PRODUCT_METADATA_OUTPUT_FILE,
    STEP5_OPERATING_CALENDAR_OUTPUT_FILE,
    STEP5_OBSERVED_DAILY_AUDIT_OUTPUT_FILE,
    STEP5_VALIDATION_SUMMARY_OUTPUT_FILE,
]:

    assert saved_file.exists(), (
        f"Expected output file was not created:\n"
        f"{saved_file}"
    )


# ------------------------------------------------------------
# Reload principal final daily dataset
# ------------------------------------------------------------

saved_step5_final = pd.read_csv(
    STEP5_FINAL_DAILY_OUTPUT_FILE,
    low_memory=False,
    parse_dates=[
        "Date",
        "ProductFirstObservedDate",
    ]
)


# ------------------------------------------------------------
# Confirm exact saved schema
# ------------------------------------------------------------

assert saved_step5_final.columns.tolist() == (
    step5_final_columns
), (
    "Saved column order does not match the approved "
    "Step 5 schema."
)


assert len(saved_step5_final.columns) == 38


# ------------------------------------------------------------
# Restore numeric data types
# ------------------------------------------------------------

saved_integer_columns = [
    "ProductAgeOperatingDays",
    "SourcePLUCount",
    "OperatingDaySequence",
    "Year",
    "Month",
    "Quarter",
    "DayOfWeekNumber",
    "ISOYear",
    "ISOWeek",
    "DayOfYear",
    "DaysSincePreviousOperatingDate",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
]


for column in saved_integer_columns:

    saved_step5_final[column] = (
        pd.to_numeric(
            saved_step5_final[column],
            errors="raise"
        )
        .round()
        .astype("int64")
    )


# ------------------------------------------------------------
# Restore Boolean data types
# ------------------------------------------------------------

saved_boolean_columns = [
    "IsMultiPLUCanonicalProduct",
    "IsWeekend",
    "IsConsecutiveCalendarDay",
    "IsObservedProductDate",
    "IsZeroDemandRow",
]


for column in saved_boolean_columns:

    saved_step5_final[column] = (
        convert_csv_boolean(
            saved_step5_final[column]
        )
    )


# ------------------------------------------------------------
# Permanent structural validation
# ------------------------------------------------------------

assert len(saved_step5_final) == 25_405


assert saved_step5_final[
    "CanonicalProductID"
].nunique() == 227


assert saved_step5_final[
    "Date"
].nunique() == 245


assert saved_step5_final[
    "Date"
].min() == pd.Timestamp(
    "2025-04-01"
)


assert saved_step5_final[
    "Date"
].max() == pd.Timestamp(
    "2026-03-30"
)


assert not saved_step5_final.duplicated(
    [
        "Date",
        "CanonicalProductID",
    ]
).any()


assert saved_step5_final[
    "CanonicalProductID"
].notna().all()


assert saved_step5_final[
    "CanonicalProductName"
].notna().all()


assert saved_step5_final[
    "ProductFirstObservedDate"
].notna().all()


# ------------------------------------------------------------
# Permanent exclusion validation
# ------------------------------------------------------------

assert not saved_step5_final[
    "CanonicalProductName"
].astype("string").str.strip().str.fullmatch(
    r"OPEN\s+UL",
    case=False,
    na=False
).any()


# ------------------------------------------------------------
# Permanent demand validation
# ------------------------------------------------------------

assert saved_step5_final[
    [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    ]
].ge(0).all().all()


assert (
    saved_step5_final[
        "NormalDemand"
    ]
    +
    saved_step5_final[
        "BulkDemand"
    ]
    ==
    saved_step5_final[
        "TotalDemand"
    ]
).all()


assert int(
    saved_step5_final[
        "NormalDemand"
    ].sum()
) == 114_186


assert int(
    saved_step5_final[
        "BulkDemand"
    ].sum()
) == 1_972


assert int(
    saved_step5_final[
        "TotalDemand"
    ].sum()
) == 116_158


assert int(
    saved_step5_final[
        "IsObservedProductDate"
    ].sum()
) == 15_138


assert int(
    saved_step5_final[
        "IsZeroDemandRow"
    ].sum()
) == 10_267


assert (
    saved_step5_final[
        "IsObservedProductDate"
    ]
    ^
    saved_step5_final[
        "IsZeroDemandRow"
    ]
).all()


assert saved_step5_final.loc[
    saved_step5_final[
        "IsZeroDemandRow"
    ],
    [
        "NormalDemand",
        "BulkDemand",
        "TotalDemand",
    ],
].eq(0).all().all()


assert saved_step5_final.loc[
    saved_step5_final[
        "IsObservedProductDate"
    ],
    "TotalDemand",
].gt(0).all()


# ------------------------------------------------------------
# Validate saved product lifecycle sequences
# ------------------------------------------------------------

saved_lifecycle_validation = (
    saved_step5_final
    .groupby(
        "CanonicalProductID",
        as_index=False
    )
    .agg(
        FirstPanelDate=(
            "Date",
            "min"
        ),

        FirstObservedDate=(
            "ProductFirstObservedDate",
            "first"
        ),

        MinimumProductAge=(
            "ProductAgeOperatingDays",
            "min"
        ),

        MaximumProductAge=(
            "ProductAgeOperatingDays",
            "max"
        ),

        ProductPanelRows=(
            "Date",
            "size"
        ),

        UniqueProductAgeValues=(
            "ProductAgeOperatingDays",
            "nunique"
        ),
    )
)


assert saved_lifecycle_validation[
    "FirstPanelDate"
].eq(
    saved_lifecycle_validation[
        "FirstObservedDate"
    ]
).all()


assert saved_lifecycle_validation[
    "MinimumProductAge"
].eq(0).all()


assert saved_lifecycle_validation[
    "MaximumProductAge"
].eq(
    saved_lifecycle_validation[
        "ProductPanelRows"
    ]
    - 1
).all()


assert saved_lifecycle_validation[
    "UniqueProductAgeValues"
].eq(
    saved_lifecycle_validation[
        "ProductPanelRows"
    ]
).all()


# ------------------------------------------------------------
# Compare saved key targets with the in-memory dataset
# ------------------------------------------------------------

comparison_columns = [
    "Date",
    "CanonicalProductID",
    "NormalDemand",
    "BulkDemand",
    "TotalDemand",
    "IsObservedProductDate",
    "IsZeroDemandRow",
]


in_memory_comparison = (
    step5_final_daily[
        comparison_columns
    ]
    .reset_index(drop=True)
)


saved_comparison = (
    saved_step5_final[
        comparison_columns
    ]
    .reset_index(drop=True)
)


pd.testing.assert_frame_equal(
    in_memory_comparison,
    saved_comparison,
    check_dtype=False,
    check_exact=True,
)


# ------------------------------------------------------------
# Final completion output
# ------------------------------------------------------------

print("=" * 78)
print("STEP 5 FINAL PRODUCT-LEVEL DAILY FORECASTING DATASET COMPLETED")
print("=" * 78)
print()

print(
    "Final rows:",
    f"{len(saved_step5_final):,}"
)

print(
    "Final columns:",
    f"{len(saved_step5_final.columns):,}"
)

print(
    "Canonical products:",
    saved_step5_final[
        "CanonicalProductID"
    ].nunique()
)

print(
    "Observed operating dates:",
    saved_step5_final[
        "Date"
    ].nunique()
)

print()

print(
    "Observed product-date rows:",
    f"{int(saved_step5_final['IsObservedProductDate'].sum()):,}"
)

print(
    "Zero-demand product-date rows:",
    f"{int(saved_step5_final['IsZeroDemandRow'].sum()):,}"
)

print()

print(
    "Normal demand units:",
    f"{int(saved_step5_final['NormalDemand'].sum()):,}"
)

print(
    "Bulk demand units:",
    f"{int(saved_step5_final['BulkDemand'].sum()):,}"
)

print(
    "Total demand units:",
    f"{int(saved_step5_final['TotalDemand'].sum()):,}"
)

print()

print("Final product-level daily forecasting dataset:")
print(STEP5_FINAL_DAILY_OUTPUT_FILE)

print()
print("Supporting product metadata:")
print(STEP5_PRODUCT_METADATA_OUTPUT_FILE)

print()
print("Operating calendar:")
print(STEP5_OPERATING_CALENDAR_OUTPUT_FILE)

print()
print("Observed daily aggregation audit:")
print(STEP5_OBSERVED_DAILY_AUDIT_OUTPUT_FILE)

print()
print("Final validation summary:")
print(STEP5_VALIDATION_SUMMARY_OUTPUT_FILE)

STEP 5 FINAL PRODUCT-LEVEL DAILY FORECASTING DATASET COMPLETED

Final rows: 25,405
Final columns: 38
Canonical products: 227
Observed operating dates: 245

Observed product-date rows: 15,138
Zero-demand product-date rows: 10,267

Normal demand units: 114,186
Bulk demand units: 1,972
Total demand units: 116,158

Final product-level daily forecasting dataset:
eden_datasets/UL_EDEN_canonical_product_daily_demand_forecasting_final.csv

Supporting product metadata:
eden_datasets/UL_EDEN_step5_final_product_metadata.csv

Operating calendar:
eden_datasets/UL_EDEN_step5_operating_calendar.csv

Observed daily aggregation audit:
eden_datasets/UL_EDEN_step5_observed_daily_demand_audit.csv

Final validation summary:
eden_datasets/UL_EDEN_step5_final_validation_summary.csv
